# Paper figures — the m25 quenched sample: AGN coupling, KS tracks, and the $A_V$ test

**Goal.** One notebook of **pure reads** that re-draws the paper-ready figures of the two analysis notebooks from their cached
tables and adds the test that combines them. Nothing is measured here: no snapshot, no particle file, no `simbanator` import —
only the `output/cis25` tree (it runs wherever that tree is visible) plus `ks_tracks_lib.py` and `obs_data/almac11/`.

* **Part 1 — the AGN coupling classifier** (`powderday_flux_quenched_m25.ipynb` Parts 2d/2e): the sequence of feedback events of
  the quenched galaxies, the strong / intermediate / weak $w_{\rm jet}$ and $f_{\rm gas}$ tracks under the adopted **pre-SFT
  threshold** rule and under the **[SFT, QT] terciles** it replaces, and the scorecard with the threshold sweeps.
* **Part 2 — the KS regions on the quench clock** (`ks_tracks_quenched_m25.ipynb` Part 5c): AGN-class composition and coupling
  strength of the galaxies ending below / on / above the reference relation, their dust and their kinematics against $t - t_{\rm QT}$.
* **Part 3 — new: the $A_V$ test.** The quenched galaxies with $10.5 < \log M_\star/M_\odot < 11.2$ at the anchor, binned by their
  **anchor $A_V$** (dust_on / dust_off RT of the m25 notebook), followed on the Kennicutt–Schmidt plane from the quench start (SFT)
  through the quench end (QT) to the track end — **one column per AGN coupling class**.
* **Part 4 — dust and rotation across the annuli** (m25 Part 8, T4/T8): the RT $A_V$, the surviving dust fraction and dust column
  of each projected annulus against the stellar age of the same annulus, one column per class; and the core (0–3.2 kpc) against the
  outskirt (3.2–10 kpc) in $A_V$, $M_{\rm dust}/M_\star$, $\Sigma_{\rm dust}$ and $\kappa_{\rm rot}^{\rm gas}$ per class; and (4d) the radial
  trend per class + SF controls across the four annuli in $A_V$, dust / gas / H$_2$ per stellar mass, age and $\kappa_{\rm rot}$.
* **Part 5 — the quench sequence in the ($\kappa_{\rm rot}$, stellar age) plane** (KS Part 3b): $\kappa_{\rm rot}$ of the H$_2$ and of the
  stars in the core sphere and in the outskirt shell against the stellar age of the same zone at the critical points — AGN ignition,
  jet-mode onset (m25 Part 2d events, attached by KS Part 1), SFT, QT, track end — per anchor-$A_V$ bin, one column per class, for the
  $\log M_\star > 10.25$ and the $\log M_\star < 10.25$ samples; dashed $\kappa_{\rm rot} = 0.3$. Plus three **sequence** figures (the
  critical points on a categorical axis, classes side by side: kinematics = $\kappa_{\rm rot}$ of the H$_2$ and of the stars | stellar =
  sSFR, age and the elapsed time of each interval over the cosmic time | ISM = $M_{\rm dust}/M_\star$, $M_{\rm H_2}/M_\star$; the range of
  the ALMA-C11 detections shaded) and the **intervals**
  figure (paired per-galaxy changes between consecutive critical points), which make the classes comparable whatever age range their
  tracks cover.

**Inputs** (every file is written by one of the two notebooks; the AGN class is `AGN_CLASSIFIER`, which must be the rule both ran with):

| file | written by | used in |
|---|---|---|
| `tables/powderday_quenched_selection.fits` | m25 Part 3 | every rule's labels (`agn_class_<tag>`), $M_\star$, $w_{\rm pre}$ |
| `tables/agn_classifier_windows.fits`, `tables/agn_classifier_tracks.fits` | m25 Part 2d | Part 1 |
| `tables/annulus_av_allincl.fits` (fallback `attenuation_vs_ism.fits`) | m25 Part 8b (7a) | Parts 3–4 |
| `tables/annulus_ism_truth.fits`, `tables/aperture_truth.fits`, `tables/annulus_kinematics.fits` | m25 Parts 8a, 7e, 8j0 | Part 4 |
| `ks_tracks[_<tag>]/ks_tracks.fits`, `ks_tracks[_<tag>]/ks_galaxies.fits` | KS Parts 4a, 5b | Parts 2–3 |
| `ks_tracks/ks_track_histories.fits` | KS Part 5a | Part 2 |
| `ks_tracks/ks_stage_kinematics.fits` | KS Part 3b (needs `vel` in the reduced files: job of 2026-08-28+, backfilled by the plan's sbatch) | Part 5 |
| `tables/agn_classifier_windows.fits` → KS Part 1 `agn_ign` / `jet_on` epochs (KS notebook of 2026-08-28 evening+; new plan entries) | m25 Part 2d, KS Parts 1–4 | Part 5 |
| `obs_data/almac11/ks_table.csv` | repo | Parts 2–3 (ALMA-C11 overlay) |
| `obs_data/almac11/almac11_gas_dust.csv` | repo (copied from `pilot_specphot/results`) | Part 5 (the age / $M_{\rm dust}/M_\star$ range of the ALMA-C11 detections) |

**Run order.** cluster: the m25 notebook Parts 0–3 (+ 2d, 7a/7e/8a/8b/8j0) and the KS notebook Parts 0–5 (incl. 3b) under the same rule → this
notebook top to bottom. **Outputs:** `output/cis25/plots/paper_m25[_<tag>]/paper_*.{png,pdf}`, `paper_ks_av_bins.csv`,
`paper_dust_vs_age_annuli.csv`, `paper_core_vs_outskirt.csv`, `paper_radial_profiles.csv`, `paper_kappa_{vs_age,sequence,intervals}.csv`.

In [ ]:
# ── Part 0 — configuration: pure reads of the caches of the two analysis notebooks (no simulation access, no simbanator) ──
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from astropy.table import Table
from scipy.stats import mannwhitneyu, ks_2samp
import ks_tracks_lib as kl                    # repo-root module: RELATIONS / relation_y, interp_track / grid_stats / ecdf

SIM_NAME = "cis25"
OUT      = os.path.join(os.getcwd(), "output", SIM_NAME)
TABLEDIR = os.path.join(OUT, "tables")
OBS_CSV  = os.path.join(os.getcwd(), "obs_data", "almac11", "ks_table.csv")
OBS_GD_CSV = os.path.join(os.getcwd(), "obs_data", "almac11", "almac11_gas_dust.csv")   # ALMA-C11 dust / CO detections + fiducial CIGALE age, M*, M_dust (Part 5 band)

# ── the AGN coupling class: the rule BOTH analysis notebooks ran with (their AGN_CLASSIFIER / KS_AGN_CLASSIFIER) ──
AGN_CLASSIFIER  = "pre_threshold"   # "quench_window" (legacy: <x_coup> over [SFT,QT], terciles) | "pre_jet" | "pre_threshold" (adopted 2026-08-28)
AGN_CLASSIFIERS = {"quench_window": "qw", "pre_jet": "pj", "pre_threshold": "pt"}
CLASS_TAG   = AGN_CLASSIFIERS[AGN_CLASSIFIER]
_CLS_SUF    = "" if AGN_CLASSIFIER == "quench_window" else f"_{CLASS_TAG}"
CLASS_SHORT = {"quench_window": "[SFT, QT] terciles", "pre_jet": "pre-SFT terciles", "pre_threshold": "pre-SFT threshold"}[AGN_CLASSIFIER]
COUP_VAR, COUP_LABEL = (("xstr_quench", r"$x_{\rm str}$ (AGN coupling over [SFT, QT])") if AGN_CLASSIFIER == "quench_window"
                        else ("w_pre", r"$\langle w_{\rm jet}\rangle$ over [SFT$-$1 Gyr, SFT] (ungated)"))

# ── inputs ──
SELECTION_FITS  = os.path.join(TABLEDIR, "powderday_quenched_selection.fits")   # m25 Part 3: every rule's labels, w_pre, log_mstar, t_sft / t_qt
P2D_FITS        = os.path.join(TABLEDIR, "agn_classifier_windows.fits")         # m25 Part 2d: one row per event galaxy (critical times, window means, labels)
P2D_TRACKS_FITS = os.path.join(TABLEDIR, "agn_classifier_tracks.fits")          # m25 Part 2d: the tracks behind the stacks (galaxy x history snapshot)
ANNULUS_AV_FITS = os.path.join(TABLEDIR, "annulus_av_allincl.fits")             # m25 Part 8b: A_V per (aperture / annulus, sightline)
ATTEN_FITS      = os.path.join(TABLEDIR, "attenuation_vs_ism.fits")             # m25 Part 7a: A_V per aperture, fiducial sightline (fallback)
ANNULUS_ISM_FITS    = os.path.join(TABLEDIR, "annulus_ism_truth.fits")          # m25 Part 8a: dust / gas / H2 per (aperture / annulus, sightline)
APERTURE_TRUTH_FITS = os.path.join(TABLEDIR, "aperture_truth.fits")             # m25 Part 7e: M*, mass-weighted stellar age per (aperture, sightline)
ANNULUS_KIN_FITS    = os.path.join(TABLEDIR, "annulus_kinematics.fits")         # m25 Part 8j0: kappa_rot of gas / H2 / stars per spherical rung / shell
KSDIR       = os.path.join(OUT, "ks_tracks")                                    # KS caches (rule-free)
KSOUT       = KSDIR + _CLS_SUF                                                  # KS derived products of this rule
TRACKS_FITS = os.path.join(KSOUT, "ks_tracks.fits")                             # KS Part 4a: epochs x apertures on the KS plane
GALS_FITS   = os.path.join(KSOUT, "ks_galaxies.fits")                           # KS Part 5b: one row per Q galaxy (region, class, clock, stage snapshots)
HIST_FITS   = os.path.join(KSDIR, "ks_track_histories.fits")                    # KS Part 5a: continuous histories (+ BH, caesar rotation)
KIN_STAGE_FITS = os.path.join(KSDIR, "ks_stage_kinematics.fits")               # KS Part 3b: kappa_rot of gas / H2 / stars + stellar age per spherical zone at every epoch
PAPERDIR    = os.path.join(OUT, "plots", "paper_m25" + _CLS_SUF)
os.makedirs(PAPERDIR, exist_ok=True)

# ── style ──
PAPER_DPI = 300
plt.rcParams.update({"font.size": 11, "axes.titlesize": 11, "axes.labelsize": 11, "legend.fontsize": 9, "xtick.labelsize": 9.5,
                     "ytick.labelsize": 9.5, "axes.titleweight": "bold", "pdf.fonttype": 42})


def _s(col):
    """FITS string column -> stripped str array."""
    return np.char.strip(np.asarray(col).astype(str))


def need(path, made_by):
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} is missing: run {made_by} (under the {AGN_CLASSIFIER!r} rule) first")
    return path


def paper_save(fig, stem):
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(PAPERDIR, f"paper_{stem}.{ext}"), dpi=PAPER_DPI, bbox_inches="tight")
    print("figure ->", os.path.join(PAPERDIR, f"paper_{stem}.{{png,pdf}}"))


def class_labels():
    """(selection table, {(snap, gal_id): AGN class}) under AGN_CLASSIFIER: column agn_class_<tag> (m25 Part 3, 2026-08-28+); on an older
    table the bare `agn_class` is accepted only when its AGNCLASS header names the same rule."""
    sel = Table.read(need(SELECTION_FITS, "powderday_flux_quenched_m25.ipynb Parts 2-3"))
    col = f"agn_class_{CLASS_TAG}"
    if col not in sel.colnames:
        wrote = str(sel.meta.get("AGNCLASS", "quench_window"))
        if wrote != AGN_CLASSIFIER:
            raise KeyError(f"{SELECTION_FITS} has no {col!r} column and its `agn_class` is the {wrote!r} rule: re-run the m25 notebook Parts 2-3")
        col = "agn_class"
    return sel, {(int(a), int(b)): c for a, b, c in zip(sel["snap"], sel["gal_id"], _s(sel[col]))}


SEL, CLASS_OF = class_labels()
_isq = _s(SEL["pop"]) == "Q"
print(f"AGN coupling class = {AGN_CLASSIFIER} ({CLASS_SHORT}); strength variable {COUP_VAR}; figures -> {PAPERDIR}")
print(f"selection table: {int(_isq.sum())} Q + {int((~_isq).sum())} SF; Q classes "
      f"{dict(zip(*np.unique([CLASS_OF[(int(a), int(b))] for a, b in zip(SEL['snap'][_isq], SEL['gal_id'][_isq])], return_counts=True)))}")
for _p, _by in ((P2D_FITS, "m25 Part 2d"), (P2D_TRACKS_FITS, "m25 Part 2d"), (ANNULUS_AV_FITS, "m25 Part 8b"), (ATTEN_FITS, "m25 Part 7a"),
                (ANNULUS_ISM_FITS, "m25 Part 8a"), (APERTURE_TRUTH_FITS, "m25 Part 7e"), (ANNULUS_KIN_FITS, "m25 Part 8j0"),
                (TRACKS_FITS, "KS Part 4a"), (GALS_FITS, "KS Part 5b"), (HIST_FITS, "KS Part 5a"), (KIN_STAGE_FITS, "KS Part 3b"),
                (OBS_CSV, "git pull"), (OBS_GD_CSV, "git pull")):
    print(f"  {'ok     ' if os.path.exists(_p) else 'MISSING'} {os.path.relpath(_p, os.getcwd()) if _p.startswith(os.getcwd()) else _p}   [{_by}]")

## Part 1 — the AGN coupling classifier: the feedback-event sequence and the two selection windows

Re-drawn from the two tables of the m25 notebook's Part 2d — `agn_classifier_windows.fits` (one row per quenched galaxy with an
SFT/QT event: the critical times $t_{\rm AGN}$ (ignition), $M_{\rm BH}$ crossing the jet-eligibility mass, the jet onset, SFT, the
$f_{\rm gas}$ gate closing, QT, the anchor; the window means $w_{\rm pre} = \langle w_{\rm jet}\rangle_{[{\rm SFT}-1,\,{\rm SFT}]}$ and
$x_{\rm str} = \langle x_{\rm coup}\rangle_{[{\rm SFT},\,{\rm QT}]}$; every rule's label) and `agn_classifier_tracks.fits` (the
$w_{\rm jet}$, $x_{\rm coup}$, $f_{\rm gas}$, $f_{\rm Edd}$, $M_{\rm BH}$, $\dot M_{\rm BH}$ tracks on the $t - t_{\rm SFT}$ clock).
The figures are those of Part 2e: (1) the sequence of events of the whole event sample, (2) the strong / intermediate / weak
tracks under the **pre-SFT threshold** (adopted) and the **[SFT, QT] terciles** (replaced), (3) the scorecard — separation of
$w_{\rm jet}$ before and after SFT, purity against the jet onset, the mass and pre-SFT gas confounds — and the sweeps of a single
threshold on the two continuous variables (the point of the figure: under the [SFT, QT] window the separation *before* SFT is flat
in the threshold, i.e. the window, not the threshold, decides what a rule can select).

In [ ]:
# ── Part 1a — the Part 2d tables + the stack / ledger helpers of the 2d/2e figures ──
G = Table.read(need(P2D_FITS, "powderday_flux_quenched_m25.ipynb Part 2d"))
_TT = Table.read(need(P2D_TRACKS_FITS, "powderday_flux_quenched_m25.ipynb Part 2d (2026-08-28+: it writes the tracks table)"))
G1 = {k: (_s(G[k]) if G[k].dtype.kind in "SU" else np.asarray(G[k], float)) for k in G.colnames}
N1 = len(G)
_gi = {(int(a), int(b)): i for i, (a, b) in enumerate(zip(G["snap"], G["gal_id"]))}
_row = np.array([_gi[(int(a), int(b))] for a, b in zip(_TT["snap"], _TT["gal_id"])], int)
_ord = np.lexsort((np.asarray(_TT["dt_sft"], float), _row))
_row = _row[_ord]
_cnt = np.bincount(_row, minlength=N1)
assert (_cnt > 0).all(), "every galaxy of the windows table needs a track (the two tables come from the same Part 2d run)"
_st0 = np.r_[0, np.cumsum(_cnt)[:-1]]
TRK = {k: [np.asarray(_TT[c], float)[_ord][s:s + n] for s, n in zip(_st0, _cnt)]
       for k, c in (("t", "dt_sft"), ("w", "w"), ("x", "x"), ("fg", "fg"), ("lfe", "lfe"), ("lmb", "lmb"), ("lmd", "lmd"))}

# rule parameters travel in the table headers; the SIMBA jet-mode / gate constants are those of the m25 notebook Part 0
PRE_WIN  = float(G.meta.get("PREWIN", 1.0))                 # pre-SFT window [Gyr]
JET_ON   = float(G.meta.get("JETON", 0.5))                  # jet onset: first snapshot with the ungated w_jet >= this
THR_WPRE = tuple(float(v) for v in str(G.meta.get("THRWPRE", "0.1,0.5")).split(","))   # pre-SFT threshold rule: weak < , strong >=
JET_FEDD, JET_LOGMBH, XRAY_FGAS_MAX = 0.2, 7.5, 0.2         # jet mode: log M_BH > 7.5 and f_Edd < 0.2; coupling gate f_gas < 0.2
IGN_FLOOR  = 1e-3                                           # BHAR floor of the ignition rule (m25 P2D_IGN_FLOOR)
FGAS_FLOOR = float(_TT.meta.get("FGASFLR", 1e-3))
CLOCK_BINS = np.arange(-2.5, 2.51, 0.25)
CTR, XL = 0.5 * (CLOCK_BINS[:-1] + CLOCK_BINS[1:]), (CLOCK_BINS[0], CLOCK_BINS[-1])
BAND_MIN, NBOOT = 15, 300
P1_COLORS = {"strong": "#c0392b", "intermediate": "#e67e22", "weak": "#2980b9", "all": "0.2"}
CRIT = [("t_agn", r"$t_{\rm AGN}$ (ignition)", "*"), ("t_mbh", r"$M_{\rm BH} > 10^{%.1f}$" % JET_LOGMBH, "D"),
        ("t_jet", r"jet onset ($w_{\rm jet} \geq %g$)" % JET_ON, "^"), ("t_sft", "SFT", "s"),
        ("t_gate", r"$f_{\rm gas} < %g$ gate" % XRAY_FGAS_MAX, "v"), ("t_qt", "QT", "o"), ("t_anchor", "anchor", "P")]
CK = [k for k, _, _ in CRIT]
SHORT_CRIT = {"t_agn": "ignition", "t_mbh": r"$M_{\rm BH}$ threshold", "t_jet": "jet onset", "t_gate": "gate", "t_qt": "QT"}
LFE_SAT = np.log10(JET_FEDD) - 1.0                          # w_jet = 1 (f_Edd one dex below the jet threshold)
ALL = np.ones(N1, bool)
TAGS = sorted(set(G1["tag"]), key=lambda s: float(s[1:].replace("p", ".")))
TAU_MED = float(np.nanmedian(G1["tau_q"]))

# the two rules of the paper: the rule adopted and the rule it replaces (labels as stored by Part 2d; the driver's own labels are cls_<tag>)
P1_RULES  = [("pre-SFT | threshold", "pre", "cls_pre_threshold", "w_pre"), ("[SFT,QT] | terciles", "win", "cls_win_terciles", "x_str")]
P1_TITLES = {"pre-SFT | threshold": r"pre-SFT window, threshold on $\langle w_{\rm jet}\rangle_{[{\rm SFT}-1,\ {\rm SFT}]}$",
             "[SFT,QT] | terciles": r"[SFT, QT] window, per-anchor terciles of $\langle x_{\rm coup}\rangle_{[{\rm SFT},\ {\rm QT}]}$"}
P1_SHORT  = {"pre-SFT | threshold": "pre-SFT\nthreshold", "[SFT,QT] | terciles": "[SFT, QT]\nterciles"}
LAB = {nm: G1[col] for nm, _, col, _ in P1_RULES}
RULE_COL = {nm: ("#c0392b" if win == "pre" else "#2980b9") for nm, win, _, _ in P1_RULES}
_drv = f"cls_{CLASS_TAG}"
if _drv in G1 and AGN_CLASSIFIER == "pre_threshold":
    _mis = int((G1[_drv] != G1["cls_pre_threshold"]).sum())
    print(f"driver labels ({_drv}) vs the 'pre-SFT | threshold' rule of the table: {_mis} mismatches" + ("" if _mis == 0 else "  <- re-run m25 Parts 2 + 2d"))
print(f"Part 1: {N1} event galaxies over {len(TAGS)} anchors; tracks: {len(_TT)} rows; pre-SFT window {PRE_WIN:g} Gyr, thresholds {THR_WPRE}, "
      f"median tau_q = {TAU_MED:.2f} Gyr")
for nm, _, _, _ in P1_RULES:
    print(f"  {nm:>22s}: " + ", ".join(f"{c} {int((LAB[nm] == c).sum())}" for c in ("strong", "intermediate", "weak", "no_AGN")))


# ── track stacks on the t - t_SFT clock ──
def binvals(key, idx, lo, hi):
    """per galaxy in `idx`: mean of track `key` over lo <= t - t_SFT < hi (NaN without finite samples)."""
    out = np.full(len(idx), np.nan)
    for n, i in enumerate(idx):
        m = (TRK["t"][i] >= lo) & (TRK["t"][i] < hi) & np.isfinite(TRK[key][i])
        if m.any():
            out[n] = np.mean(TRK[key][i][m])
    return out


def stack(key, mask):
    out = np.full((len(CTR), 4), np.nan)
    idx = np.where(mask)[0]
    for j, (lo, hi) in enumerate(zip(CLOCK_BINS[:-1], CLOCK_BINS[1:])):
        v = binvals(key, idx, lo, hi)
        v = v[np.isfinite(v)]
        if len(v) >= 5:
            out[j] = (np.median(v), np.percentile(v, 16), np.percentile(v, 84), len(v))
    return out


def track(ax, key, mask, color, label=None, lw=2.2, band=True, ls="-", alpha=1.0):
    st = stack(key, mask)
    ok = st[:, 3] >= BAND_MIN
    if band:
        ax.fill_between(CTR, np.where(ok, st[:, 1], np.nan), np.where(ok, st[:, 2], np.nan), color=color, alpha=0.10, lw=0)
    ax.plot(CTR, st[:, 0], ls, color=color, lw=lw, label=label, alpha=alpha)
    return st


def crit_stats(mask):
    """per critical point: median / 16 / 84 of t - t_SFT, the fraction of the group where it is defined, the fraction before SFT."""
    out = {}
    for k in CK:
        v = G1[k][mask]
        f = np.isfinite(v)
        out[k] = ((np.nanmedian(v), np.nanpercentile(v, 16), np.nanpercentile(v, 84), f.mean(), (v[f] < 0).mean() if f.any() else np.nan)
                  if f.sum() >= 3 else (np.nan,) * 5)
    return out


def crit_timing(ax, groups, xlim):
    """horizontal ladder of the critical points: median (marker), 16-84 (thick), 5-95 (thin) of t_k - t_SFT per group."""
    ng = len(groups)
    off = np.linspace(-0.3, 0.3, ng) if ng > 1 else [0.0]
    xhi = xlim[1] - 0.95                                     # text column on the right
    for gi, (gname, mask, color) in enumerate(groups):
        for ki, (k, lab, mk) in enumerate(CRIT):
            v = G1[k][mask]
            v = v[np.isfinite(v)]
            if len(v) < 3:
                continue
            y = ki + off[gi]
            p5, p16, p50, p84, p95 = np.percentile(v, [5, 16, 50, 84, 95])
            ax.plot([max(p5, xlim[0]), min(p95, xhi)], [y, y], "-", color=color, lw=0.9, alpha=0.7)
            ax.plot([max(p16, xlim[0]), min(p84, xhi)], [y, y], "-", color=color, lw=3.2, alpha=0.9, solid_capstyle="butt")
            ax.plot(np.clip(p50, xlim[0], xhi), y, mk, ms=8 if mk != "*" else 12, color=color, mec="k", mew=0.6, zorder=5,
                    label=gname if ki == 0 and ng > 1 else None)
            ax.text(xlim[1] - 0.03, y, f"{100 * np.isfinite(G1[k][mask]).mean():.0f}% | {100 * (v < 0).mean():.0f}%",
                    fontsize=7.5, color=color, ha="right", va="center")
    ax.set_yticks(range(len(CRIT))); ax.set_yticklabels([lab for _, lab, _ in CRIT]); ax.set_ylim(len(CRIT) - 0.5, -1.0)
    ax.axvline(0, color="k", lw=1); ax.axvline(xhi, color="0.7", lw=0.6); ax.set_xlim(*xlim); ax.set_xlabel(r"$t - t_{\rm SFT}$ [Gyr]")
    ax.text(xlim[1] - 0.03, -0.7, "defined |\nbefore SFT", fontsize=7.5, ha="right", va="center", color="0.3")
    ax.grid(axis="y", lw=0.4, alpha=0.4)
    if ng > 1:
        ax.legend(frameon=False, loc="lower left", fontsize=9)


def vlines(ax, cs, color="k"):
    for k, lab, mk in CRIT:
        if k in ("t_sft", "t_anchor") or not np.isfinite(cs[k][0]):
            continue
        ax.axvline(cs[k][0], color=color, lw=0.8, ls=":", alpha=0.8)
    ax.axvline(0, color="k", lw=1.1)


def sig_strip(ax, key, lab, y0, y1):
    """bottom strip: strong vs weak Mann-Whitney per clock bin (black p < 0.001, dark grey < 0.01, light grey < 0.05)."""
    s_idx, w_idx = np.where(lab == "strong")[0], np.where(lab == "weak")[0]
    for lo, hi in zip(CLOCK_BINS[:-1], CLOCK_BINS[1:]):
        a, b = binvals(key, s_idx, lo, hi), binvals(key, w_idx, lo, hi)
        a, b = a[np.isfinite(a)], b[np.isfinite(b)]
        if len(a) < 5 or len(b) < 5:
            continue
        p = mannwhitneyu(a, b).pvalue
        col = "k" if p < 1e-3 else ("0.55" if p < 0.01 else ("0.85" if p < 0.05 else "none"))
        if col != "none":
            ax.fill_between([lo, hi], y0, y1, color=col, lw=0, transform=ax.get_xaxis_transform())


# ── the scorecard: strong - weak contrasts and the purities (bootstrap within the classes) ──
def score(lab, rng=None):
    s_idx, w_idx = np.where(lab == "strong")[0], np.where(lab == "weak")[0]
    if rng is not None:
        s_idx, w_idx = rng.choice(s_idx, len(s_idx)), rng.choice(w_idx, len(w_idx))
    keys = ("dw_pre", "dw_post", "pur_s", "pur_w", "dlogm", "dtau", "dclock", "dlfg_pre")
    if len(s_idx) < 5 or len(w_idx) < 5:
        return dict((k, np.nan) for k in keys)

    def dmed(a, b):
        a, b = a[np.isfinite(a)], b[np.isfinite(b)]
        return np.median(a) - np.median(b) if len(a) and len(b) else np.nan

    return dict(dw_pre=dmed(binvals("w", s_idx, -1.0, 0.0), binvals("w", w_idx, -1.0, 0.0)),
                dw_post=dmed(binvals("w", s_idx, 0.0, 0.5), binvals("w", w_idx, 0.0, 0.5)),
                pur_s=np.mean(G1["lead_jet"][s_idx] > 0),                       # strong: jets on before SFT
                pur_w=np.mean(~(G1["lead_jet"][w_idx] > 0)),                    # weak: jet mode only after SFT (or never)
                dlogm=dmed(G1["log_mstar"][s_idx], G1["log_mstar"][w_idx]), dtau=dmed(G1["tau_q"][s_idx], G1["tau_q"][w_idx]),
                dclock=dmed(G1["dt_qt"][s_idx], G1["dt_qt"][w_idx]),
                dlfg_pre=dmed(np.log10(np.clip(binvals("fg", s_idx, -1.0, 0.0), FGAS_FLOOR, None)),
                              np.log10(np.clip(binvals("fg", w_idx, -1.0, 0.0), FGAS_FLOOR, None))))


def sweep(v, grid):
    """single threshold s on the continuous variable v: strong = v >= s, weak = the rest -> the contrasts per s."""
    out = {k: np.full(len(grid), np.nan) for k in ("frac", "dw_pre", "dw_post", "dlogm", "dtau")}
    for n, s in enumerate(grid):
        lab = np.array(["no_AGN"] * N1, dtype=object)
        f = np.isfinite(v)
        lab[f] = "weak"; lab[f & (v >= s)] = "strong"
        if (lab == "strong").sum() < 10 or (lab == "weak").sum() < 10:
            continue
        sc = score(lab)
        out["frac"][n] = (lab == "strong").sum() / (lab != "no_AGN").sum()
        for k in ("dw_pre", "dw_post", "dlogm", "dtau"):
            out[k][n] = sc[k]
    return out


_rng = np.random.default_rng(2026)
SCORE = {}
print(f"\n{'rule':>22s} {'dw_pre':>7s} {'dw_post':>7s} {'pur_s':>6s} {'pur_w':>6s} {'dlogM*':>7s} {'dtau_q':>7s} {'dlfg_pre':>8s}")
for nm, _, _, _ in P1_RULES:
    sc = score(LAB[nm])
    bs = [score(LAB[nm], _rng) for _ in range(NBOOT)]
    SCORE[nm] = dict(point=sc, lo={k: np.nanpercentile([b[k] for b in bs], 16) for k in sc}, hi={k: np.nanpercentile([b[k] for b in bs], 84) for k in sc})
    print(f"{nm:>22s} {sc['dw_pre']:+7.2f} {sc['dw_post']:+7.2f} {100 * sc['pur_s']:5.0f}% {100 * sc['pur_w']:5.0f}% {sc['dlogm']:+7.2f} {sc['dtau']:+7.2f} {sc['dlfg_pre']:+8.2f}")
print("dw = strong - weak median of the per-galaxy window mean of w_jet; pur_s = strong whose jet onset precedes SFT, pur_w = weak in jet mode only after SFT (or never)")
CS_ALL = crit_stats(ALL)
print("critical points, whole event sample (median t - t_SFT [Gyr] | defined | before SFT): "
      + "; ".join(f"{k[2:]} {CS_ALL[k][0]:+.2f} ({100 * CS_ALL[k][3]:.0f}% | {100 * CS_ALL[k][4]:.0f}%)" for k in CK if np.isfinite(CS_ALL[k][0])))

In [ ]:
# ── Part 1b — paper figure 1: the sequence of feedback events of the whole event sample (2 x 2) ──
fig, axes = plt.subplots(2, 2, figsize=(13.5, 9.8))
a, b, c, d = axes.ravel()
crit_timing(a, [("all", ALL, P1_COLORS["all"])], XL)
a.set_title(f"(a) the sequence of feedback events, {N1} quenched galaxies", loc="left")
a.text(0.02, 0.03, "marker: median; thick: 16-84%; thin: 5-95%", transform=a.transAxes, fontsize=9, color="0.3", ha="left", va="bottom")
for ax, key, ylab, ttl in ((b, "lmd", r"median $\log \dot M_{\rm BH}$ [M$_\odot$ yr$^{-1}$]", "(b) BH accretion rate"),
                           (c, "lfe", r"median $\log f_{\rm Edd}$", "(c) Eddington ratio")):
    track(ax, key, ALL, P1_COLORS["all"]); vlines(ax, CS_ALL); ax.set_xlim(*XL); ax.set_ylabel(ylab); ax.set_title(ttl, loc="left")
b.axhline(np.log10(IGN_FLOOR), color="k", lw=0.8, ls="--"); b.text(XL[0] + 0.05, np.log10(IGN_FLOOR), "ignition floor", fontsize=9, va="bottom")
c.axhline(np.log10(JET_FEDD), color="k", lw=0.8, ls="--"); c.text(XL[0] + 0.05, np.log10(JET_FEDD), r"$w_{\rm jet}=0$", fontsize=9, va="bottom")
c.axhline(LFE_SAT, color="k", lw=0.8, ls="-"); c.text(XL[0] + 0.05, LFE_SAT, r"$w_{\rm jet}=1$", fontsize=9, va="top")
for key, col, lab in (("w", "#c0392b", r"$w_{\rm jet}$ (ungated)"), ("x", "#8e44ad", r"$x_{\rm coup}$ (gated)")):
    track(d, key, ALL, col, label=lab)
_gate_frac = np.array([np.mean(v[np.isfinite(v)] < XRAY_FGAS_MAX) if np.isfinite(v).sum() >= 5 else np.nan
                       for v in (binvals("fg", np.where(ALL)[0], lo, hi) for lo, hi in zip(CLOCK_BINS[:-1], CLOCK_BINS[1:]))])
d.plot(CTR, _gate_frac, "--", color="#16a085", lw=2, label=r"fraction with $f_{\rm gas}<%g$ (gate closed)" % XRAY_FGAS_MAX)
vlines(d, CS_ALL); d.set_xlim(*XL); d.set_ylabel("median across galaxies"); d.legend(frameon=False, loc="center left")
d.set_title("(d) jet weight, gated coupling and the gate", loc="left"); d.set_ylim(-0.42, 1.03); d.set_yticks(np.arange(0, 1.01, 0.2))
for ax in (b, c, d):
    for k, lab in SHORT_CRIT.items():
        if np.isfinite(CS_ALL[k][0]):
            ax.text(CS_ALL[k][0] - 0.02, 0.02, lab, transform=ax.get_xaxis_transform(), rotation=90, ha="right", va="bottom", fontsize=9, color="0.25")
    ax.set_xlabel(r"$t - t_{\rm SFT}$ [Gyr]")
fig.tight_layout(); paper_save(fig, "agn_feedback_sequence"); plt.show()

In [ ]:
# ── Part 1c — paper figure 2: the w_jet and f_gas tracks per class under the two rules (2 x 2) ──
fig, axes = plt.subplots(2, 2, figsize=(13, 8.4), sharex=True)
for j, (nm, win, _, _) in enumerate(P1_RULES):
    lab = LAB[nm]
    cls = [cl for cl in ("strong", "intermediate", "weak") if (lab == cl).sum() >= 5]
    w0, w1 = (-PRE_WIN, 0.0) if win == "pre" else (0.0, TAU_MED)
    for i, (key, ylab) in enumerate((("w", r"median $w_{\rm jet}$ (ungated)"), ("fg", r"median $f_{\rm gas} = M_{\rm gas}/M_\star$"))):
        ax = axes[i, j]
        for cl in cls:
            track(ax, key, lab == cl, P1_COLORS[cl], label=f"{cl} ({int((lab == cl).sum())})", band=(cl != "intermediate"))
        sig_strip(ax, key, lab, 0.0, 0.035)
        ax.axvline(0, color="k", lw=1); ax.axvline(TAU_MED, color="k", lw=0.8, ls="--")
        ax.axvspan(w0, w1, color="0.5", alpha=0.10, lw=0); ax.set_xlim(XL[0], 2.0)
        if key == "fg":
            ax.set_yscale("log"); ax.set_ylim(3e-3, 20); ax.axhline(XRAY_FGAS_MAX, color="k", lw=0.8, ls="-.")
            ax.text(1.97, XRAY_FGAS_MAX, r"$f_{\rm gas}$ gate", fontsize=9, ha="right", va="bottom")
            ax.set_xlabel(r"$t - t_{\rm SFT}$ [Gyr]")
            ax.text(0.5 * (w0 + w1), 0.985, "selection\nwindow", transform=ax.get_xaxis_transform(), ha="center", va="top", fontsize=9, color="0.35")
            ax.text(TAU_MED + 0.03, 0.07, "median QT", transform=ax.get_xaxis_transform(), ha="left", va="bottom", fontsize=9, color="0.35")
        else:
            ax.set_ylim(-0.03, 1.03)
        if j == 0:
            ax.set_ylabel(ylab)
        ax.set_title(f"({'abcd'[2 * i + j]}) " + (P1_TITLES[nm] if i == 0 else ""), loc="left")
        if i == 0:
            ax.legend(frameon=False, loc="center left")
fig.tight_layout()
fig.text(0.5, -0.01, "bottom strips: strong vs weak Mann-Whitney p < 0.001 (black), < 0.01 (dark grey), < 0.05 (light grey); shaded: 16-84% of the class",
         fontsize=9.5, color="0.3", ha="center", va="top")
paper_save(fig, "agn_selection_windows"); plt.show()

In [ ]:
# ── Part 1d — paper figure 3: the reduced scorecard — the contrasts of the two rules + the two threshold sweeps ──
P1_METRICS = [("dw_pre", r"$\Delta\langle w_{\rm jet}\rangle$ over [SFT$-$1, SFT)", "(a) separation before SFT"),
              ("dw_post", r"$\Delta\langle w_{\rm jet}\rangle$ over [SFT, SFT+0.5)", "(b) separation after SFT"),
              ("pur", "% of the class", "(c) purity against the jet onset"),
              ("dex", r"strong $-$ weak [dex]", "(d) mass and pre-SFT gas confounds")]
P1_SWEEPS = [("w_pre", np.linspace(0.02, 0.98, 25)), ("x_str", np.linspace(0.02, 1.0, 25))]
_t23 = np.nanmedian([np.nanquantile(G1["x_str"][(G1["tag"] == tg) & np.isfinite(G1["x_str"])], 2 / 3)
                     for tg in TAGS if ((G1["tag"] == tg) & np.isfinite(G1["x_str"])).sum() >= 3])
_mark = {"w_pre": [(THR_WPRE[1], "strong threshold")], "x_str": [(_t23, "median upper tercile")]}
fig = plt.figure(figsize=(15, 8))
gs = fig.add_gridspec(2, 4, height_ratios=[0.5, 1.0], hspace=0.55, wspace=0.45, bottom=0.12, top=0.96, left=0.09, right=0.98)
_ypos = np.arange(len(P1_RULES))[::-1]
_names = [nm for nm, _, _, _ in P1_RULES]
for j, (mk, xlab, ttl) in enumerate(P1_METRICS):
    ax = fig.add_subplot(gs[0, j])
    for y, nm in zip(_ypos, _names):
        S, col, first = SCORE[nm], RULE_COL[nm], y == _ypos[0]
        if mk == "pur":
            for k, dy, m, mfc, lab in (("pur_s", 0.15, "s", col, "strong: jet mode before SFT"), ("pur_w", -0.15, "o", "white", "weak: jet mode only after SFT, or never")):
                ax.plot([100 * S["lo"][k], 100 * S["hi"][k]], [y + dy, y + dy], "-", color=col, lw=1.6)
                ax.plot(100 * S["point"][k], y + dy, m, color=col, mfc=mfc, mec=col if mfc == "white" else "k", ms=9, label=lab if first else None)
        elif mk == "dex":
            for k, dy, m, mfc, lab in (("dlogm", 0.15, "D", col, r"$\Delta \log M_\star$"), ("dlfg_pre", -0.15, "v", "white", r"$\Delta\langle\log f_{\rm gas}\rangle$ over [SFT$-$1, SFT)")):
                ax.plot([S["lo"][k], S["hi"][k]], [y + dy, y + dy], "-", color=col, lw=1.6)
                ax.plot(S["point"][k], y + dy, m, color=col, mfc=mfc, mec=col if mfc == "white" else "k", ms=9, label=lab if first else None)
        else:
            ax.plot([S["lo"][mk], S["hi"][mk]], [y, y], "-", color=col, lw=2.2)
            ax.plot(S["point"][mk], y, "o", color=col, mec="k", ms=10)
    ax.set_yticks(_ypos); ax.set_yticklabels([P1_SHORT[nm] for nm in _names] if j == 0 else [""] * len(_names)); ax.set_ylim(-0.8, len(_names) - 0.2)
    for tl, nm in zip(ax.get_yticklabels(), _names):
        tl.set_color(RULE_COL[nm])
    ax.axvline(0 if mk != "pur" else 50, color="k", lw=0.8, ls=":"); ax.grid(axis="y", lw=0.4, alpha=0.4)
    ax.set_xlabel(xlab); ax.set_title(ttl, loc="left")
    if mk in ("dw_pre", "dw_post"):
        ax.set_xlim(-0.05, 1.0)
    if mk == "pur":
        ax.set_xlim(0, 104)
    if mk in ("pur", "dex"):
        ax.legend(frameon=False, loc="lower left", fontsize=8.5, handletextpad=0.3)
for j, (var, grid) in enumerate(P1_SWEEPS):
    ax = fig.add_subplot(gs[1, 2 * j:2 * j + 2])
    sw = sweep(G1[var], grid)
    ax.plot(grid, sw["dw_pre"], "-", color="#c0392b", lw=2.4, label=r"$\Delta\langle w_{\rm jet}\rangle$ before SFT, [SFT$-$1, SFT)")
    ax.plot(grid, sw["dw_post"], "--", color="#c0392b", lw=1.8, label=r"$\Delta\langle w_{\rm jet}\rangle$ after SFT, [SFT, SFT+0.5)")
    ax.plot(grid, sw["dlogm"], "-", color="#7f8c8d", lw=2, label=r"$\Delta \log M_\star$ [dex]")
    ax.plot(grid, sw["frac"], ":", color="k", lw=1.8, label="fraction selected as strong")
    for s, lab in _mark[var]:
        _right = s > 0.5 * (grid[0] + grid[-1])
        ax.axvline(s, color="k", lw=0.9, ls="--", alpha=0.7)
        ax.text(s + (-0.01 if _right else 0.01), 0.97, lab, transform=ax.get_xaxis_transform(), fontsize=9, ha="right" if _right else "left", va="top", color="0.3")
    ax.axhline(0, color="k", lw=0.6); ax.set_ylim(-0.25, 1.0); ax.set_ylabel(r"strong $-$ rest (single threshold $s$)")
    ax.set_xlabel((r"$s$ on $\langle w_{\rm jet}\rangle_{[{\rm SFT}-1,\ {\rm SFT}]}$" if var == "w_pre" else r"$s$ on $\langle x_{\rm coup}\rangle_{[{\rm SFT},\ {\rm QT}]}$")
                  + " (strong: value $\\geq s$)")
    ax.set_title(f"({'ef'[j]}) " + ("pre-SFT window: sweep of the threshold" if var == "w_pre" else "[SFT, QT] window: sweep of the threshold"), loc="left")
    if j == 0:
        _h, _l = ax.get_legend_handles_labels()
fig.legend(_h, _l, frameon=False, loc="lower center", ncol=4, fontsize=10, bbox_to_anchor=(0.5, 0.0))
paper_save(fig, "agn_selection_scorecard"); plt.show()

## Part 2 — the KS regions on the quench clock

Re-drawn from the KS notebook's products: `ks_tracks.fits` (every critical epoch of every quenched galaxy in the six apertures on
the KS plane; the fiducial aperture is $R_{50}({\rm H_2})$ with the observed $0.5M/\pi R_{50}^2$ convention, $\times1.36$ He,
100 Myr archaeological SFR), `ks_galaxies.fits` (one row per quenched galaxy: the **region of its track end** with respect to the
reference relation — below / on / above Bigiel+08 by more than its 0.20 dex scatter — the AGN class, the quench clock
$t_{\rm QT}$, the snapshots of the drawn stages) and `ks_track_histories.fits` (the continuous catalogue histories of the tracked
progenitors, with the BH history and the caesar rotation measures). The three figures are the presentation figures of the KS
notebook's Part 5c: (A) the AGN-class composition of each region and the ECDF of the rule's own strength variable; (B) the
dust-fraction / stellar-age plane, $M_{\rm dust}/M_\star$ and the dust-to-gas ratio on the clock; (C) the gas $\kappa_{\rm rot}$
and the stellar B/T on the clock, each with its ECDF at the track end. Columns: all anchors (thin frame), then the three
anchor-redshift ranges; grey = the mass-matched star-forming controls at their anchor.

In [ ]:
# ── Part 2a — the KS products (Parts 4a / 5a / 5b of ks_tracks_quenched_m25) + the plane / clock / presentation helpers of its Part 5c ──
TRACKS = Table.read(need(TRACKS_FITS, "ks_tracks_quenched_m25.ipynb Part 4"))
GALS   = Table.read(need(GALS_FITS, "ks_tracks_quenched_m25.ipynb Part 5b (2026-08-28+: it writes ks_galaxies.fits)"))
HIST   = Table.read(need(HIST_FITS, "ks_tracks_quenched_m25.ipynb Part 5a"))
OBS    = pd.read_csv(need(OBS_CSV, "git pull"))
OBS_S  = OBS[OBS["co_det"].astype(bool) & np.isfinite(OBS["logSigmaH2"])].reset_index(drop=True)


def _keys(tab):
    if "anchor_snap" in tab.colnames:
        return list(zip(np.asarray(tab["anchor_snap"], int), np.asarray(tab["gal_id"], int)))
    return [tuple(int(v) for v in g.split("_")) for g in _s(tab["gkey"])]


def relabel(tab, what):
    """agn_class of the Q rows <- the selection table under AGN_CLASSIFIER. The KS products carry the labels of the rule that notebook ran
    with; a difference means the two notebooks disagree on the rule (relabelled here, but the KS products should be regenerated)."""
    keys = _keys(tab)
    isq = (_s(tab["pop"]) == "Q") if "pop" in tab.colnames else np.ones(len(tab), bool)
    old = _s(tab["agn_class"])
    new = np.array([CLASS_OF.get(k, o) if q else o for k, o, q in zip(keys, old, isq)]).astype(str)
    if (new != old).any():
        warnings.warn(f"{what}: {int((new != old).sum())} rows carried another rule's class (KS notebook run with KS_AGN_CLASSIFIER != "
                      f"{AGN_CLASSIFIER!r}?) -> relabelled; re-run the KS notebook under {AGN_CLASSIFIER!r} for consistent region products")
    tab["agn_class"] = new
    return tab


TRACKS, GALS = relabel(TRACKS, "TRACKS"), relabel(GALS, "GALS")

# ── conventions of the KS notebook (Parts 0 / 4b / 4c / 5b / 5c) ──
FIDUCIAL_AP = "R50_H2"
HE_FACTOR   = kl.HE_FACTOR
STAGES_DRAW = ["sft", "qt", "end"]
END_STAGES  = ("anchor", "end")
STAGE_MARKER = {"sft": "^", "qt": "s", "end": "o", "anchor": "o"}
STAGE_LABEL  = {"sft": "SFT (quench start)", "qt": "QT (quench end)", "end": "track end"}
END_DEF = r"track end = the anchor, or the last snapshot with $M_{\rm H_2}/M_\star > 10^{-4}$ when the anchor holds no H$_2$"
RELATIONS_DRAW, RELATIONS_FAINT = ("B08",), ("K98",)
KS_REF  = RELATIONS_DRAW[0]
KS_BAND = float(kl.RELATIONS[KS_REF]["sig"])
AP_TITLE = {"R50_H2": r"$r<R_{50}({\rm H_2})$"}
XLIM, YLIM = (-0.5, 3.6), (-4.6, 1.1)
C_OBS, C_OBS_EDGE = "#e7298a", "#3b0f2a"
NMIN_BIN = 4                                       # uncensored galaxies needed for a stage median inside one bin
Z_PANELS = [(0.0, 0.55, r"$z\leq0.5$"), (0.55, 1.05, r"$0.7\leq z\leq1$"), (1.05, 2.5, r"$1.15\leq z\leq2$")]
Z_ALL    = (-1.0, 99.0, r"all anchors, $0.3\leq z\leq2$")
PRES_COLS = [Z_ALL] + Z_PANELS
REGION_GROUPS = ["below", "within", "above"]
REGION_COLOR  = {"below": "#2166ac", "within": "#4dac26", "above": "#d6604d", "undef": "0.35"}
REGION_SHORT  = {"below": f"below {KS_REF}", "within": f"on {KS_REF}", "above": f"above {KS_REF}", "undef": "undetermined"}
DT_GRID = np.arange(-2.0, 3.001, 0.25)             # quench clock t - t_QT [Gyr]
NMIN_GRID, NMIN_ECDF = 5, 4
NGAS_MIN = 10                                      # kappa_rot(gas) needs this many member gas particles
CLASS_ORDER = ["weak", "intermediate", "strong", "no_event"]
# class palette: three distinct hues (teal / amber / dark red) — replaces the sequential purples (too little contrast), 2026-08-29
CLASS_COLOR_PRES = {"weak": "#1b9e77", "intermediate": "#e08214", "strong": "#a50f15", "no_event": "0.85"}
CLASS_TEXT_PRES  = {"weak": "white", "intermediate": "0.15", "strong": "white", "no_event": "0.15"}
CLASS_NAME = {"weak": "weak", "intermediate": "intermediate", "strong": "strong", "no_event": "no event"}
C_SF = "0.5"


# ── the KS plane ──
def draw_relations(ax, xs=None, tdep_lines=(0.1, 1.0, 10.0)):
    """The reference relation(s) with the published scatter band, the faint ones thin, + constant-t_dep lines. Returns legend items."""
    xs = np.linspace(XLIM[0] - 0.5, XLIM[1] + 0.5, 80) if xs is None else xs
    items = []
    for t, ls in zip(tdep_lines, (":", "-", "--")):
        ax.plot(xs, xs + 6 - np.log10(t * 1e9), color="0.65", ls=ls, lw=0.9, zorder=1)
        ax.text(XLIM[1] - 0.05, XLIM[1] - 0.05 + 6 - np.log10(t * 1e9) - 0.25, rf"$t_{{\rm dep}}$={t:g} Gyr",
                fontsize=7.5, color="0.45", ha="right", rotation=42, rotation_mode="anchor")
    for key in RELATIONS_DRAW:
        rel = kl.RELATIONS[key]
        y = rel["A"] + rel["N"] * xs
        ax.fill_between(xs, y - rel["sig"], y + rel["sig"], color=rel["color"], alpha=0.10, lw=0, zorder=0)
        ax.plot(xs, y, color=rel["color"], lw=1.4, ls=rel["ls"], zorder=2)
        items.append((Line2D([], [], color=rel["color"], lw=1.4, ls=rel["ls"]), rf"{rel['label']} $\pm$ {rel['scat_label']}"))
    for key in RELATIONS_FAINT:
        rel = kl.RELATIONS[key]
        ax.plot(xs, rel["A"] + rel["N"] * xs, color=rel["color"], lw=0.9, ls=rel["ls"], alpha=0.55, zorder=1)
        items.append((Line2D([], [], color=rel["color"], lw=0.9, ls=rel["ls"], alpha=0.55), rf"{rel['label']} (for reference)"))
    return items


def overlay_obs(ax, alpha=1.0):
    """ALMA-C11 points: filled = size measured (asymmetric errors), open + slope-1 arrow = unresolved (both Sigma lower limits)."""
    for _, r in OBS_S.iterrows():
        x, y = float(r["logSigmaH2"]), float(r["logSigmaSFR"])
        if bool(r["sigma_is_ll"]):
            ax.annotate("", xy=(x + 0.3, y + 0.3), xytext=(x, y),
                        arrowprops=dict(arrowstyle="-|>", color=C_OBS_EDGE, lw=1.2, mutation_scale=10, alpha=alpha), zorder=8)
            ax.scatter(x, y, s=110, marker="o", facecolors="white", edgecolors=C_OBS_EDGE, linewidths=1.4, alpha=alpha, zorder=9)
            ax.scatter(x, y, s=40, marker="o", c=C_OBS, edgecolors="none", alpha=alpha, zorder=10)
        else:
            elo = r["logSigmaSFR_elo"] if np.isfinite(r["logSigmaSFR_elo"]) else 0.0
            ehi = r["logSigmaSFR_ehi"] if np.isfinite(r["logSigmaSFR_ehi"]) else 0.0
            elo = min(elo, y - YLIM[0])
            xe = r["logSigmaH2_err"] if np.isfinite(r["logSigmaH2_err"]) else 0.0
            ax.errorbar(x, y, xerr=xe, yerr=[[elo], [ehi]], fmt="none", ecolor=C_OBS_EDGE, elinewidth=1.0, alpha=alpha, zorder=8)
            ax.scatter(x, y, s=110, marker="o", c=C_OBS, edgecolors=C_OBS_EDGE, linewidths=1.4, alpha=alpha, zorder=9)
    return [(Line2D([], [], marker="o", ls="", ms=9, mfc=C_OBS, mec=C_OBS_EDGE), rf"ALMA-C11 QGs, $z\approx0.4$ (N={len(OBS_S)}; size measured)"),
            (Line2D([], [], marker="o", ls="", ms=9, mfc="white", mec=C_OBS_EDGE), r"ALMA-C11 unresolved: $\Sigma$ lower limits")]


def ks_axes(ax, ap_label=FIDUCIAL_AP, sfr_note="SFR over 100 Myr (stars)"):
    conv = r"$0.5M/\pi R_{50}^2$" if ap_label.startswith("R50") else r"$M(<r)/\pi r^2$"
    ax.set_xlabel(r"$\log(\Sigma_{\rm H_2}/M_\odot\,{\rm pc}^{-2})$  [" + AP_TITLE[ap_label] + ", " + conv + rf", $\times{HE_FACTOR:g}$ He]")
    ax.set_ylabel(r"$\log(\Sigma_{\rm SFR}/M_\odot\,{\rm yr}^{-1}\,{\rm kpc}^{-2})$  [" + sfr_note + "]")
    ax.set_xlim(XLIM); ax.set_ylim(YLIM)
    ax.tick_params(direction="in", top=True, right=True); ax.grid(False)


def _sel_ap(ap_label, pop="Q"):
    return TRACKS[(_s(TRACKS["ap_label"]) == ap_label) & (_s(TRACKS["pop"]) == pop)]


def stage_medians(T, xcol="logSigmaH2", ycol="logSigmaSFR", stages=STAGES_DRAW, nmin=NMIN_BIN):
    """Per-stage median of x and y over the uncensored rows (keys x, y when >= nmin) and over ALL finite rows with the censored SFRs
    at their one-particle floor (x_all, y_all: an upper limit on the stage median)."""
    stg, ul = _s(T["stage"]), np.asarray(T["is_ul"], bool)
    x, y = np.asarray(T[xcol], float), np.asarray(T[ycol], float)
    out = []
    for st in stages:
        m_all = (stg == st) & np.isfinite(x) & np.isfinite(y)
        m = m_all & ~ul
        d = dict(stage=st, n=int(m.sum()), n_ul=int(((stg == st) & np.isfinite(x) & ul).sum()), n_all=int(m_all.sum()))
        if m_all.sum() >= nmin:
            d.update(x_all=np.median(x[m_all]), y_all=np.median(y[m_all]))
        if m.sum() >= nmin:
            d.update(x=np.median(x[m]), x16=np.percentile(x[m], 16), x84=np.percentile(x[m], 84),
                     y=np.median(y[m]), y16=np.percentile(y[m], 16), y84=np.percentile(y[m], 84))
        out.append(d)
    return out


def draw_median_track(ax, T, color, ycol="logSigmaSFR", lw=2.2, ls="-", zorder=6, ms=1.0):
    """Per-stage medians of T joined in stage order, stage symbols as markers; a stage with < NMIN_BIN uncensored galaxies but
    >= NMIN_BIN in total is drawn at its all-rows median: open marker + down arrow = an upper limit on the stage median."""
    meds = stage_medians(T, ycol=ycol)
    pts = [(m["x"], m["y"], m["stage"], False) if "x" in m else (m["x_all"], m["y_all"], m["stage"], True)
           for m in meds if ("x" in m) or ("x_all" in m)]
    if len(pts) >= 2:
        ax.plot([p[0] for p in pts], [p[1] for p in pts], ls=ls, color=color, lw=lw, alpha=0.9, zorder=zorder)
    for x, y, st, is_ul in pts:
        s = (110 if st in END_STAGES else 70) * ms
        if is_ul:
            ax.scatter(x, y, s=s, marker=STAGE_MARKER[st], facecolors="white", edgecolors=color, linewidths=1.4, zorder=zorder + 1)
            ax.annotate("", xy=(x, y - 0.35), xytext=(x, y), zorder=zorder + 1,
                        arrowprops=dict(arrowstyle="-|>", color=color, lw=1.2, mutation_scale=9, shrinkA=4, shrinkB=0))
        else:
            ax.scatter(x, y, s=s, marker=STAGE_MARKER[st], c=[color], edgecolors="k" if st in END_STAGES else "none", linewidths=0.9, zorder=zorder + 1)
    return meds


# ── the galaxies and the histories on the t - t_QT clock ──
GK = _s(GALS["gkey"])
AZ = np.asarray(GALS["anchor_z"], float)


def members(zlo, zhi, region=None, need_clock=False):
    """Boolean mask over GALS: anchor z in (zlo, zhi], optional region of the track end, optional detected quench event."""
    m = (AZ > zlo) & (AZ <= zhi)
    if region is not None:
        m &= _s(GALS["region"]) == region
    if need_clock:
        m &= np.asarray(GALS["has_clock"], bool)
    return m


_hk = _s(HIST["gkey"])
_hp = _s(HIST["pop"])
_haz = np.asarray(HIST["anchor_z"], float)
_tq = dict(zip(GK, np.asarray(GALS["t_qt_gyr"], float)))
HIST["dt_qt_gyr"] = np.asarray(HIST["t_gyr"], float) - np.array([_tq.get(g, np.nan) for g in _hk])
_o = np.argsort(_hk, kind="stable")
_ug, _st0, _cnt = np.unique(_hk[_o], return_index=True, return_counts=True)
HROWS = {g: _o[s:s + c] for g, s, c in zip(_ug, _st0, _cnt)}                           # gkey -> its HIST rows
HKEY = {(g, int(s)): i for i, (g, s) in enumerate(zip(_hk, np.asarray(HIST["snap"])))}  # (gkey, snap) -> HIST row


def hist_values(col):
    return np.asarray(HIST[col], float) if isinstance(col, str) else np.asarray(col, float)


def hist_at_stage(stage, col):
    """`col` of the HIST row at the galaxy's `stage` snapshot, aligned to GALS (NaN without that row)."""
    v = hist_values(col)
    sn = np.asarray(GALS[f"snap_{stage}"], float)
    return np.array([v[HKEY[(g, int(s))]] if (np.isfinite(s) and (g, int(s)) in HKEY) else np.nan for g, s in zip(GK, sn)])


def grid_tracks(gkeys, col):
    """(n_gal, n_grid) array: `col` of every galaxy's history resampled onto DT_GRID (NaN outside its coverage)."""
    v, dt = hist_values(col), np.asarray(HIST["dt_qt_gyr"], float)
    out = np.full((len(gkeys), len(DT_GRID)), np.nan)
    for i, g in enumerate(gkeys):
        r = HROWS.get(g)
        if r is not None:
            out[i] = kl.interp_track(dt[r], v[r], DT_GRID)
    return out


def _sf_mask(zlo, zhi):
    return (_hp == "SF") & (_haz > zlo) & (_haz <= zhi)


def sf_band(ax, zlo, zhi, col):
    """Median (dashed) and 16-84 % band of `col` over the SF controls whose anchor lies in (zlo, zhi]; returns the values."""
    v = hist_values(col)[_sf_mask(zlo, zhi)]
    v = v[np.isfinite(v)]
    if len(v) >= NMIN_ECDF:
        ax.axhspan(np.percentile(v, 16), np.percentile(v, 84), color=C_SF, alpha=0.13, lw=0, zorder=0)
        ax.axhline(np.median(v), color=C_SF, lw=1.0, ls="--", zorder=1)
    return v


with np.errstate(divide="ignore", invalid="ignore"):
    _md, _ms, _mg, _ng = (np.asarray(HIST[c], float) for c in ("mdust", "mstar", "mgas", "ngas"))
    AGE_MW    = np.asarray(HIST["age_mw"], float)                                     # mass-weighted stellar age [Gyr]
    LOG_FDUST = np.where((_md > 0) & (_ms > 0), np.log10(_md / _ms), np.nan)
    LOG_DGR   = np.where((_md > 0) & (_mg > 0), np.log10(_md / _mg), np.nan)          # dust-to-gas ratio (all member gas)
    KAPPA_GAS = np.where(_ng >= NGAS_MIN, np.asarray(HIST["kappa_gas"], float), np.nan)
    BT_STAR   = np.asarray(HIST["bt_star"], float)


# ── presentation grids: no titles, no grid, column names above the first row, thin frame around the first column ──
def pres_grid(nrows, names, height=3.1, width=4.4):
    fig, axs = plt.subplots(nrows, len(names), figsize=(width * len(names), height * nrows), squeeze=False)
    for ax in axs.ravel():
        ax.grid(False)
        ax.tick_params(direction="in", top=True, right=True)
    for j, name in enumerate(names):
        axs[0, j].text(0.0, 1.02, name, transform=axs[0, j].transAxes, ha="left", va="bottom", fontsize=9.5, color="0.25")
    return fig, axs


def pres_finish(fig, axs, stem, pad=0.008, frame=True):
    """Common limits along every row (columns comparable), thin frame around the first column; paper_save."""
    for row in axs:
        row_ = [ax for ax in row if ax.has_data()]
        if len(row_) <= 1:
            continue
        if all(getattr(a, "_auto_y", False) for a in row_):
            logy = row_[0].get_yscale() == "log"
            ys = np.concatenate([np.asarray(l.get_ydata(), float) for a in row_ for l in a.get_lines()
                                 if len(l.get_ydata()) > 1 and l.get_transform() == a.transData])
            ys = ys[np.isfinite(ys) & ((ys > 0) if logy else True)]
            if logy:
                ys = np.log10(ys)
            if ys.size:
                lo, hi = ys.min(), ys.max()
                lo, hi = lo - 0.15 * (hi - lo), hi + 0.15 * (hi - lo)
                for a in row_:
                    a.set_ylim(*((10 ** lo, 10 ** hi) if logy else (lo, hi)))
        else:
            lims = np.array([a.get_ylim() for a in row_])
            lo, hi = (lims.min(), lims.max()) if lims[0, 0] <= lims[0, 1] else (lims.max(), lims.min())
            for a in row_:
                a.set_ylim(lo, hi)
        lims = np.array([a.get_xlim() for a in row_])
        for a in row_:
            a.set_xlim(lims.min(), lims.max())
    plt.tight_layout(w_pad=1.2, h_pad=1.0)
    if frame:
        fig.canvas.draw()
        inv = fig.transFigure.inverted()
        bbs = [ax.get_tightbbox(fig.canvas.get_renderer()).transformed(inv) for ax in axs[:, 0]]
        x0, x1 = min(b.x0 for b in bbs) - pad, max(b.x1 for b in bbs) + pad
        y0, y1 = min(b.y0 for b in bbs) - pad, max(b.y1 for b in bbs) + pad * 1.6
        fig.add_artist(Rectangle((x0, y0), x1 - x0, y1 - y0, transform=fig.transFigure, fill=False, ec="0.4", lw=0.9, zorder=20))
    paper_save(fig, stem)
    plt.show()


def _legend(ax, loc="best", **kw):
    kw.setdefault("handlelength", 1.8)
    ax.legend(frameon=False, fontsize=8, loc=loc, **kw)


def pres_clock(ax, zlo, zhi, col, ylabel, ylim=None, legend=True):
    for reg in REGION_GROUPS:
        gk = GK[members(zlo, zhi, reg, need_clock=True)]
        if not len(gk):
            continue
        st = kl.grid_stats(grid_tracks(gk, col), NMIN_GRID)
        ok = np.isfinite(st["med"])
        if not ok.any():
            continue
        ax.plot(DT_GRID[ok], st["med"][ok], color=REGION_COLOR[reg], lw=2.2, zorder=4, label=f"{REGION_SHORT[reg]} (N={len(gk)})")
        ax.fill_between(DT_GRID[ok], st["p16"][ok], st["p84"][ok], color=REGION_COLOR[reg], alpha=0.12, lw=0, zorder=2)
    v = sf_band(ax, zlo, zhi, col)
    if len(v) >= NMIN_ECDF:
        ax.plot([], [], color=C_SF, lw=1.0, ls="--", label=f"SF controls (N={len(v)})")
    ax.axvline(0, color="0.5", lw=0.8, ls=":")
    ax.set_xlim(DT_GRID[0], DT_GRID[-1]); ax.set_xlabel(r"$t - t_{\rm QT}$ [Gyr]"); ax.set_ylabel(ylabel)
    if ylim is not None:
        ax.set_ylim(*ylim)
    ax._auto_y = ylim is None
    if legend:
        _legend(ax)


def pres_ecdf(ax, zlo, zhi, values, xlabel, sf_values=None, xlim=None, legend=True):
    vals = np.asarray(values, float)
    for reg in REGION_GROUPS:
        m = members(zlo, zhi, reg) & np.isfinite(vals)
        if m.sum() < NMIN_ECDF:
            continue
        x, F = kl.ecdf(vals[m])
        ax.step(np.r_[x[0], x], np.r_[0, F], where="post", color=REGION_COLOR[reg], lw=2.0, zorder=4, label=f"{REGION_SHORT[reg]} (N={int(m.sum())})")
        ax.plot([np.median(vals[m])], [0.5], marker="|", ms=12, mew=2.0, color=REGION_COLOR[reg], zorder=5)
    if sf_values is not None:
        sv = np.asarray(sf_values, float)
        sv = sv[np.isfinite(sv)]
        if len(sv) >= NMIN_ECDF:
            x, F = kl.ecdf(sv)
            ax.step(np.r_[x[0], x], np.r_[0, F], where="post", color=C_SF, lw=1.2, ls="--", zorder=3, label=f"SF controls (N={len(sv)})")
    ax.set_ylim(0, 1.02); ax.set_ylabel("cumulative fraction"); ax.set_xlabel(xlabel)
    if xlim is not None:
        ax.set_xlim(*xlim)
    if legend:
        _legend(ax, loc="lower right")


print(f"TRACKS: {len(TRACKS)} rows; GALS: {len(GALS)} Q galaxies; HIST: {len(HIST)} rows ({len(np.unique(_hk[_hp == 'Q']))} Q + "
      f"{len(np.unique(_hk[_hp == 'SF']))} SF); observed: {len(OBS_S)} ALMA-C11 sources with a surface density")
print(f"{'column':>28s} " + " ".join(f"{REGION_SHORT[r]:>10s}" for r in REGION_GROUPS) + f" {'undef':>6s} {'no clock':>9s}")
for zlo, zhi, name in PRES_COLS:
    m = members(zlo, zhi)
    print(f"{name.replace('$', '').replace(chr(92) + 'leq', '<='):>28s} " + " ".join(f"{int(members(zlo, zhi, r).sum()):10d}" for r in REGION_GROUPS)
          + f" {int((m & (_s(GALS['region']) == 'undef')).sum()):6d} {int((m & ~np.asarray(GALS['has_clock'], bool)).sum()):9d}")
print("AGN classes of the Q galaxies:", dict(zip(*np.unique(_s(GALS["agn_class"]), return_counts=True))))

In [ ]:
# ── Part 2b — figure A: AGN class composition of the KS regions + the rule's coupling strength ──
PRES_NAME = [Z_ALL[2]] + [t for _, _, t in Z_PANELS]
_cls = _s(GALS["agn_class"])
fig, axs = pres_grid(2, PRES_NAME, height=3.1)
for j, (zlo, zhi, _) in enumerate(PRES_COLS):
    ax = axs[0, j]
    for k, reg in enumerate(REGION_GROUPS):
        m = members(zlo, zhi, reg)
        n, left = int(m.sum()), 0.0
        for c in CLASS_ORDER:
            f = float((m & (_cls == c)).sum()) / n if n else 0.0
            ax.barh(k, f, left=left, color=CLASS_COLOR_PRES[c], edgecolor="white", lw=0.6, height=0.62, label=CLASS_NAME[c] if k == 0 else None)
            if f >= 0.08:
                ax.text(left + f / 2, k, f"{100 * f:.0f}%", ha="center", va="center", fontsize=7.5, color=CLASS_TEXT_PRES[c])
            left += f
        ax.text(1.02, k, f"N={n}", va="center", fontsize=8)
    ax.set_yticks(range(len(REGION_GROUPS))); ax.set_yticklabels([REGION_SHORT[r] for r in REGION_GROUPS], fontsize=8.5)
    for k, r in enumerate(REGION_GROUPS):
        ax.get_yticklabels()[k].set_color(REGION_COLOR[r])
    ax.set_xlim(0, 1.2); ax.set_ylim(3.35, -0.55); ax.set_xticks(np.linspace(0, 1, 6)); ax.set_xlabel("fraction of the group")
    ax.tick_params(axis="y", length=0)
    if j == 0:
        _legend(ax, loc="lower left", ncol=4, columnspacing=1.0, handlelength=1.2)
    pres_ecdf(axs[1, j], zlo, zhi, np.asarray(GALS[COUP_VAR], float), COUP_LABEL, legend=(j == 0))
pres_finish(fig, axs, f"ks_agn_{FIDUCIAL_AP}")

In [ ]:
# ── Part 2c — figure B: dust fraction vs age, dust fraction and dust-to-gas ratio on the clock, DGR at the end ──
_end_age, _end_fd, _end_dgr = hist_at_stage("end", AGE_MW), hist_at_stage("end", LOG_FDUST), hist_at_stage("end", LOG_DGR)
_TICK = np.isin(np.round(DT_GRID, 3), np.round(np.arange(DT_GRID[0], DT_GRID[-1] + 1e-6, 0.5), 3))
fig, axs = pres_grid(4, PRES_NAME, height=3.2)
for j, (zlo, zhi, _) in enumerate(PRES_COLS):
    ax = axs[0, j]
    msf = _sf_mask(zlo, zhi)
    ax.scatter(AGE_MW[msf], LOG_FDUST[msf], s=7, c=C_SF, alpha=0.35, lw=0, zorder=1, label=f"SF controls (N={int(msf.sum())})")
    for reg in REGION_GROUPS:
        m_all = members(zlo, zhi, reg)
        gk = GK[m_all & np.asarray(GALS["has_clock"], bool)]
        col = REGION_COLOR[reg]
        ax.scatter(_end_age[m_all], _end_fd[m_all], s=9, c=[col], alpha=0.45, lw=0, zorder=2)
        if len(gk):
            sa, sf_ = kl.grid_stats(grid_tracks(gk, AGE_MW), NMIN_GRID), kl.grid_stats(grid_tracks(gk, LOG_FDUST), NMIN_GRID)
            ok = np.isfinite(sa["med"]) & np.isfinite(sf_["med"])
            if ok.sum() >= 2:
                ax.plot(sa["med"][ok], sf_["med"][ok], color=col, lw=2.2, zorder=4, label=f"{REGION_SHORT[reg]} (N={int(m_all.sum())})")
                ax.scatter(sa["med"][ok & _TICK], sf_["med"][ok & _TICK], s=12, c=[col], zorder=5)
        for st in STAGES_DRAW:
            a_st, f_st = hist_at_stage(st, AGE_MW)[m_all], hist_at_stage(st, LOG_FDUST)[m_all]
            okst = np.isfinite(a_st) & np.isfinite(f_st)
            if okst.sum() >= NMIN_ECDF:
                ax.scatter(np.median(a_st[okst]), np.median(f_st[okst]), s=90 if st in END_STAGES else 58, marker=STAGE_MARKER[st], c=[col],
                           edgecolors="k", linewidths=0.9, zorder=6)
    ax.set_xlabel("mass-weighted stellar age [Gyr]"); ax.set_ylabel(r"$\log(M_{\rm dust}/M_\star)$")
    ax.set_xlim(0, 9.5); ax.set_ylim(-6.2, -1.6)
    if j == 0:
        _legend(ax, loc="lower left")
    pres_clock(axs[1, j], zlo, zhi, LOG_FDUST, r"$\log(M_{\rm dust}/M_\star)$", legend=(j == 0))
    pres_clock(axs[2, j], zlo, zhi, LOG_DGR, r"$\log(M_{\rm dust}/M_{\rm gas})$", legend=(j == 0))
    pres_ecdf(axs[3, j], zlo, zhi, _end_dgr, r"$\log(M_{\rm dust}/M_{\rm gas})$ at the track end", sf_values=LOG_DGR[msf], legend=(j == 0))
_items = [(Line2D([], [], marker=STAGE_MARKER[st], ls="", ms=8, mfc="0.55", mec="k"), STAGE_LABEL[st]) for st in STAGES_DRAW]
_items.append((Line2D([], [], marker="o", ls="", ms=4, mfc="0.55", mec="none"), "median track, dots every 0.5 Gyr"))
axs[0, 1].legend([h for h, _ in _items], [l for _, l in _items], frameon=False, fontsize=8, loc="lower left", handlelength=1.8)
pres_finish(fig, axs, f"ks_dust_{FIDUCIAL_AP}")

In [ ]:
# ── Part 2d — figure C: gas rotation and stellar bulge fraction on the clock, each with its ECDF at the track end ──
_end_kg, _end_bt = hist_at_stage("end", KAPPA_GAS), hist_at_stage("end", BT_STAR)
fig, axs = pres_grid(4, PRES_NAME, height=3.1)
for j, (zlo, zhi, _) in enumerate(PRES_COLS):
    msf = _sf_mask(zlo, zhi)
    pres_clock(axs[0, j], zlo, zhi, KAPPA_GAS, r"$\kappa_{\rm rot}$ (gas)", ylim=(0, 1), legend=(j == 0))
    pres_ecdf(axs[1, j], zlo, zhi, _end_kg, r"$\kappa_{\rm rot}$ (gas) at the track end", sf_values=KAPPA_GAS[msf], xlim=(0, 1), legend=(j == 0))
    pres_clock(axs[2, j], zlo, zhi, BT_STAR, "B/T (stars)", ylim=(0, 1), legend=(j == 0))
    pres_ecdf(axs[3, j], zlo, zhi, _end_bt, "B/T (stars) at the track end", sf_values=BT_STAR[msf], xlim=(0, 1), legend=(j == 0))
pres_finish(fig, axs, f"ks_kinematics_{FIDUCIAL_AP}")

## Part 3 — the $A_V$ test: mass-limited quenched galaxies binned by their anchor attenuation on the KS plane, per AGN class

**Sample.** The quenched galaxies with $10.5 < \log M_\star/M_\odot < 11.2$ at the anchor (catalogue mass of the selection table),
a detected quench event (so a track exists) and an RT attenuation at the anchor. **$A_V$** is the dust_on / dust_off attenuation of
the m25 notebook (`annulus_av_allincl.fits`) in the aperture `AV_APERTURE` — the default is the **core** ($r < 3.2$ kpc, the
facet of the red-core analysis), **median over the four sightlines**; the fiducial-sightline column of `attenuation_vs_ism.fits`
is the fallback. **Bins**: `AV_EDGES = None` → terciles of the whole mass-limited sample, so the **same edges apply in every
column** (a class column can be dominated by one bin — that is information, not a bug); fixed edges (e.g. `(0.1, 0.5)` mag) are
the alternative. **Columns**: all classes together (thin frame), then weak / intermediate / strong; the `no_event` galaxies have
no SFT/QT and drop out. **Rows**: one KS panel per entry of `P3_ROWS` (default: all anchors — the mass-limited sample is
~90 galaxies, per-redshift rows are for orientation only) + the $A_V$ ECDF of the column's galaxies with the bin edges.

**What is drawn.** In every KS panel the SF controls of the same mass range at their anchor (grey cloud), the reference relation
with its scatter band, the constant-$t_{\rm dep}$ lines and (faded) the ALMA-C11 points; per $A_V$ bin, every galaxy's own track
SFT → QT → track end as a thin line (`SHOW_INDIVIDUAL`) and the **bin median track** thick (stage medians over the uncensored
galaxies, open marker + arrow where the median is an upper limit because too many galaxies have SFR = 0). The legend of every
panel carries the number of galaxies per bin in that panel. The per-(row, column, bin, stage) medians go to `paper_ks_av_bins.csv`.

**Reading it.** Same edges everywhere, so the question is twofold: (i) *within* a class, do dustier quenched galaxies end
closer to the relation (dust as a proxy for surviving cold gas) or below it (dust surviving the SFR decline)? (ii) *across* the
columns, does the AGN coupling class change that ordering — i.e. is the $A_V$ ordering a gas-content ordering or a feedback
ordering? The printed table gives the class × bin counts, the median $\log M_\star$ per bin (mass confound inside the narrow range)
and the KS p-values of the $A_V$ distributions between classes.

In [ ]:
# ── Part 3 — mass-limited quenched galaxies binned by their anchor A_V on the KS plane, one column per AGN coupling class ──
MASS_RANGE  = (10.5, 11.2)      # log M*/Msun at the anchor (selection-table catalogue mass)
AV_APERTURE = "ap3kpc"          # the core (r < 3.2 kpc; the 'CORE' facet of m25 Part 8); any ap* / ann* label of annulus_av_allincl.fits
AV_EDGES    = None              # None: terciles of the mass-limited sample (the same edges in every column) | fixed edges, e.g. (0.1, 0.5) [mag]
AV_COLORS   = ["#2c7bb6", "#fdae61", "#d7191c", "#7b3294"]
P3_CLASSES  = ["weak", "intermediate", "strong"]   # one column each (no_event galaxies have no SFT/QT -> no track)
P3_ROWS     = [Z_ALL]           # one KS row per entry; add Z_PANELS entries for per-redshift rows (thin)
SHOW_INDIVIDUAL = True          # thin lines: every galaxy's own track under its bin's median track
OBS_ALPHA   = 0.35              # the ALMA-C11 points (z ~ 0.4), faded
_AP_NAME = {"ap1kpc": r"$r<1$ kpc", "ap3kpc": r"$r<3.2$ kpc", "ap10kpc": r"$r<10$ kpc", "ap32kpc": r"$r<32$ kpc", "ap100kpc": r"$r<100$ kpc",
            "ann3kpc": r"$1<r<3.2$ kpc", "ann10kpc": r"$3.2<r<10$ kpc", "ann32kpc": r"$10<r<32$ kpc", "ann100kpc": r"$32<r<100$ kpc"}


def anchor_av():
    """{(snap, gal_id): A_V at the anchor} in AV_APERTURE: the median over the sightlines of annulus_av_allincl (m25 Part 8b), else the
    fiducial-sightline column of attenuation_vs_ism (m25 Part 7a)."""
    if os.path.exists(ANNULUS_AV_FITS):
        A = Table.read(ANNULUS_AV_FITS)
        A = A[_s(A["aperture"]) == AV_APERTURE]
        if not len(A):
            raise KeyError(f"{ANNULUS_AV_FITS} has no aperture {AV_APERTURE!r}")
        df = pd.DataFrame(dict(snap=np.asarray(A["snap"], int), gal_id=np.asarray(A["gal_id"], int), av=np.asarray(A["A_V"], float)))
        g = df.groupby(["snap", "gal_id"])["av"].median()
        return {(int(s), int(i)): float(v) for (s, i), v in g.items()}, f"median over {len(np.unique(_s(A['incl'])))} sightlines"
    A = Table.read(need(ATTEN_FITS, "powderday_flux_quenched_m25.ipynb Part 7a (or Part 8b for annulus_av_allincl.fits)"))
    col = f"A_V_{AV_APERTURE}"
    if col not in A.colnames:
        raise KeyError(f"{ATTEN_FITS} has no column {col!r}")
    return {(int(s), int(i)): float(v) for s, i, v in zip(A["snap"], A["gal_id"], np.asarray(A[col], float))}, "fiducial sightline"


AV_OF, AV_NOTE = anchor_av()
AV_LABEL = r"$A_V$ at the anchor [mag]"
AV_DEF = rf"$A_V$: dust_on / dust_off RT in the projected aperture {_AP_NAME.get(AV_APERTURE, AV_APERTURE)} at the anchor, {AV_NOTE}"
GALS["av_anchor"] = np.array([AV_OF.get(k, np.nan) for k in _keys(GALS)])
_lm, _cl, _av = np.asarray(GALS["log_mstar_anchor"], float), _s(GALS["agn_class"]), np.asarray(GALS["av_anchor"], float)
P3_MASS   = (_lm > MASS_RANGE[0]) & (_lm < MASS_RANGE[1])
P3_SAMPLE = P3_MASS & np.isfinite(_av) & np.isin(_cl, P3_CLASSES) & np.asarray(GALS["has_clock"], bool)
AV_EDGE = np.quantile(_av[P3_SAMPLE], [1 / 3, 2 / 3]) if AV_EDGES is None else np.asarray(AV_EDGES, float)
AV_BIN = np.where(P3_SAMPLE, np.searchsorted(AV_EDGE, np.where(np.isfinite(_av), _av, -1.0), side="right"), -1)
NB_AV = len(AV_EDGE) + 1
AV_BIN_LABEL = [rf"$A_V \leq {AV_EDGE[0]:.2f}$"] + [rf"${AV_EDGE[k - 1]:.2f} < A_V \leq {AV_EDGE[k]:.2f}$" for k in range(1, NB_AV - 1)] + [rf"$A_V > {AV_EDGE[-1]:.2f}$"]
B_OF = dict(zip(GK, AV_BIN))
print(f"A_V source: {AV_NOTE} in {AV_APERTURE}; Q galaxies with an A_V: {int(np.isfinite(_av).sum())}/{len(GALS)}")
print(f"mass-limited sample {MASS_RANGE[0]} < log M* < {MASS_RANGE[1]}: {int(P3_MASS.sum())} Q galaxies, of which {int(P3_SAMPLE.sum())} with a quench event, "
      f"an A_V and a class in {P3_CLASSES} (dropped: {dict(zip(*np.unique(_cl[P3_MASS & ~P3_SAMPLE], return_counts=True)))})")
print(f"A_V bins ({'terciles of the sample' if AV_EDGES is None else 'fixed edges'}): edges {np.round(AV_EDGE, 3)} mag")
print(f"{'class':>13s} " + " ".join(f"{'bin ' + str(k):>10s}" for k in range(NB_AV)) + f" {'N':>4s} {'med A_V':>8s} {'med logM*':>10s}")
for c in ["all"] + P3_CLASSES:
    m = P3_SAMPLE & ((_cl == c) if c != "all" else True)
    print(f"{c:>13s} " + " ".join(f"{int((m & (AV_BIN == k)).sum()):10d}" for k in range(NB_AV)) + f" {int(m.sum()):4d} {np.median(_av[m]):8.3f} {np.median(_lm[m]):10.2f}")
print("per bin (all classes): median log M* = " + ", ".join(f"{np.median(_lm[P3_SAMPLE & (AV_BIN == k)]):.2f}" for k in range(NB_AV))
      + "; median anchor z = " + ", ".join(f"{np.median(AZ[P3_SAMPLE & (AV_BIN == k)]):.2f}" for k in range(NB_AV)))
for a, b in (("weak", "strong"), ("weak", "intermediate"), ("intermediate", "strong")):
    va, vb = _av[P3_SAMPLE & (_cl == a)], _av[P3_SAMPLE & (_cl == b)]
    if len(va) >= NMIN_ECDF and len(vb) >= NMIN_ECDF:
        print(f"A_V distributions, {a} vs {b}: KS p = {ks_2samp(va, vb).pvalue:.2g} (medians {np.median(va):.3f} / {np.median(vb):.3f} mag)")

# the SF controls of the same mass range (anchor rows of the fiducial aperture)
_SF = _sel_ap(FIDUCIAL_AP, "SF")
_SF = _SF[(np.asarray(_SF["log_mstar_anchor"], float) > MASS_RANGE[0]) & (np.asarray(_SF["log_mstar_anchor"], float) < MASS_RANGE[1])]
_TQ = _sel_ap(FIDUCIAL_AP, "Q")
_TQ = _TQ[np.isin(_s(_TQ["stage"]), STAGES_DRAW)]
_tq_gk, _tq_st = _s(_TQ["gkey"]), _s(_TQ["stage"])
_stage_rank = {st: i for i, st in enumerate(STAGES_DRAW)}


def av_panel(ax, zlo, zhi, cls, obs_alpha=OBS_ALPHA):
    """KS panel of the mass-limited galaxies with anchor z in (zlo, zhi] and class `cls` (None = all P3_CLASSES): the SF cloud, then per
    A_V bin the individual tracks (thin) and the bin median track (thick). Returns [(bin, n_gal, meds), ...]."""
    m = P3_SAMPLE & (AZ > zlo) & (AZ <= zhi) & ((_cl == cls) if cls else np.ones(len(GALS), bool))
    gk_in = set(GK[m])
    sfz = np.asarray(_SF["anchor_z"], float)
    S = _SF[(sfz > zlo) & (sfz <= zhi)]
    ax.scatter(np.asarray(S["logSigmaH2"], float), np.asarray(S["logSigmaSFR"], float), s=7, c="0.6", alpha=0.3, lw=0, zorder=2)
    out = []
    for k in range(NB_AV):
        gks = sorted(g for g in gk_in if B_OF[g] == k)
        T = _TQ[np.isin(_tq_gk, gks)]
        if SHOW_INDIVIDUAL:
            for g in gks:
                r = T[_s(T["gkey"]) == g]
                r = r[np.argsort([_stage_rank[s] for s in _s(r["stage"])])]
                x, y = np.asarray(r["logSigmaH2"], float), np.asarray(r["logSigmaSFR"], float)
                ok = np.isfinite(x) & np.isfinite(y)
                if ok.sum() >= 2:
                    ax.plot(x[ok], y[ok], "-", color=AV_COLORS[k], lw=0.7, alpha=0.35, zorder=3)
        meds = draw_median_track(ax, T, AV_COLORS[k]) if len(gks) else []
        out.append((k, len(gks), meds))
    draw_relations(ax)
    overlay_obs(ax, alpha=obs_alpha)
    ks_axes(ax)
    ax.legend([Line2D([], [], color=AV_COLORS[k], lw=2.2, marker="o", mec="k", ms=6) for k, _, _ in out],
              [f"{AV_BIN_LABEL[k]}  (N={n})" for k, n, _ in out], loc="lower right", frameon=False, fontsize=8, handlelength=1.8)
    return out


COL_NAMES = ["all classes"] + P3_CLASSES
fig, axs = pres_grid(len(P3_ROWS) + 1, [""] * len(COL_NAMES), height=4.6, width=4.9)
_rows_csv = []
for i, (zlo, zhi, zname) in enumerate(P3_ROWS):
    for j, cname in enumerate(COL_NAMES):
        cls = None if j == 0 else cname
        out = av_panel(axs[i, j], zlo, zhi, cls)
        n_col = sum(n for _, n, _ in out)
        axs[i, j].text(0.0, 1.02, f"{cname} (N={n_col}), {zname}", transform=axs[i, j].transAxes, ha="left", va="bottom", fontsize=9.5, color="0.25")
        for k, n, meds in out:
            for md_ in meds:
                _rows_csv.append(dict(z_row=zname.replace("$", ""), column=cname, av_bin=k, av_bin_label=AV_BIN_LABEL[k].replace("$", ""), n_gal=n,
                                      stage=md_["stage"], n_uncensored=md_["n"], n_censored=md_["n_ul"],
                                      logSigmaH2=md_.get("x", np.nan), logSigmaSFR=md_.get("y", np.nan),
                                      logSigmaH2_all=md_.get("x_all", np.nan), logSigmaSFR_all_upper=md_.get("y_all", np.nan)))
# last row: the A_V distribution of every column's galaxies (all anchors), SF controls of the same mass range dashed, the bin edges dotted
_sf_av = np.array([AV_OF.get(k, np.nan) for k in zip(np.asarray(_SF["anchor_snap"], int), np.asarray(_SF["gal_id"], int))])
for j, cname in enumerate(COL_NAMES):
    ax = axs[-1, j]
    m = P3_SAMPLE & ((_cl == cname) if j else np.ones(len(GALS), bool))
    x, F = kl.ecdf(_av[m])
    ax.step(np.r_[x[0], x], np.r_[0, F], where="post", color="k", lw=2.0, zorder=4, label=f"{cname} (N={int(m.sum())})")
    ax.plot([np.median(_av[m])], [0.5], marker="|", ms=12, mew=2.0, color="k", zorder=5)
    sv = _sf_av[np.isfinite(_sf_av)]
    if len(sv) >= NMIN_ECDF:
        xs_, Fs_ = kl.ecdf(sv)
        ax.step(np.r_[xs_[0], xs_], np.r_[0, Fs_], where="post", color=C_SF, lw=1.2, ls="--", zorder=3, label=f"SF controls, same mass range (N={len(sv)})")
    xlo = max(1e-3, 0.5 * np.nanmin(_av[P3_SAMPLE & (_av > 0)]))
    xhi = 1.5 * max(np.nanmax(_av[P3_SAMPLE]), np.nanmax(sv) if len(sv) else 0.0)
    for e in AV_EDGE:
        ax.axvline(e, color="0.4", lw=0.8, ls=":")
    for k in range(NB_AV):
        ax.axvspan(AV_EDGE[k - 1] if k else xlo, AV_EDGE[k] if k < NB_AV - 1 else xhi, color=AV_COLORS[k], alpha=0.08, lw=0, zorder=0)
    ax.set_xscale("log"); ax.set_xlim(xlo, xhi)
    ax.set_ylim(0, 1.02); ax.set_xlabel(AV_LABEL); ax.set_ylabel("cumulative fraction")
    _legend(ax, loc="lower right")
_items = [(Line2D([], [], marker=STAGE_MARKER[st], ls="", ms=8, mfc="0.4", mec="k" if st in END_STAGES else "none"), STAGE_LABEL[st]) for st in STAGES_DRAW]
_items += [(Line2D([], [], color="0.4", lw=2.2), "bin median track (stages in time order)"),
           (Line2D([], [], color="0.4", lw=0.7, alpha=0.5), "individual galaxy tracks (SFR = 0 epochs at the one-particle floor)"),
           (Line2D([], [], marker="o", ls="", ms=8, mfc="white", mec="0.4", mew=1.4), rf"open + $\downarrow$: < {NMIN_BIN} uncensored galaxies, median with SFR = 0 at the floor (upper limit)"),
           (Line2D([], [], marker="o", ls="", ms=5, mfc="0.6", mec="none"), f"SF controls, {MASS_RANGE[0]} < log $M_\\star$ < {MASS_RANGE[1]}, at their anchor")]
_rel = [(Line2D([], [], color=kl.RELATIONS[k]["color"], lw=1.4 if k in RELATIONS_DRAW else 0.9, ls=kl.RELATIONS[k]["ls"], alpha=1 if k in RELATIONS_DRAW else 0.55),
         rf"{kl.RELATIONS[k]['label']} $\pm$ {kl.RELATIONS[k]['scat_label']}" if k in RELATIONS_DRAW else rf"{kl.RELATIONS[k]['label']} (for reference)")
        for k in RELATIONS_DRAW + RELATIONS_FAINT]
_items += _rel + [(Line2D([], [], marker="o", ls="", ms=9, mfc=C_OBS, mec=C_OBS_EDGE), rf"ALMA-C11 QGs, $z\approx0.4$ (N={len(OBS_S)})")]
# the legend hangs below the axes (upper center at y = 0; bbox_inches='tight' keeps it); its title is the caption of the conventions
_caption = (f"{AV_DEF}; bins: the same edges in every column ({'terciles of the mass-limited sample' if AV_EDGES is None else 'fixed edges'}); "
            f"{END_DEF}")
fig.legend([h for h, _ in _items], [l for _, l in _items], loc="upper center", ncol=3, frameon=False, fontsize=8.5, bbox_to_anchor=(0.5, 0.0),
           title=_caption, title_fontsize=8.5)
pres_finish(fig, axs, f"ks_av_bins_by_class_{FIDUCIAL_AP}")
AV_BINS = pd.DataFrame(_rows_csv)
AV_BINS.to_csv(os.path.join(PAPERDIR, "paper_ks_av_bins.csv"), index=False)
print(f"bin medians -> {os.path.join(PAPERDIR, 'paper_ks_av_bins.csv')} ({len(AV_BINS)} rows)")
print(AV_BINS[AV_BINS["z_row"] == AV_BINS["z_row"].iloc[0]].pivot_table(index=["column", "av_bin"], columns="stage", values="logSigmaSFR", aggfunc="first")
      .reindex(columns=STAGES_DRAW).round(2).to_string())

## Part 4 — dust and rotation across the annuli, per AGN coupling class

Two figures of the m25 notebook's Part 8 (T4 / T8) re-drawn from its caches, in the presentation grammar of Parts 2–3: no
scatter, no rank statistics, one running median per group with a galaxy-bootstrap band, the class palette of the other figures.

* **Figure D — the annulus age clock** (m25 8k): $A_V$ (RT dust_on / dust_off), $M_{\rm dust}/M_\star$ and
  $\Sigma_{\rm dust}$ of each projected annulus (log-scaled axes, linear values) against the mass-weighted stellar age of the
  *same* annulus (m25 7e), one colour per annulus inside → out, one column per class (+ the star-forming controls). Rows are
  (galaxy, sightline, annulus): age and the dust quantities are projected, so each sightline is one measurement; the band
  resamples galaxies.
* **Figure E — the radial ladder per class** (m25 8f / 8j / 8p4): the per-galaxy value (median of its four sightlines) in the
  core disc, the outskirt annulus and the 10–32 kpc annulus, summarised per quenched class as the median with its
  galaxy-bootstrap 16–84 % interval; the zones run down the *y* axis, the quantity along *x*, one narrow panel per quantity
  ($A_V$, $M_{\rm dust}/M_\star$, $\Sigma_{\rm dust}$, stellar age, $\kappa_{\rm rot}^{\rm gas}$) — the tilt of a line is the
  radial gradient, the horizontal offsets between lines are the class differences. $\kappa_{\rm rot}^{\rm gas}$ is the Sales+12
  rotation support of the gas in the same (spherical) rung / shell (m25 8j0), sightline-independent.
* **Figure F — the radial trend per class** (Part 4d, `paper_radial_profiles_{mgt10p25,mlt10p25}`): the four disjoint annuli
  (0–1, 1–3.2, 3.2–10, 10–32 kpc) along $x$, one panel per quantity — $A_V$, $M_{\rm dust}/M_\star$, $M_{\rm gas}/M_\star$,
  $M_{\rm H_2}/M_\star$ (each annulus' own stellar mass), stellar age, $\kappa_{\rm rot}$ of the gas / H$_2$ / stars — one line per
  quenched class plus the mass-matched SF controls (grey), median with a galaxy-bootstrap band, for the two Part 5 mass samples. It
  answers directly whether a class has a more attenuated core, a dustier or gas-richer one, a younger one, or just a more rotating one;
  a zero dust / gas / H$_2$ mass is dropped, not floored (`n_dropped` in `paper_radial_profiles.csv`).

In [ ]:
# ── Part 4a — the (galaxy, sightline, annulus) rows: RT A_V, surviving dust, stellar age and gas rotation (m25 Parts 7e / 8a / 8b / 8j0) ──
ANNULI      = ["ap1kpc", "ann3kpc", "ann10kpc", "ann32kpc"]   # disjoint annuli, inside -> out (m25 T7 vocabulary; add "ann100kpc" for the 32–100 kpc CGM shell)
ANN_NAME    = {"ap1kpc": "0–1 kpc", "ann3kpc": "1–3.2 kpc", "ann10kpc": "3.2–10 kpc", "ann32kpc": "10–32 kpc", "ann100kpc": "32–100 kpc",
               "ap3kpc": "0–3.2 kpc", "ap10kpc": "0–10 kpc", "ap32kpc": "0–32 kpc", "ap100kpc": "0–100 kpc"}
ANN_COLOR   = dict(zip(ANNULI, plt.cm.viridis(np.linspace(0.05, 0.85, len(ANNULI)))))
CORE_LAB, OUT_LAB = "ap3kpc", "ann10kpc"      # figure E: core = the 0–3.2 kpc disc (m25 CORE9), outskirt = the 3.2–10 kpc annulus (OUT9)
E_ZONES     = [CORE_LAB, OUT_LAB, "ann32kpc"]  # figure E: the radial zones down the y axis, inside -> out (contrast = first minus second)
E_ZONE_NAME = {CORE_LAB: f"core\n{ANN_NAME[CORE_LAB]}", OUT_LAB: f"outskirt\n{ANN_NAME[OUT_LAB]}", "ann32kpc": ANN_NAME["ann32kpc"], "ann100kpc": ANN_NAME["ann100kpc"]}
P4_CLASSES  = ["star_forming", "weak", "intermediate", "strong"]   # columns of figure D (no_event has no class)
E_CLASSES   = ["weak", "intermediate", "strong"]                   # lines of figure E: the quenched classes only
P4_MASS_RANGE = None                          # e.g. (10.5, 11.2): anchor log M* window (Q and SF alike); None = the whole sample
KIN_VAR     = "kappa_gas"                     # kappa_gas | kappa_H2 | kappa_star  (m25 8j0: spherical rungs / shells, sightline-independent)
NGAS_ANN_MIN, NSTAR_AP_MIN = 10, 20           # m25 8c / 7e sampling floors: Sigma_dust needs NGAS_ANN_MIN gas particles, M* and age NSTAR_AP_MIN stars
KAPPA_DISC  = 0.5                             # Sales+12: rotation-supported above this
N_AGE_BINS, NMIN_TRACK, NBOOT_TRACK = 5, 10, 500   # figure D running medians: equal-count bins, galaxies per bin, bootstrap draws
P4_COLOR    = dict(CLASS_COLOR_PRES, star_forming=C_SF)
P4_NAME     = dict(CLASS_NAME, star_forming="SF controls")
KIN_TEX     = {"kappa_gas": r"$\kappa_{\rm rot}^{\rm gas}$", "kappa_H2": r"$\kappa_{\rm rot}^{\rm H_2}$", "kappa_star": r"$\kappa_{\rm rot}^{\star}$"}[KIN_VAR]
KEY4        = ["snap", "gal_id", "incl", "aperture"]
# quantities: working column (dex for the dust ones), axis label, log-scaled axis (values drawn as 10**column), name in the CSVs
Y4 = [("lA", r"$A_V$ [mag]", True, "A_V"), ("lF", r"$M_{\rm dust}/M_\star$", True, "Mdust_over_Mstar"),
      ("lS", r"$\Sigma_{\rm dust}$ [M$_\odot$ kpc$^{-2}$]", True, "Sigma_dust")]
Q4 = Y4 + [("age", "stellar age [Gyr]", False, "age_Gyr"), ("kap", KIN_TEX, False, KIN_VAR)]


def _frame(path, made_by, cols):
    t = Table.read(need(path, made_by))
    d = {}
    for c in cols:
        v = t[c]
        v = v.filled(np.nan) if hasattr(v, "filled") else v
        v = np.asarray(v)
        d[c] = _s(v) if c in ("incl", "aperture") else v.astype(v.dtype.type)   # native byte order (FITS is big-endian; pandas refuses it)
    return pd.DataFrame(d)


ISM = _frame(ANNULUS_ISM_FITS, "m25 Part 8a", KEY4 + ["ngas", "M_dust", "Sigma_dust"])
AVT = _frame(ANNULUS_AV_FITS, "m25 Part 8b", KEY4 + ["A_V"])
APT = _frame(APERTURE_TRUTH_FITS, "m25 Part 7e", KEY4 + ["nstar_ap", "mstar", "age_m_star_myr"])
KIN = _frame(ANNULUS_KIN_FITS, "m25 Part 8j0", ["snap", "gal_id", "aperture", KIN_VAR])
R4 = ISM.merge(AVT, on=KEY4, how="inner").merge(APT, on=KEY4, how="inner").merge(KIN, on=["snap", "gal_id", "aperture"], how="left")
R4["cls"] = [CLASS_OF.get((int(s), int(g)), "unclassified") for s, g in zip(R4["snap"], R4["gal_id"])]
R4["gkey"] = [f"{int(s)}_{int(g)}" for s, g in zip(R4["snap"], R4["gal_id"])]
_lm_of = {(int(a), int(b)): float(m) for a, b, m in zip(SEL["snap"], SEL["gal_id"], SEL["log_mstar"])}
R4["log_mstar"] = [_lm_of.get((int(s), int(g)), np.nan) for s, g in zip(R4["snap"], R4["gal_id"])]

# sampling floors (m25 8c / 7e), then the log quantities: zero dust is NaN, never a floor
_a, _sg, _md = (np.asarray(R4[c], float) for c in ("A_V", "Sigma_dust", "M_dust"))
_ms, _age = np.asarray(R4["mstar"], float), np.asarray(R4["age_m_star_myr"], float) / 1e3
_sg[np.asarray(R4["ngas"], int) < NGAS_ANN_MIN] = np.nan
_lowS = np.asarray(R4["nstar_ap"], int) < NSTAR_AP_MIN
_ms[_lowS], _age[_lowS] = np.nan, np.nan
with np.errstate(divide="ignore", invalid="ignore"):
    R4["lA"] = np.where(np.isfinite(_a) & (_a > 0), np.log10(_a), np.nan)
    R4["lS"] = np.where(np.isfinite(_sg) & (_sg > 0), np.log10(_sg), np.nan)
    R4["lF"] = np.where(np.isfinite(_md) & (_md > 0) & np.isfinite(_ms) & (_ms > 0), np.log10(_md / _ms), np.nan)
R4["age"] = _age
R4["kap"] = np.asarray(R4[KIN_VAR], float)
_in = R4["cls"].isin(P4_CLASSES).to_numpy()
if P4_MASS_RANGE is not None:
    _in &= (R4["log_mstar"].to_numpy() > P4_MASS_RANGE[0]) & (R4["log_mstar"].to_numpy() < P4_MASS_RANGE[1])
R4 = R4[_in].reset_index(drop=True)
CLS4 = R4["cls"].to_numpy()
LAB4 = R4["aperture"].to_numpy()
GK4 = R4["gkey"].to_numpy()


def gboot(vals, keys, n=1000, seed=0):
    # per-galaxy medians over the sightline rows first, then a bootstrap over galaxies -> (median, lo16, hi84, n_gal)
    v, k = np.asarray(vals, float), np.asarray(keys)
    ok = np.isfinite(v)
    v, k = v[ok], k[ok]
    if not len(v):
        return np.nan, np.nan, np.nan, 0
    uk, inv = np.unique(k, return_inverse=True)
    pg = np.array([np.median(v[inv == i]) for i in range(len(uk))])
    bs = np.median(pg[np.random.default_rng(seed).integers(0, len(pg), (n, len(pg)))], axis=1)
    return float(np.median(pg)), float(np.percentile(bs, 16)), float(np.percentile(bs, 84)), len(pg)


def run_median(x, y, keys, nbins=N_AGE_BINS, nmin=NMIN_TRACK, n=NBOOT_TRACK, seed=0):
    # equal-count bins in x over the rows; per bin the median y and a galaxy-bootstrap 16-84 band (all rows of a galaxy resampled together)
    x, y, k = np.asarray(x, float), np.asarray(y, float), np.asarray(keys)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y, k = x[ok], y[ok], k[ok]
    nb = min(nbins, len(set(k)) // nmin)
    if nb < 2:
        return pd.DataFrame(columns=["x", "y", "lo", "hi", "n_gal", "n_rows"])
    e = np.quantile(x, np.linspace(0, 1, nb + 1))
    ib = np.clip(np.searchsorted(e, x, side="right") - 1, 0, nb - 1)
    rng, out = np.random.default_rng(seed), []
    for b in range(nb):
        idx = np.where(ib == b)[0]
        uk, inv = np.unique(k[idx], return_inverse=True)
        rows = [idx[inv == i] for i in range(len(uk))]
        bs = [np.median(y[np.concatenate([rows[j] for j in rng.integers(0, len(uk), len(uk))])]) for _ in range(n)]
        out.append((np.median(x[idx]), np.median(y[idx]), np.percentile(bs, 16), np.percentile(bs, 84), len(uk), len(idx)))
    return pd.DataFrame(out, columns=["x", "y", "lo", "hi", "n_gal", "n_rows"])


def rlim(v, k=1.7, pad=0.04):
    # robust axis range: Tukey fences (quartiles ± k IQR) clipped to the data
    v = np.asarray(v, float)
    v = v[np.isfinite(v)]
    if v.size < 4:
        return None
    q1, q3 = np.percentile(v, [25, 75])
    lo, hi = max(v.min(), q1 - k * (q3 - q1)), min(v.max(), q3 + k * (q3 - q1))
    return (lo - pad * (hi - lo), hi + pad * (hi - lo)) if hi > lo else None


print(f"Part 4 rows: {len(R4)} (galaxy, sightline, label) with an RT A_V, the 8a dust and the 7e stars"
      + (f"; anchor log M* in {P4_MASS_RANGE}" if P4_MASS_RANGE else "; the whole sample"))
print(f"{'class':>14s} {'galaxies':>8s} " + " ".join(f"{ANN_NAME[l]:>11s}" for l in ANNULI) + "   (galaxies with a finite log A_V & age per annulus)")
for c in P4_CLASSES:
    mc = CLS4 == c
    cells = [len(set(GK4[mc & (LAB4 == l) & np.isfinite(R4['lA']) & np.isfinite(R4['age'])])) for l in ANNULI]
    print(f"{P4_NAME[c]:>14s} {len(set(GK4[mc])):8d} " + " ".join(f"{n:11d}" for n in cells))
print(f"{'':>14s} {'':>8s} " + " ".join(f"{len(set(GK4[(LAB4 == l) & np.isfinite(R4['kap'])])):11d}" for l in ANNULI) + f"   (with a {KIN_VAR}: >= {int(Table.read(ANNULUS_KIN_FITS).meta.get('NKIN_MIN', 10))} gas particles)")

In [ ]:
# ── Part 4b — figure D: A_V, dust fraction and dust column of each annulus against the stellar age of the same annulus, per class ──
_fin_age = np.isfinite(R4["age"].to_numpy())
_xlim = rlim(R4["age"].to_numpy()[_fin_age & np.isfinite(R4["lA"].to_numpy())])
fig, axs = pres_grid(len(Y4), [P4_NAME[c] for c in P4_CLASSES], height=2.9, width=4.0)
_rows, _ngal = [], {}
for j, c in enumerate(P4_CLASSES):
    mc = (CLS4 == c) & _fin_age
    for i, (yv, ylab, islog, qname) in enumerate(Y4):
        ax = axs[i, j]
        yy = R4[yv].to_numpy()
        f = (lambda v: 10 ** np.asarray(v, float)) if islog else (lambda v: np.asarray(v, float))   # medians are computed in dex, drawn linear
        for lab in ANNULI:
            m = mc & (LAB4 == lab) & np.isfinite(yy)
            tr = run_median(R4["age"].to_numpy()[m], yy[m], GK4[m])
            _ngal[(yv, c, lab)] = len(set(GK4[m])) if len(tr) else 0
            for _, r in tr.iterrows():
                _rows.append(dict(cls=c, annulus=lab, quantity=qname, age=r.x, value=f(r.y), lo=f(r.lo), hi=f(r.hi), n_gal=int(r.n_gal), n_rows=int(r.n_rows)))
            if not len(tr):
                continue
            ax.plot(tr.x, f(tr.y), color=ANN_COLOR[lab], lw=2.0, zorder=4, label=ANN_NAME[lab] if (i, j) == (0, 0) else None)
            ax.fill_between(tr.x, f(tr.lo), f(tr.hi), color=ANN_COLOR[lab], alpha=0.14, lw=0, zorder=2)
        if islog:
            ax.set_yscale("log")
        ax._auto_y = True
        if _xlim:
            ax.set_xlim(*_xlim)
        if j == 0:
            ax.set_ylabel(ylab)
        else:
            ax.tick_params(labelleft=False)
        if i == len(Y4) - 1:
            ax.set_xlabel("stellar age of the annulus [Gyr]")
        else:
            ax.tick_params(labelbottom=False)
_legend(axs[0, 0], loc="lower left", title="annulus", title_fontsize=8, ncol=1)
pres_finish(fig, axs, "dust_vs_age_annuli", frame=False)

D4 = pd.DataFrame(_rows)
D4.to_csv(os.path.join(PAPERDIR, "paper_dust_vs_age_annuli.csv"), index=False)
print(f"tracks -> {os.path.join(PAPERDIR, 'paper_dust_vs_age_annuli.csv')} ({len(D4)} bins)")
print("\ngalaxies behind each running median (class x annulus; 0 = fewer than 2 x NMIN_TRACK galaxies, no track drawn):")
for yv, _, _, qname in Y4:
    print(f"  {qname}:")
    print(pd.DataFrame({ANN_NAME[l]: [_ngal.get((yv, c, l), 0) for c in P4_CLASSES] for l in ANNULI}, index=[P4_NAME[c] for c in P4_CLASSES]).to_string())

In [ ]:
# ── Part 4c — figure E: the radial ladder (core, outskirt, 10–32 kpc) in A_V, dust fraction, dust column, stellar age and gas rotation, per quenched class ──
_nz, _ncls = len(E_ZONES), len(E_CLASSES)
_ypos = np.arange(_nz, dtype=float)                        # zones down the y axis, core on top
fig, axs = pres_grid(1, [""] * len(Q4), height=0.62 * _nz + 1.3, width=2.55)
_rows = []
for j, (qv, qlab, islog, qname) in enumerate(Q4):
    ax = axs[0, j]
    vv = R4[qv].to_numpy()
    f = (lambda v: 10 ** np.asarray(v, float)) if islog else (lambda v: np.asarray(v, float))
    for ci, c in enumerate(E_CLASSES):
        mc = CLS4 == c
        pts = [gboot(vv[mc & (LAB4 == lab)], GK4[mc & (LAB4 == lab)]) for lab in E_ZONES]
        # per-galaxy contrast first zone - second zone (each a sightline median first; dex for the log quantities)
        _pg = {}
        for lab, sgn in ((E_ZONES[0], 1), (E_ZONES[1], -1)):
            m = mc & (LAB4 == lab) & np.isfinite(vv)
            for g in np.unique(GK4[m]):
                _pg.setdefault(g, {})[sgn] = np.median(vv[m & (GK4 == g)])
        _d = np.array([d[1] - d[-1] for d in _pg.values() if len(d) == 2])
        dmed = gboot(_d, np.arange(len(_d))) if len(_d) else (np.nan, np.nan, np.nan, 0)
        for lab, (med, lo, hi, n) in zip(E_ZONES, pts):
            _rows.append(dict(quantity=qname, cls=c, zone=lab, label=ANN_NAME[lab], median=f(med), lo16=f(lo), hi84=f(hi), n_gal=n,
                              contrast_median=dmed[0], contrast_lo16=dmed[1], contrast_hi84=dmed[2], n_contrast=dmed[3],
                              contrast_unit="dex" if islog else "native"))
        x = np.array([p[0] for p in pts], float)
        ok = np.isfinite(x)
        if ok.sum() < 2:
            continue
        y = _ypos + (ci - (_ncls - 1) / 2) * 0.16
        err = np.array([[p[0] - p[1] for p in pts], [p[2] - p[0] for p in pts]])
        if islog:
            xl, err = 10 ** x, np.array([10 ** x - 10 ** (x - err[0]), 10 ** (x + err[1]) - 10 ** x])
        else:
            xl = x
        ax.errorbar(xl[ok], y[ok], xerr=err[:, ok], color=P4_COLOR[c], marker="o", ms=4.5, lw=1.6, capsize=0, elinewidth=1.0, zorder=4,
                    label=P4_NAME[c] if j == 0 else None)
    if islog:
        ax.set_xscale("log")
    if qv == "kap":
        ax.axvline(KAPPA_DISC, color="0.6", lw=0.8, ls=":", zorder=1)
    ax.set_ylim(_nz - 0.5, -0.5)                             # inverted: core on top, outer zones below
    ax.set_yticks(_ypos)
    ax.set_yticklabels([E_ZONE_NAME[l] for l in E_ZONES] if j == 0 else [])
    ax.set_xlabel(qlab)
plt.tight_layout(w_pad=0.8, rect=(0, 0, 1, 0.91))
_h, _l = axs[0, 0].get_legend_handles_labels()                           # one row above the panels, out of the data
fig.legend(_h, _l, loc="upper left", bbox_to_anchor=(axs[0, 0].get_position().x0, 0.995), ncol=len(E_CLASSES), frameon=False, fontsize=8,
           handlelength=1.8, columnspacing=1.2, borderaxespad=0.0)
paper_save(fig, "core_vs_outskirt")
plt.show()

E4 = pd.DataFrame(_rows)
E4.to_csv(os.path.join(PAPERDIR, "paper_core_vs_outskirt.csv"), index=False)
print(f"table -> {os.path.join(PAPERDIR, 'paper_core_vs_outskirt.csv')}")
print("\nper class: median over galaxies [galaxy-bootstrap 16, 84] (N galaxies with a finite value) per zone | "
      f"{ANN_NAME[E_ZONES[0]]} − {ANN_NAME[E_ZONES[1]]} per galaxy (dex for the log-scaled quantities):")
for qv, _, islog, qname in Q4:
    print(f"  {qname}:")
    for c in E_CLASSES:
        e = E4[(E4.quantity == qname) & (E4.cls == c)].set_index("zone")
        fmt = (lambda v: f"{v:.3g}") if islog else (lambda v: f"{v:.2f}")
        cell = lambda r: (f"{fmt(r['median'])} [{fmt(r['lo16'])},{fmt(r['hi84'])}] ({int(r['n_gal']):3d})" if np.isfinite(r["median"]) else f"{'—':>24s}")
        r0 = e.loc[E_ZONES[0]]
        ctr = (f"{r0['contrast_median']:+.2f} [{r0['contrast_lo16']:+.2f},{r0['contrast_hi84']:+.2f}] ({int(r0['n_contrast']):3d})"
               if np.isfinite(r0["contrast_median"]) else "—")
        print(f"    {P4_NAME[c]:>14s}  " + " | ".join(cell(e.loc[l]) for l in E_ZONES) + f" | {ctr}")

In [ ]:
# ── Part 4d — figure F: the radial trend per class (+ SF controls): A_V, dust / gas / H2 per stellar mass, stellar age and rotation support of each annulus ──
# Is a class more attenuated in its core, richer in dust or in gas there, younger, or just more rotating? One panel per quantity, the four
# disjoint annuli inside -> out along x, one line per class = the median over its galaxies (each galaxy = the median of its sightlines)
# with a galaxy-bootstrap 16–84 % band; the mass-matched SF controls in grey. One figure per mass sample (F_MASS_SAMPLES).
F_ANNULI   = ["ap1kpc", "ann3kpc", "ann10kpc", "ann32kpc"]          # the disjoint annuli (m25 T7 vocabulary; add "ann100kpc" for the CGM shell)
F_CLASSES  = ["star_forming"] + list(E_CLASSES)                      # lines: SF controls first (grey), then the quenched classes
F_MASS_SAMPLES = [(r"$\log M_\star > 10.25$", (10.25, np.inf), "mgt10p25"), (r"$\log M_\star < 10.25$", (-np.inf, 10.25), "mlt10p25")]
F_KAPPA_LINE = 0.3                                                   # dashed: rotation- / pressure-supported (Part 5's KAPPA_PRESSURE)
F_NBOOT    = 500
F_NMIN     = 4                                                       # galaxies with a finite value for a point
# (column, axis label, log axis, CSV name): the per-galaxy value of an annulus; the ratios use the annulus' own stellar mass (m25 7e)
F_QUANT = [("A_V", r"$A_V$ [mag]", True, "A_V"), ("fdust", r"$M_{\rm dust}/M_\star$", True, "Mdust_over_Mstar"),
           ("fgas", r"$M_{\rm gas}/M_\star$", True, "Mgas_over_Mstar"), ("fH2", r"$M_{\rm H_2}/M_\star$", True, "MH2_over_Mstar"),
           ("age", "stellar age [Gyr]", False, "age_Gyr"), ("kappa_gas", r"$\kappa_{\rm rot}^{\rm gas}$", False, "kappa_gas"),
           ("kappa_H2", r"$\kappa_{\rm rot}^{\rm H_2}$", False, "kappa_H2"), ("kappa_star", r"$\kappa_{\rm rot}^{\star}$", False, "kappa_star")]
F_LAYOUT = (2, 4)

_ism = _frame(ANNULUS_ISM_FITS, "m25 Part 8a", KEY4 + ["ngas", "M_gas", "M_dust", "M_H2"])
_avt = _frame(ANNULUS_AV_FITS, "m25 Part 8b", KEY4 + ["A_V"])
_apt = _frame(APERTURE_TRUTH_FITS, "m25 Part 7e", KEY4 + ["nstar_ap", "mstar", "age_m_star_myr"])
_kin4 = _frame(ANNULUS_KIN_FITS, "m25 Part 8j0", ["snap", "gal_id", "aperture", "kappa_gas", "kappa_H2", "kappa_star"])
RF = _ism.merge(_avt, on=KEY4, how="inner").merge(_apt, on=KEY4, how="inner").merge(_kin4, on=["snap", "gal_id", "aperture"], how="left")
RF = RF[RF["aperture"].isin(F_ANNULI)].reset_index(drop=True)
RF["cls"] = [CLASS_OF.get((int(s), int(g)), "unclassified") for s, g in zip(RF["snap"], RF["gal_id"])]
RF["gkey"] = [f"{int(s)}_{int(g)}" for s, g in zip(RF["snap"], RF["gal_id"])]
_lmF = {(int(a), int(b)): float(m) for a, b, m in zip(SEL["snap"], SEL["gal_id"], SEL["log_mstar"])}
RF["log_mstar"] = [_lmF.get((int(s), int(g)), np.nan) for s, g in zip(RF["snap"], RF["gal_id"])]
_msF = np.asarray(RF["mstar"], float); _ageF = np.asarray(RF["age_m_star_myr"], float) / 1e3
_low = np.asarray(RF["nstar_ap"], int) < NSTAR_AP_MIN                # m25 7e floor: M* and the age need NSTAR_AP_MIN stars
_msF[_low], _ageF[_low] = np.nan, np.nan
with np.errstate(divide="ignore", invalid="ignore"):
    for c, num in (("fdust", "M_dust"), ("fgas", "M_gas"), ("fH2", "M_H2")):
        v = np.asarray(RF[num], float)
        RF[c] = np.where(np.isfinite(v) & (v > 0) & np.isfinite(_msF) & (_msF > 0), v / _msF, np.nan)   # zero mass -> NaN, never a floor
    RF["A_V"] = np.where(np.isfinite(RF["A_V"]) & (RF["A_V"] > 0), RF["A_V"], np.nan)
RF["age"] = _ageF
RF = RF[RF["cls"].isin(F_CLASSES)].reset_index(drop=True)
F_NAME = dict(P4_NAME)
F_COLOR = dict(P4_COLOR)


def _fboot(vals, keys, n=F_NBOOT, seed=0):
    """Per-galaxy medians over the sightline rows, then the bootstrap of their median -> (median, lo16, hi84, n_gal); NaN below F_NMIN."""
    v, k = np.asarray(vals, float), np.asarray(keys)
    ok = np.isfinite(v)
    v, k = v[ok], k[ok]
    uk, inv = np.unique(k, return_inverse=True)
    if len(uk) < F_NMIN:
        return np.nan, np.nan, np.nan, len(uk)
    pg = np.array([np.median(v[inv == i]) for i in range(len(uk))])
    bs = np.median(pg[np.random.default_rng(seed).integers(0, len(pg), (n, len(pg)))], axis=1)
    return float(np.median(pg)), float(np.percentile(bs, 16)), float(np.percentile(bs, 84)), len(uk)


F_ROWS = []
_xF = np.arange(len(F_ANNULI))
for sname, mrange, stag in F_MASS_SAMPLES:
    S = RF[(RF["log_mstar"] > mrange[0]) & (RF["log_mstar"] < mrange[1])]
    nsel = {c: len(set(S["gkey"][S["cls"] == c])) for c in F_CLASSES}
    print(f"\n{sname.replace('$', '').replace(chr(92) + 'log', 'log').replace(chr(92) + 'star', '*')}: " + ", ".join(f"{F_NAME[c]} {nsel[c]}" for c in F_CLASSES))
    fig, axs = pres_grid(F_LAYOUT[0], [""] * F_LAYOUT[1], height=2.9, width=3.4)
    for q, (qv, qlab, islog, qname) in enumerate(F_QUANT):
        ax = axs[q // F_LAYOUT[1], q % F_LAYOUT[1]]
        for ci, c in enumerate(F_CLASSES):
            Sc = S[S["cls"] == c]
            pts = [_fboot(Sc[qv][Sc["aperture"] == lab], Sc["gkey"][Sc["aperture"] == lab]) for lab in F_ANNULI]
            for lab, (med, lo, hi, n) in zip(F_ANNULI, pts):
                nz = len(set(Sc["gkey"][(Sc["aperture"] == lab)])) - n
                F_ROWS.append(dict(sample=stag, quantity=qname, cls=c, annulus=lab, label=ANN_NAME[lab], n_gal=n, n_dropped=nz,
                                   median=med, lo16=lo, hi84=hi))
            y = np.array([p[0] for p in pts]); lo = np.array([p[1] for p in pts]); hi = np.array([p[2] for p in pts])
            ok = np.isfinite(y)
            if ok.sum() == 0:
                continue
            off = (ci - (len(F_CLASSES) - 1) / 2) * 0.06
            ax.fill_between(_xF[ok] + off, lo[ok], hi[ok], color=F_COLOR[c], alpha=0.18 if c != "star_forming" else 0.12, lw=0, zorder=2)
            ax.plot(_xF[ok] + off, y[ok], "-", color=F_COLOR[c], lw=1.8 if c != "star_forming" else 1.4, marker="o", ms=4.5, mec="k", mew=0.4, zorder=5,
                    label=f"{F_NAME[c]} (N={nsel[c]})" if q == 0 else None)
        if islog:
            ax.set_yscale("log")
        if qv.startswith("kappa"):
            ax.axhline(F_KAPPA_LINE, color="0.3", lw=1.0, ls="--", zorder=1)
            ax.set_ylim(0.0, 1.0)
        ax.set_xlim(-0.5, len(F_ANNULI) - 0.5)
        ax.set_xticks(_xF)
        ax.set_xticklabels([ANN_NAME[l].replace(" kpc", "") for l in F_ANNULI] if q // F_LAYOUT[1] == F_LAYOUT[0] - 1 else [])
        ax.set_ylabel(qlab)
        if q // F_LAYOUT[1] == F_LAYOUT[0] - 1:
            ax.set_xlabel("annulus [kpc]")
        ax.tick_params(direction="in", top=True, right=True); ax.grid(False)
    axs[0, 0].legend(loc="best", frameon=False, fontsize=8, handlelength=1.8)
    fig.text(0.0, 1.0, f"{sname}: the radial trend of each quantity at the anchor, per AGN class and for the SF controls", fontsize=9.5, color="0.25", ha="left", va="bottom")
    fig.legend([Line2D([], [], color="0.4", lw=1.8, marker="o", mec="k", mew=0.4, ms=4.5), Rectangle((0, 0), 1, 1, fc="0.4", alpha=0.2, lw=0), Line2D([], [], color="0.3", lw=1.0, ls="--")],
               [f"class median over galaxies (each galaxy = the median of its 4 sightlines; >= {F_NMIN} galaxies)", "galaxy-bootstrap 16–84 %",
                rf"$\kappa_{{\rm rot}} = {F_KAPPA_LINE:g}$: rotation- / pressure-supported"],
               loc="upper center", ncol=3, frameon=False, fontsize=8.5, bbox_to_anchor=(0.5, 0.0),
               title=(f"projected annuli of the m25 ladder (dust_on / dust_off RT A_V; 8a dust / gas / H2; 7e M* and age, >= {NSTAR_AP_MIN} stars); a zero dust / gas / H2 mass is dropped "
                      f"(n_dropped in the CSV); kappa in the spherical rung / shell (8j0, >= 10 weighted particles, sightline-independent)"), title_fontsize=8.5)
    plt.tight_layout(w_pad=1.0, h_pad=0.8)
    paper_save(fig, f"radial_profiles_{stag}")
    plt.show()

FR = pd.DataFrame(F_ROWS)
FR.to_csv(os.path.join(PAPERDIR, "paper_radial_profiles.csv"), index=False)
print(f"\ntable -> {os.path.join(PAPERDIR, 'paper_radial_profiles.csv')} ({len(FR)} rows)")
print("innermost annulus (0–1 kpc) per class: median [boot 16, 84] (N galaxies with a finite value):")
for stag in [s[2] for s in F_MASS_SAMPLES]:
    print(f"  {stag}:")
    for qv, _, islog, qname in F_QUANT:
        fmt = (lambda v: f"{v:.3g}") if islog else (lambda v: f"{v:.2f}")
        cells = []
        for c in F_CLASSES:
            r = FR[(FR["sample"] == stag) & (FR["quantity"] == qname) & (FR["cls"] == c) & (FR["annulus"] == F_ANNULI[0])].iloc[0]
            cells.append(f"{F_NAME[c][:6]:>6s} " + (f"{fmt(r['median'])} [{fmt(r['lo16'])},{fmt(r['hi84'])}] ({int(r['n_gal'])})" if np.isfinite(r["median"]) else f"— ({int(r['n_gal'])})"))
        print(f"    {qname:>16s}  " + " | ".join(cells))


## Part 5 — rotation support and stellar age at every critical point: core and outskirt, per $A_V$ bin, per class

The quench sequence in the ($\kappa_{\rm rot}$, stellar age) plane and, because the tracks of the different classes cover very
different age intervals, two companion figures that put the **same critical points side by side** on a categorical axis.
**Zones**: the core (the $r<3.2$ kpc sphere) and the outskirt (the $3.2<r<10$ kpc shell) of the KS notebook's Part 3b
(`ks_stage_kinematics.fits`: the Sales+12 $\kappa_{\rm rot} = K_{\rm rot}/K$ of the H$_2$-weighted gas, of the stars and of all gas
about the zone's own spin axis, and the mass-weighted age of the zone's stars, measured on the reduced particle files at every
critical epoch). **Critical points** (`P5_STAGES`, canonical order): **AGN ignition** (the BHAR crosses half its pre-QT peak) →
**jet-mode onset** (ungated $w_{\rm jet} \geq 0.5$) → **SFT** → **QT** → **track end** — the first two are the m25 Part 2d events,
attached to the KS epochs by its Part 1 (2026-08-28 evening+); until the KS notebook has been re-run with them (Part 1 → Part 2
`BUILD_PLAN` + sbatch for the new files → Parts 3, 3b, 4) the figures fall back to SFT → QT → end and say so. Per galaxy the
points are drawn in their actual time order (a weak galaxy's jet onset can follow SFT); the group medians are joined in the
canonical order. **Bins**: the anchor-$A_V$ terciles of Part 3 (core $A_V$, median over the sightlines; `P5_AV_EDGES = None` =
terciles of each mass sample). The dashed line is $\kappa_{\rm rot} = $ `KAPPA_PRESSURE` $= 0.3$: below it the component is
pressure- rather than rotation-supported. Two **mass samples** (`P5_MASS_SAMPLES`): $\log M_\star > 10.25$ and
$\log M_\star < 10.25$ (the low-mass sample holds only two `strong` galaxies).

* **A — the tracks** (`paper_kappa_vs_age_{H2,star}_{mgt10p25,mlt10p25}`): rows core / outskirt, one column per class
  (`P5_COLUMNS`; prepend `"all"`), one thick track per $A_V$ bin through the bin medians of each critical point (median age,
  median $\kappa$ over the galaxies with a measurable value; ≥ `NMIN_BIN`), `P5_SHOW_INDIVIDUAL` draws the galaxies' own tracks.
  The star-forming controls are off by default (`P5_SHOW_SF`: grey cloud at their anchor, log $y$).
* **B — the sequences** (`paper_kinematics_sequence_<sample>`, `paper_stellar_sequence_<sample>`, `paper_ism_sequence_<sample>`):
  $x$ = the critical points, equally spaced; one column per $A_V$ bin with the **three classes side by side** in every panel
  (`P5_SEQ_LINES = "av"` swaps lines and columns): group median with a galaxy-bootstrap 16–84 % interval; rows = core / outskirt of
  each quantity, one $y$ range per quantity. Three figures (`P5_SEQ_FIGS`): **kinematics** — $\kappa_{\rm rot}^{\rm H_2}$ (full
  0–1 range) and $\kappa_{\rm rot}^\star$ ($y$ zoomed to the data); **stellar** — sSFR (SFR over `P5_SSFR_COL` = 100 Myr / $M_\star$ of
  the aperture, a zero SFR floored at `P5_SSFR_FLOOR` = $10^{-13}$ yr$^{-1}$ and kept, dashed $10^{-11}$), stellar age, and a last
  row with the elapsed time of each interval divided by the cosmic time at its end (`P5_DT_NORM`) — fast and slow quenchers separate
  there; **ISM** — $M_{\rm dust}/M_\star$ and $M_{\rm H_2}/M_\star$. sSFR and the two fractions come from the KS tracks' projected
  apertures ($r<3.2$ kpc disc, 3.2–10 kpc annulus; a zero mass dropped), the ages and $\kappa$ from the spherical zones. The range
  covered by the **ALMA-C11 detections** is shaded (`P5_OBS_DET = "det_dust"`: whole-galaxy fiducial CIGALE sSFR, age and
  $M_{\rm dust}/M_\star$ of the 9 dust-detected sources; `P5_OBS_DET_H2 = "det_co"`: $M_{\rm H_2}/M_\star$ of the 11 CO-detected,
  `MH2_fid` divided by `P5_OBS_HE` = 1.36 because SIMBA's H$_2$ is hydrogen-only; `almac11_gas_dust.csv`). The zone of a row is its
  frame / tag colour (core deep blue, outskirt bronze; `P5_ZONE_COLOR`); the class colours (teal / amber / dark red) are the paper's
  `CLASS_COLOR_PRES`. Reading: the vertical offsets between the lines are the class differences at each critical point, whatever ages
  the classes reach.
* **C — the intervals** (`paper_kappa_intervals_{mgt10p25,mlt10p25}`): the change **over each interval between consecutive
  critical points**, computed *per galaxy* (both epochs measured: $\Delta\kappa = \kappa_{\rm after} - \kappa_{\rm before}$, and
  the elapsed time $\Delta t$), then the group median with its bootstrap interval; same grid as B. A paired difference removes
  the galaxy's own level, so it compares what happens to the classes during ignition → jet, jet → SFT, SFT → QT and QT → end
  even when their coverage differs; a negative $\Delta t$ means the canonical order was inverted for that group (jet mode only
  after SFT).

**Censoring.** A $\kappa$ needs `NKIN_MIN` = 10 weighted particles *and* velocities in the reduced file; the H$_2$ of a
quenched core is often below that at QT and at the track end, so the H$_2$ rows thin out along the track — the printed table
gives, per critical point, how many galaxies of each bin still have a measurable $\kappa_{\rm rot}^{\rm H_2}$, and a median is
drawn only from those (a median over the *survivors*, not an upper limit as in Part 3; the paired intervals need both endpoints).
Files without velocities (`has_vel = False`) leave every $\kappa$ NaN — the cell says so. Tables: `paper_kappa_vs_age.csv`,
`paper_kappa_sequence.csv` (one row per figure, zone, quantity, critical point, column, line), `paper_kappa_intervals.csv` (incl. the
normalised elapsed time, `quantity = dt_norm_end`).


In [ ]:
# ── Part 5 — kappa_rot of the H2 / of the stars and the stellar age of the same zone at the critical points, per A_V bin, per class ──
# Per mass sample: (A) the tracks in the (kappa, age) plane, one column per class; (B) three critical-point sequence figures on a
# categorical axis (ignition -> jet onset -> SFT -> QT -> end), classes side by side in every panel: gas rotation | stellar rotation
# (zoomed) | stellar age, M_dust/M* and the elapsed time of each interval / cosmic time, with the range of the ALMA-C11 detections;
# (C) the paired change of kappa (and the elapsed time) over each interval between consecutive critical points, per galaxy first.
P5_MASS_SAMPLES = [(r"$\log M_\star > 10.25$", (10.25, np.inf), "mgt10p25"), (r"$\log M_\star < 10.25$", (-np.inf, 10.25), "mlt10p25")]
P5_COMPONENTS   = [("H2", "kappa_H2", "n_H2", r"$\kappa_{\rm rot}^{\rm H_2}$"), ("star", "kappa_star", "n_star", r"$\kappa_{\rm rot}^{\star}$")]
P5_ZONES        = [CORE_LAB, OUT_LAB]      # core (0–3.2 kpc sphere), outskirt (3.2–10 kpc shell): the KS Part 3b zones (spherical)
P5_COLUMNS      = list(P3_CLASSES)         # figure A: one column per class; prepend "all" for an all-classes column (thin frame)
P5_AV_EDGES     = None                     # None: terciles of each mass sample | fixed edges, e.g. (0.1, 0.5) [mag]
P5_SHOW_SF      = False                    # figure A: SF controls of the same mass sample at their anchor (grey cloud) -> y on a log scale
P5_YSCALE       = "log" if P5_SHOW_SF else "linear"
KAPPA_PRESSURE  = 0.3                      # dashed line: below it the component is pressure- rather than rotation-supported
P5_SHOW_INDIVIDUAL = False                 # figure A: thin lines, every galaxy's own track under its bin's median track
P5_ZONE_NAME    = {CORE_LAB: "core, $r<3.2$ kpc", OUT_LAB: "outskirt, $3.2<r<10$ kpc", "ap10kpc": "$r<10$ kpc"}
P5_ZONE_SHORT   = {CORE_LAB: "core", OUT_LAB: "outskirt", "ap10kpc": "$r<10$ kpc"}
# the critical points in the canonical event order; per galaxy the drawn order follows the actual times (a weak galaxy's jet onset can
# follow SFT). agn_ign / jet_on need the KS notebook of 2026-08-28 evening+ (Part 1 attaches them from the m25 Part 2d windows table,
# Part 2 BUILD_PLAN + sbatch extracts their files, Parts 3-4 measure and join them); without them the figures use SFT / QT / end alone.
P5_STAGES       = ["agn_ign", "jet_on"] + list(STAGES_DRAW)
P5_STAGE_MARKER = dict(STAGE_MARKER, agn_ign="*", jet_on="D")
P5_STAGE_LABEL  = dict(STAGE_LABEL, agn_ign="AGN ignition (BHAR crosses half its pre-QT peak)", jet_on=r"jet-mode onset (ungated $w_{\rm jet} \geq 0.5$)")
P5_STAGE_SHORT  = {"agn_ign": "ignition", "jet_on": "jet on", "sft": "SFT", "qt": "QT", "end": "end"}
P5_SEQ_LINES    = "class"                  # figures B / C: lines = "class" (one column per A_V bin) | "av" (one column per class)
P5_NBOOT        = 500                      # galaxy bootstrap of every group median (16–84 % interval)
# ── figures B / C design: three sequence figures, zone-coloured frames + tags, larger text (2026-08-29) ──
P5_FONT         = dict(label=13, tick=11.5, legend=10.5, header=12.5, tag=11.5, note=10.5)
P5_ZONE_COLOR   = {CORE_LAB: "#0f4c81", OUT_LAB: "#8c6d31"}                 # frames + tags: core deep blue, outskirt bronze (never used for lines)
P5_ZONE_TAG     = {CORE_LAB: "CORE\n$r<3.2$ kpc", OUT_LAB: "OUTSKIRT\n$3.2<r<10$ kpc"}
# (file stem, [(column, axis label, kind)], title, elapsed-time row): kind = kappa (0–1) | kappa_zoom (limits from the data) | lin | log;
# one y range per quantity over its core / outskirt rows and every column
P5_SEQ_FIGS     = [("kinematics_sequence", [("kappa_H2", r"$\kappa_{\rm rot}^{\rm H_2}$", "kappa"), ("kappa_star", r"$\kappa_{\rm rot}^{\star}$", "kappa_zoom")],
                                            "rotation support of the H$_2$ and of the stars", False),
                   ("stellar_sequence",    [("ssfr", r"sSFR [yr$^{-1}$]", "log"), ("age_mw_gyr", "stellar age [Gyr]", "lin")], "sSFR, stellar age and elapsed time", True),
                   ("ism_sequence",        [("fdust", r"$M_{\rm dust}/M_\star$", "log"), ("fH2", r"$M_{\rm H_2}/M_\star$", "log")], "dust and H$_2$ fractions", False)]
P5_SSFR_COL     = "sfr100"                 # sSFR rows: SFR of the KS tracks' apertures averaged over 100 Myr (sfr100 | sfr25 | sfr_inst) over their M*
P5_SSFR_FLOOR   = 1e-13                    # [yr^-1] a zero SFR is floored here (kept in the medians: a group median on the floor = most of its galaxies have no SFR)
P5_SSFR_LINE    = 1e-11                    # [yr^-1] dashed reference on the sSFR rows
P5_DT_NORM      = "end"                    # elapsed-time row of the third figure: dt / cosmic time at the "end" | "start" of the interval | "anchor" | None (Gyr)
P5_OBS_DET      = "det_dust"               # age / M_dust band = the range covered by the ALMA-C11 sources with this flag (det_dust | det_co); None: no band
P5_OBS_DET_H2   = "det_co"                 # M_H2 / M* band = the CO-detected sources (MH2_fid: alpha_CO = 4.36 incl. He -> divided by P5_OBS_HE, SIMBA H2 is H-only)
P5_OBS_HE       = 1.36
P5_OBS_COLS     = ("logage", "logMstar", "logMdust")   # fiducial-run columns of almac11_gas_dust.csv (CIGALE age [log yr], log M*, log M_dust)
P5_OBS_COLOR    = C_OBS

KIN = Table.read(need(KIN_STAGE_FITS, "ks_tracks_quenched_m25.ipynb Part 3b (2026-08-28+)"))
NKIN_MIN = int(KIN.meta.get("NKIN_MIN", 10))
_kin = pd.DataFrame({c: (_s(KIN[c]) if c == "zone" else np.asarray(KIN[c]).astype(np.asarray(KIN[c]).dtype.type))
                     for c in ("snap", "gx", "zone", "kappa_gas", "kappa_H2", "kappa_star", "n_gas", "n_H2", "n_star", "m_H2", "m_star",
                               "age_mw_gyr", "has_vel")})
_kin = _kin[_kin["zone"].isin(P5_ZONES)].reset_index(drop=True)
# the epoch keys of the critical points (aperture-independent: one (snap, gx) per galaxy and stage) and their cosmic times, from TRACKS
_stgT, _popT = _s(TRACKS["stage"]), _s(TRACKS["pop"])
P5_HAVE = [st for st in P5_STAGES if st in set(_stgT[_popT == "Q"])]
_absent = [st for st in P5_STAGES if st not in P5_HAVE]
if _absent:
    print(f"[note] stages {_absent} are not in {os.path.relpath(TRACKS_FITS, os.getcwd())}: the KS notebook of 2026-08-28 evening+ adds them "
          "(Part 1 rebuild -> Part 2 BUILD_PLAN=True + sbatch -> Parts 3, 3b, 4); drawing the critical points " + " -> ".join(P5_HAVE))
_e = TRACKS[(_popT == "Q") & np.isin(_stgT, P5_HAVE)]
_ep = pd.DataFrame(dict(gkey=_s(_e["gkey"]), stage=_s(_e["stage"]), snap=np.asarray(_e["snap"], int), gx=np.asarray(_e["gx"], int),
                        t_gyr=np.asarray(_e["t_stage_gyr"], float))).drop_duplicates(["gkey", "stage"])
K5 = _ep.merge(_kin, on=["snap", "gx"], how="inner")
_sfe = TRACKS[(_popT == "SF") & (_stgT == "anchor")]
_sf = pd.DataFrame(dict(snap=np.asarray(_sfe["snap"], int), gx=np.asarray(_sfe["gx"], int),
                        log_mstar=np.asarray(_sfe["log_mstar_anchor"], float))).drop_duplicates()
K5_SF = _sf.merge(_kin, on=["snap", "gx"], how="inner")
# M_dust / M*, M_H2 / M* and sSFR of the zones at every critical point from the KS tracks' projected apertures (m_dust, m_H2, sfr, m_star,
# n_star of ap3kpc and ap10kpc): core = the r < 3.2 kpc disc, outskirt = the 3.2–10 kpc annulus (ap10kpc - ap3kpc); NaN without
# NSTAR_AP_MIN stars; a zero dust / H2 mass -> NaN (never a floor), a zero SFR -> P5_SSFR_FLOOR (kept: quenching is the point)
_apT = _s(TRACKS["ap_label"])
_td = TRACKS[(_popT == "Q") & np.isin(_stgT, P5_HAVE) & np.isin(_apT, ["ap3kpc", "ap10kpc"])]
_td = pd.DataFrame(dict(gkey=_s(_td["gkey"]), stage=_s(_td["stage"]), ap=_s(_td["ap_label"]), m_dust=np.asarray(_td["m_dust"], float),
                        m_H2=np.asarray(_td["m_H2"], float), sfr=np.asarray(_td[P5_SSFR_COL], float), m_star=np.asarray(_td["m_star"], float),
                        n_star=np.asarray(_td["n_star"], float))).drop_duplicates(["gkey", "stage", "ap"])
_pv = _td.pivot_table(index=["gkey", "stage"], columns="ap", values=["m_dust", "m_H2", "sfr", "m_star", "n_star"], aggfunc="first")
_fd_rows = []
for (gk, st), r in _pv.iterrows():
    for zone, sel in ((CORE_LAB, lambda c: r[(c, "ap3kpc")]), (OUT_LAB, lambda c: r[(c, "ap10kpc")] - r[(c, "ap3kpc")])):
        ms, ns = sel("m_star"), sel("n_star")
        ok = np.isfinite(ms) and ms > 0 and ns >= NSTAR_AP_MIN
        d = dict(gkey=gk, stage=st, zone=zone)
        for c, num in (("fdust", "m_dust"), ("fH2", "m_H2")):
            v = sel(num)
            d[c] = (v / ms) if (ok and np.isfinite(v) and v > 0) else np.nan
        v = sel("sfr")
        d["ssfr"] = max(v / ms, P5_SSFR_FLOOR) if (ok and np.isfinite(v) and v >= 0) else np.nan
        _fd_rows.append(d)
K5 = K5.merge(pd.DataFrame(_fd_rows), on=["gkey", "stage", "zone"], how="left")
for c in ("fdust", "fH2", "ssfr"):
    if c not in K5:
        K5[c] = np.nan
# the parameter space covered by the observed sample: ALMA-C11 detections (fiducial CIGALE run), min–max of the age and of M_dust / M*
P5_OBS_BAND, P5_OBS_N = {}, 0
if P5_OBS_DET:
    OBS_GD = pd.read_csv(need(OBS_GD_CSV, "git pull"))
    _det = OBS_GD[OBS_GD[P5_OBS_DET].astype(bool)]
    _oage = 10 ** _det[P5_OBS_COLS[0]].to_numpy(float) / 1e9
    _ofd = 10 ** (_det[P5_OBS_COLS[2]].to_numpy(float) - _det[P5_OBS_COLS[1]].to_numpy(float))
    _oss = _det["SFR"].to_numpy(float) / 10 ** _det[P5_OBS_COLS[1]].to_numpy(float)      # fiducial CIGALE SFR over the fiducial M*
    P5_OBS_BAND = {"age_mw_gyr": (float(np.nanmin(_oage)), float(np.nanmax(_oage))), "fdust": (float(np.nanmin(_ofd)), float(np.nanmax(_ofd))),
                   "ssfr": (float(np.nanmin(_oss)), float(np.nanmax(_oss)))}
    P5_OBS_N = int(len(_det))
    print(f"observed band: {P5_OBS_N} ALMA-C11 sources with {P5_OBS_DET}: CIGALE age {P5_OBS_BAND['age_mw_gyr'][0]:.2f}–{P5_OBS_BAND['age_mw_gyr'][1]:.2f} Gyr, "
          f"M_dust/M* {P5_OBS_BAND['fdust'][0]:.2e}–{P5_OBS_BAND['fdust'][1]:.2e}, sSFR {P5_OBS_BAND['ssfr'][0]:.2e}–{P5_OBS_BAND['ssfr'][1]:.2e} /yr ({', '.join(_det['id'].astype(str))})")
    _co = OBS_GD[OBS_GD[P5_OBS_DET_H2].astype(bool)] if P5_OBS_DET_H2 else OBS_GD.iloc[:0]
    P5_OBS_N_H2 = int(len(_co))
    if P5_OBS_N_H2:
        _ofh = _co["MH2_fid"].to_numpy(float) / P5_OBS_HE / 10 ** _co[P5_OBS_COLS[1]].to_numpy(float)
        P5_OBS_BAND["fH2"] = (float(np.nanmin(_ofh)), float(np.nanmax(_ofh)))
        print(f"               {P5_OBS_N_H2} sources with {P5_OBS_DET_H2}: M_H2/M* {P5_OBS_BAND['fH2'][0]:.2e}–{P5_OBS_BAND['fH2'][1]:.2e} "
              f"(MH2_fid / {P5_OBS_HE:g} for He, over the fiducial M*; {', '.join(_co['id'].astype(str))})")
_nvel = int((~K5["has_vel"].astype(bool)).sum()) // max(len(P5_ZONES), 1)
_nz = max(len(P5_ZONES), 1)
print(f"stage kinematics: {len(KIN)} rows ({len(_kin) // _nz} files in the drawn zones); Q epochs matched: {len(K5) // _nz} of {len(_ep)} "
      + "(" + ", ".join(f"{st} {int((K5['stage'] == st).sum()) // _nz}/{int((_ep['stage'] == st).sum())}" for st in P5_HAVE) + f"); SF anchors: {len(K5_SF) // _nz}"
      + (f"; [WARN] {_nvel} epochs come from files WITHOUT velocities (kappa NaN): re-run the Part 2 sbatch of the KS notebook + its Part 3b" if _nvel else ""))
_gal_av = dict(zip(GK, np.asarray(GALS["av_anchor"], float)))
_gal_cls = dict(zip(GK, _cl))
_gal_lm = dict(zip(GK, _lm))
_gal_tanc = dict(zip(GK, np.asarray(GALS["t_anchor_gyr"], float)))
_rank_canon = {st: i for i, st in enumerate(P5_STAGES)}


def p5_sample(mrange):
    """Galaxies (gkeys) of a mass sample with a quench event, an anchor A_V and a class in P3_CLASSES + their A_V bin edges."""
    m = (_lm > mrange[0]) & (_lm < mrange[1]) & np.isfinite(_av) & np.isin(_cl, P3_CLASSES) & np.asarray(GALS["has_clock"], bool)
    edges = np.quantile(_av[m], [1 / 3, 2 / 3]) if P5_AV_EDGES is None else np.asarray(P5_AV_EDGES, float)
    b = np.where(m, np.searchsorted(edges, np.where(np.isfinite(_av), _av, -1.0), side="right"), -1)
    return m, edges, dict(zip(GK, b))


def _bin_labels(edges):
    nb = len(edges) + 1
    return [rf"$A_V \leq {edges[0]:.2f}$"] + [rf"${edges[k - 1]:.2f} < A_V \leq {edges[k]:.2f}$" for k in range(1, nb - 1)] + [rf"$A_V > {edges[-1]:.2f}$"]


def _bmed(v, n=P5_NBOOT, seed=0):
    """(median, 16, 84 of the bootstrap of the median, N) over the finite values; NaNs with fewer than NMIN_BIN."""
    v = np.asarray(v, float)
    v = v[np.isfinite(v)]
    if len(v) < NMIN_BIN:
        return np.nan, np.nan, np.nan, len(v)
    bs = np.median(v[np.random.default_rng(seed).integers(0, len(v), (n, len(v)))], axis=1)
    return float(np.median(v)), float(np.percentile(bs, 16)), float(np.percentile(bs, 84)), len(v)


def _gal_order(r):
    """Rows of one galaxy in the order of the actual epoch times (ties: the canonical order)."""
    return r.iloc[np.lexsort(([_rank_canon[s] for s in r["stage"]], r["t_gyr"].to_numpy(float)))]


# ── figure A: the (kappa, age) plane ──
def kappa_panel(ax, comp, zone, cls, m_sample, bin_of, nb):
    """One panel: per A_V bin the individual tracks (thin) and the bin-median track (thick) of `comp` vs the zone's stellar age through
    the critical points; galaxies of class `cls` (None = all P3_CLASSES) in the mass sample. Returns [(bin, n_gal, [stage dicts])]."""
    _, kcol, ncol, _ = comp
    gk_in = [g for g in GK[m_sample] if (cls is None or _gal_cls[g] == cls)]
    R = K5[(K5["zone"] == zone) & K5["gkey"].isin(gk_in)]
    out = []
    for k in range(nb):
        gks = [g for g in gk_in if bin_of[g] == k]
        Rk = R[R["gkey"].isin(gks)]
        pts = []
        for st in P5_HAVE:
            r = Rk[(Rk["stage"] == st) & np.isfinite(Rk[kcol].to_numpy(float)) & np.isfinite(Rk["age_mw_gyr"].to_numpy(float))]
            d = dict(stage=st, n_gal=len(gks), n_stage=int((Rk["stage"] == st).sum()), n=len(r))
            if len(r) >= NMIN_BIN:
                km = _bmed(r[kcol])
                d.update(x=float(np.median(r["age_mw_gyr"])), y=km[0], ylo=km[1], yhi=km[2],
                         y16=float(np.percentile(r[kcol], 16)), y84=float(np.percentile(r[kcol], 84)), t=float(np.median(r["t_gyr"])))
            pts.append(d)
        if P5_SHOW_INDIVIDUAL:
            for g in gks:
                r = _gal_order(Rk[Rk["gkey"] == g])
                x, y = r["age_mw_gyr"].to_numpy(float), r[kcol].to_numpy(float)
                ok = np.isfinite(x) & np.isfinite(y)
                if ok.sum() >= 2:
                    ax.plot(x[ok], y[ok], "-", color=AV_COLORS[k], lw=0.7, alpha=0.35, zorder=3)
        drawn = [(d["x"], d["y"], d["stage"]) for d in pts if "x" in d]      # canonical order of the points present
        if len(drawn) >= 2:
            ax.plot([p[0] for p in drawn], [p[1] for p in drawn], "-", color=AV_COLORS[k], lw=2.2, alpha=0.9, zorder=6)
        for x, y, st in drawn:
            ax.scatter(x, y, s=110 if st in END_STAGES else (120 if st == "agn_ign" else 70), marker=P5_STAGE_MARKER[st], c=[AV_COLORS[k]],
                       edgecolors="k" if st in END_STAGES else "none", linewidths=0.9, zorder=7)
        out.append((k, len(gks), pts))
    ax.axhline(KAPPA_PRESSURE, color="0.3", lw=1.0, ls="--", zorder=1)
    ax.set_yscale(P5_YSCALE)
    ax.set_ylim((0.02, 1.15) if P5_YSCALE == "log" else (-0.02, 1.02))
    ax.tick_params(direction="in", top=True, right=True); ax.grid(False)
    return out


# ── figures B / C: one grammar — categorical x (critical points | intervals), groups side by side with a bootstrap interval ──
def _groups_and_columns(labels, nb):
    """(lines, columns) of the sequence figures: each a list of (key, colour, name); key = ('cls', name) | ('av', bin)."""
    cls_items = [(("cls", c), CLASS_COLOR_PRES[c], CLASS_NAME[c]) for c in P3_CLASSES]
    av_items = [(("av", k), AV_COLORS[k], labels[k]) for k in range(nb)]
    return (cls_items, av_items) if P5_SEQ_LINES == "class" else (av_items, cls_items)


def _members(gks, sel):
    """Galaxies of `gks` in the selection sel = ('cls', class) | ('av', bin) | None."""
    if sel is None:
        return list(gks)
    kind, val = sel
    return [g for g in gks if (_gal_cls[g] == val if kind == "cls" else _bin_of[g] == val)]


def seq_panel(ax, xs, groups, values, ylabel=None, zero=None, ylim=None, band=None):
    """values(group_key, i) -> array of per-galaxy values at x index i. Draws median + bootstrap 16–84 per group (offset in x);
    `band` = (lo, hi) shaded across the panel (the observed range). Returns {(group_key, i): (med, lo, hi, n)}."""
    res = {}
    ng = len(groups)
    for gi, (key, color, name) in enumerate(groups):
        off = (gi - (ng - 1) / 2) * 0.18
        xx, yy, lo, hi = [], [], [], []
        for i, x in enumerate(xs):
            med, l16, h84, n = _bmed(values(key, i))
            res[(key, i)] = (med, l16, h84, n)
            if np.isfinite(med):
                xx.append(i + off); yy.append(med); lo.append(med - l16); hi.append(h84 - med)
        if len(xx):
            ax.errorbar(xx, yy, yerr=np.array([lo, hi]), color=color, lw=2.2, elinewidth=1.4, capsize=3, marker="o", ms=6.5, mec="k", mew=0.6, zorder=5, label=name)
    if band is not None and np.all(np.isfinite(band)):
        ax.axhspan(band[0], band[1], color=P5_OBS_COLOR, alpha=0.13, lw=0, zorder=0)
        ax.axhline(band[0], color=P5_OBS_COLOR, lw=0.8, ls=":", alpha=0.7, zorder=1); ax.axhline(band[1], color=P5_OBS_COLOR, lw=0.8, ls=":", alpha=0.7, zorder=1)
    if zero is not None:
        ax.axhline(zero, color="0.3", lw=1.0, ls="--", zorder=1)
    ax.set_xlim(-0.6, len(xs) - 0.4)
    ax.set_xticks(range(len(xs)))
    ax.set_xticklabels(xs)
    if ylim is not None:
        ax.set_ylim(*ylim)
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=P5_FONT["label"])
    ax.tick_params(direction="in", top=True, right=True, labelsize=P5_FONT["tick"]); ax.tick_params(axis="x", labelsize=P5_FONT["tick"] - 1); ax.grid(False)
    return res


def _zone_frame(ax, zone, tag=False):
    """The zone of a row: spines + tick marks in the zone colour; `tag` adds the rotated zone box left of the y label (first column)."""
    if zone not in P5_ZONE_COLOR:
        return
    c = P5_ZONE_COLOR[zone]
    for s in ax.spines.values():
        s.set_edgecolor(c); s.set_linewidth(1.7)
    ax.tick_params(color=c)
    if tag:
        ax.text(-0.36, 0.5, P5_ZONE_TAG[zone], transform=ax.transAxes, rotation=90, ha="center", va="center", color="white",
                fontsize=P5_FONT["tag"], fontweight="bold", bbox=dict(boxstyle="round,pad=0.35", fc=c, ec="none"), zorder=30)


def _seq_header(ax, cname_, lines, gk_col):
    """Column header: the column's name and, per line group, its number of galaxies (the class legend itself is figure-level)."""
    ax.text(0.0, 1.02, f"{cname_}  (N = {len(gk_col)})\n" + " $\\cdot$ ".join(f"{nm} {len(_members(gk_col, key))}" for key, _, nm in lines),
            transform=ax.transAxes, ha="left", va="bottom", fontsize=P5_FONT["header"] - 1.5, color="0.2", linespacing=1.15)


def _seq_class_legend(fig, lines):
    """The line groups (classes | A_V bins) once per figure, top right, on the title line."""
    fig.legend([Line2D([], [], color=c, lw=2.2, marker="o", mec="k", mew=0.6, ms=6.5) for _, c, _ in lines], [nm for _, _, nm in lines],
               loc="lower right", bbox_to_anchor=(1.0, 1.0), ncol=len(lines), frameon=False, fontsize=P5_FONT["legend"] + 1, handlelength=1.8, columnspacing=1.4)


def _pack(clauses, width=150, sep="; "):
    """Caption lines from clauses (never split inside a clause, so mathtext stays balanced), greedily packed to `width` characters."""
    lines, cur = [], ""
    for c in clauses:
        if cur and len(cur) + len(sep) + len(c) > width:
            lines.append(cur); cur = c
        else:
            cur = (cur + sep + c) if cur else c
    return "\n".join(lines + ([cur] if cur else []))


def _row_limits(res_list, kind, band=None):
    """Common y limits of one quantity over all its panels from the drawn medians +- bootstrap (and the observed band)."""
    vals = np.array([v for res in res_list for (m, l, h, n) in res.values() for v in (l, h) if np.isfinite(v)], float)
    if band is not None and np.all(np.isfinite(band)) and kind == "log":
        vals = np.append(vals, band)
    if kind == "kappa":
        return (-0.02, 1.02)
    if vals.size == 0:
        return None
    if kind == "log":
        vals = vals[vals > 0]
        return (vals.min() / 1.8, vals.max() * 1.8) if vals.size else None
    lo, hi = vals.min(), vals.max()
    if kind == "lin" and band is not None and np.all(np.isfinite(band)):
        d = np.array([v for res in res_list for (m, l, h, n) in res.values() for v in (l, h) if np.isfinite(v)], float)
        if d.size:                                         # the data decide the range; the band's lower edge always shows, its upper edge only when close
            lo, top = d.min(), max(d.max(), band[0] * 1.15)
            hi = band[1] if band[1] <= 1.4 * top else top
    pad = 0.08 * (hi - lo if hi > lo else 1.0)
    if kind == "kappa_zoom":
        return (max(-0.02, lo - pad), min(1.02, hi + pad))
    return (max(0.0, lo - pad) if lo >= 0 else lo - pad, hi + pad)


def _stage_vals(R, gks, stage, col):
    r = R[(R["gkey"].isin(gks)) & (R["stage"] == stage)]
    return r[col].to_numpy(float)


def _interval_vals(R, gks, a, b, col, norm=None):
    """Paired per-galaxy change col(b) - col(a) over the interval a -> b (both epochs measured); `norm` divides it by the cosmic
    time at the "end" (b) | "start" (a) of the interval | the "anchor" of the galaxy."""
    ra = R[(R["gkey"].isin(gks)) & (R["stage"] == a)].set_index("gkey")
    rb = R[(R["gkey"].isin(gks)) & (R["stage"] == b)].set_index("gkey")
    j = ra.index.intersection(rb.index)
    if not len(j):
        return np.zeros(0)
    d = rb.loc[j, col].to_numpy(float) - ra.loc[j, col].to_numpy(float)
    if norm == "end":
        d = d / rb.loc[j, "t_gyr"].to_numpy(float)
    elif norm == "start":
        d = d / ra.loc[j, "t_gyr"].to_numpy(float)
    elif norm == "anchor":
        d = d / np.array([_gal_tanc.get(g, np.nan) for g in j], float)
    return d


P5_ROWS_CSV, SEQ_ROWS, INT_ROWS = [], [], []
P5_INTERVALS = [(P5_HAVE[i], P5_HAVE[i + 1]) for i in range(len(P5_HAVE) - 1)]
for sname, mrange, stag in P5_MASS_SAMPLES:
    m_sample, edges, _bin_of = p5_sample(mrange)
    nb = len(edges) + 1
    labels = _bin_labels(edges)
    gk_all = list(GK[m_sample])
    print(f"\n{sname.replace('$', '').replace(chr(92) + 'log', 'log').replace(chr(92) + 'star', '*')}: {int(m_sample.sum())} Q galaxies "
          f"(quench event, anchor A_V, class in {P3_CLASSES}); A_V edges {np.round(edges, 3)} mag; per class x bin: "
          + "; ".join(f"{c} " + "/".join(str(sum(1 for g in gk_all if _gal_cls[g] == c and _bin_of[g] == k)) for k in range(nb)) for c in P3_CLASSES))
    sfm = (K5_SF["log_mstar"] > mrange[0]) & (K5_SF["log_mstar"] < mrange[1])
    # ── figure A, one per component ──
    for comp in P5_COMPONENTS:
        cname, kcol, ncol, klabel = comp
        fig, axs = pres_grid(len(P5_ZONES), [""] * len(P5_COLUMNS), height=3.4, width=4.4)
        for i, zone in enumerate(P5_ZONES):
            for j, col in enumerate(P5_COLUMNS):
                ax = axs[i, j]
                cls = None if col == "all" else col
                if P5_SHOW_SF:
                    S = K5_SF[sfm & (K5_SF["zone"] == zone)]
                    ax.scatter(S["age_mw_gyr"], S[kcol], s=7, c="0.6", alpha=0.3, lw=0, zorder=2)
                out = kappa_panel(ax, comp, zone, cls, m_sample, _bin_of, nb)
                n_col = sum(n for _, n, _ in out)
                if i == 0:
                    ax.text(0.0, 1.02, f"{'all classes' if cls is None else cls} (N={n_col})", transform=ax.transAxes, ha="left", va="bottom", fontsize=9.5, color="0.25")
                if j == 0:
                    ax.set_ylabel(f"{P5_ZONE_NAME[zone]}\n{klabel}")
                else:
                    ax.tick_params(labelleft=False)
                if i == len(P5_ZONES) - 1:
                    ax.set_xlabel("stellar age of the zone [Gyr]")
                else:
                    ax.tick_params(labelbottom=False)
                ax.legend([Line2D([], [], color=AV_COLORS[k], lw=2.2, marker="o", mec="k", ms=6) for k, _, _ in out],
                          [f"{labels[k]}  (N={n})" for k, n, _ in out], loc="upper right", frameon=False, fontsize=7.5, handlelength=1.8)
                for k, n, pts in out:
                    for d in pts:
                        P5_ROWS_CSV.append(dict(sample=stag, component=cname, zone=zone, column=col, av_bin=k, av_bin_label=labels[k].replace("$", ""),
                                                n_gal=n, stage=d["stage"], n_with_epoch=d["n_stage"], n_measured=d["n"],
                                                age_gyr=d.get("x", np.nan), t_gyr=d.get("t", np.nan), kappa=d.get("y", np.nan),
                                                kappa_boot16=d.get("ylo", np.nan), kappa_boot84=d.get("yhi", np.nan),
                                                kappa16=d.get("y16", np.nan), kappa84=d.get("y84", np.nan)))
        fig.text(0.0, 1.0, f"{sname}: {klabel} of the zone against its stellar age, " + " $\\rightarrow$ ".join(P5_STAGE_SHORT[st] for st in P5_HAVE),
                 fontsize=9.5, color="0.25", ha="left", va="bottom")
        _items = [(Line2D([], [], marker=P5_STAGE_MARKER[st], ls="", ms=9 if st == "agn_ign" else 7, mfc="0.4", mec="k" if st in END_STAGES else "none"), P5_STAGE_LABEL[st]) for st in P5_HAVE]
        _items += [(Line2D([], [], color="0.4", lw=2.2), f"bin median track (points in the event order; >= {NMIN_BIN} galaxies with a measurable value)"),
                   (Line2D([], [], color="0.3", lw=1.0, ls="--"), rf"$\kappa_{{\rm rot}} = {KAPPA_PRESSURE:g}$: rotation- / pressure-supported")]
        if P5_SHOW_INDIVIDUAL:
            _items.append((Line2D([], [], color="0.4", lw=0.7, alpha=0.5), "individual galaxy tracks (their own time order)"))
        if P5_SHOW_SF:
            _items.append((Line2D([], [], marker="o", ls="", ms=5, mfc="0.6", mec="none"), f"SF controls, same mass sample, at their anchor (N={int(sfm.sum()) // _nz})"))
        fig.legend([h for h, _ in _items], [l for _, l in _items], loc="upper center", ncol=3, frameon=False, fontsize=8.5, bbox_to_anchor=(0.5, 0.0),
                   title=(f"{AV_DEF}; bins: {'terciles of the mass sample' if P5_AV_EDGES is None else 'fixed edges'}; zones spherical about the galaxy centre; "
                          f"a kappa needs >= {NKIN_MIN} weighted particles; {END_DEF}"), title_fontsize=8.5)
        pres_finish(fig, axs, f"kappa_vs_age_{cname}_{stag}", frame=(P5_COLUMNS[0] == "all"))

    # ── figure B (three figures): the critical-point sequence, groups side by side — gas rotation | stellar rotation (zoomed) |
    #    age, dust fraction and the elapsed time of every interval normalised to the cosmic time ──
    lines, columns = _groups_and_columns(labels, nb)
    xs = [P5_STAGE_SHORT[st] for st in P5_HAVE]
    xsC = [f"{P5_STAGE_SHORT[a]}\n$\\rightarrow$ {P5_STAGE_SHORT[b]}" for a, b in P5_INTERVALS]
    _dt_lab = r"$\Delta t\,/\,t_{\rm cosmic}$" if P5_DT_NORM else r"$\Delta t$ [Gyr]"
    _dt_def = {"end": "the cosmic time at the end of the interval", "start": "the cosmic time at the start of the interval", "anchor": "the cosmic time of the anchor", None: None}[P5_DT_NORM]
    _grammar = ("lines = AGN class, columns = anchor-$A_V$ bins" if P5_SEQ_LINES == "class" else "lines = anchor-$A_V$ bins, columns = AGN class") \
               + " (" + " $\\rightarrow$ ".join(P5_STAGE_SHORT[st] for st in P5_HAVE) + " = the canonical order of the critical points)"
    for stem, quants, ftitle, dt_row in P5_SEQ_FIGS:
        B_ROWS = [(zone, col, lab, kind) for col, lab, kind in quants for zone in P5_ZONES]
        nrow = len(B_ROWS) + (1 if (dt_row and P5_INTERVALS) else 0)
        fig, axs = pres_grid(nrow, [""] * len(columns), height=2.7, width=4.6)
        res_by_q = {col: [] for col, _, _ in quants}
        for j, (ckey, _, cname_) in enumerate(columns):
            gk_col = _members(gk_all, ckey)
            for i, (zone, col, ylab, kind) in enumerate(B_ROWS):
                ax = axs[i, j]
                R = K5[K5["zone"] == zone]
                band = P5_OBS_BAND.get(col)
                res = seq_panel(ax, xs, lines, lambda key, k, R=R, gk_col=gk_col, col=col: _stage_vals(R, _members(gk_col, key), P5_HAVE[k], col),
                                ylabel=ylab if j == 0 else None, zero=KAPPA_PRESSURE if kind.startswith("kappa") else (P5_SSFR_LINE if col == "ssfr" else None), band=band)
                if kind == "log":
                    ax.set_yscale("log")
                res_by_q[col].append(res)
                _zone_frame(ax, zone, tag=(j == 0))
                for (key, k), (med, lo, hi, n) in res.items():
                    SEQ_ROWS.append(dict(sample=stag, figure=stem, zone=zone, quantity=col, stage=P5_HAVE[k], column=cname_.replace("$", ""), line=str(key[1]),
                                         n_line=len(_members(gk_col, key)), n=n, median=med, lo16=lo, hi84=hi))
                if i == 0:
                    _seq_header(ax, cname_, lines, gk_col)
                if j > 0:
                    ax.tick_params(labelleft=False)
                if i < nrow - 1:
                    ax.tick_params(labelbottom=False)
            if dt_row and P5_INTERVALS:
                ax = axs[len(B_ROWS), j]
                R = K5[K5["zone"] == P5_ZONES[0]]
                res = seq_panel(ax, xsC, lines, lambda key, k, R=R, gk_col=gk_col: _interval_vals(R, _members(gk_col, key), *P5_INTERVALS[k], "t_gyr", norm=P5_DT_NORM),
                                ylabel=_dt_lab if j == 0 else None, zero=0.0)
                ax.tick_params(axis="x", labelsize=P5_FONT["tick"] - 1)
                for (key, k), (med, lo, hi, n) in res.items():
                    INT_ROWS.append(dict(sample=stag, zone="", quantity=f"dt_norm_{P5_DT_NORM}" if P5_DT_NORM else "t_gyr", interval=f"{P5_INTERVALS[k][0]}->{P5_INTERVALS[k][1]}",
                                         column=cname_.replace("$", ""), line=str(key[1]), n_line=len(_members(gk_col, key)), n_paired=n, median=med, lo16=lo, hi84=hi, frac_negative=np.nan))
                if j > 0:
                    ax.tick_params(labelleft=False)
        for col, _, kind in quants:                       # one y range per quantity over both zones and every column
            lim = _row_limits(res_by_q[col], kind, P5_OBS_BAND.get(col))
            if lim is not None:
                for i, (zone, c_, _, _) in enumerate(B_ROWS):
                    if c_ == col:
                        for ax in axs[i]:
                            ax.set_ylim(*lim)
        fig.text(0.0, 1.0, f"{sname} — {ftitle} at the critical points", fontsize=P5_FONT["header"], color="0.2", ha="left", va="bottom")
        _seq_class_legend(fig, lines)
        _items = [(Line2D([], [], color="0.4", lw=2.2, marker="o", mec="k", mew=0.6, ms=6.5),
                   f"group median at each critical point, galaxy-bootstrap 16–84 % (>= {NMIN_BIN} galaxies with a measurable value" + ("; H$_2$ rows: survivors)" if "kappa_H2" in [q[0] for q in quants] else ")"))]
        if any(q[2].startswith("kappa") for q in quants):
            _items.append((Line2D([], [], color="0.3", lw=1.0, ls="--"), rf"$\kappa_{{\rm rot}} = {KAPPA_PRESSURE:g}$: rotation- / pressure-supported"))
        if any(q[0] in P5_OBS_BAND for q in quants):
            _qs = [q[0] for q in quants]
            _parts = [t for q, t in (("ssfr", f"sSFR {P5_OBS_BAND['ssfr'][0]:.1e}–{P5_OBS_BAND['ssfr'][1]:.1e} yr$^{{-1}}$"),
                                     ("age_mw_gyr", f"age {P5_OBS_BAND['age_mw_gyr'][0]:.1f}–{P5_OBS_BAND['age_mw_gyr'][1]:.1f} Gyr"),
                                     ("fdust", f"$M_{{\\rm dust}}/M_\\star$ {P5_OBS_BAND['fdust'][0]:.1e}–{P5_OBS_BAND['fdust'][1]:.1e}")) if q in _qs and q in P5_OBS_BAND]
            _btxt = "range of the ALMA-C11 detections (whole galaxy, fiducial CIGALE run): " + (", ".join(_parts) + f" of the {P5_OBS_N} {P5_OBS_DET.replace('det_', '')}-detected" if _parts else "")
            if "fH2" in P5_OBS_BAND and "fH2" in _qs:
                _btxt += ("; " if _parts else "") + f"$M_{{\\rm H_2}}/M_\\star$ {P5_OBS_BAND['fH2'][0]:.1e}–{P5_OBS_BAND['fH2'][1]:.1e} of the {P5_OBS_N_H2} CO-detected (H-only, $\\div${P5_OBS_HE:g} He)"
            _items.append((Rectangle((0, 0), 1, 1, fc=P5_OBS_COLOR, alpha=0.2, ec=P5_OBS_COLOR, ls=":"), _btxt))
        if dt_row and P5_INTERVALS:
            _items.append((Line2D([], [], color="0.3", lw=1.0, ls="--"), "last row: the elapsed time of each interval, paired per galaxy" + (f", divided by {_dt_def}" if _dt_def else "")
                           + "; negative = the canonical order was inverted (e.g. jet mode only after SFT)"))
        _clauses = [_grammar, AV_DEF, "zones spherical about the galaxy centre", END_DEF]
        if any(q[2].startswith("kappa") for q in quants):
            _clauses.append(f"a kappa needs >= {NKIN_MIN} weighted particles")
        if {"fdust", "fH2", "ssfr"} & {q[0] for q in quants}:
            _clauses.append(f"$M_{{\\rm dust}}/M_\\star$, $M_{{\\rm H_2}}/M_\\star$ and sSFR from the KS tracks' projected apertures (core: $r<3.2$ kpc disc; outskirt: 3.2–10 kpc annulus), "
                            f">= {NSTAR_AP_MIN} stars, a zero mass dropped")
        if "ssfr" in [q[0] for q in quants]:
            _clauses.append(f"sSFR = SFR({P5_SSFR_COL.replace('sfr', '')} Myr) / $M_\\star$ of the aperture, a zero SFR floored at {P5_SSFR_FLOOR:g} yr$^{{-1}}$ "
                            f"(a group median on the floor = most of its galaxies form no stars there); dashed: {P5_SSFR_LINE:g} yr$^{{-1}}$")
        fig.legend([h for h, _ in _items], [l for _, l in _items], loc="upper center", ncol=1, frameon=False, fontsize=P5_FONT["note"], bbox_to_anchor=(0.5, 0.0),
                   title=_pack(_clauses), title_fontsize=P5_FONT["note"])
        pres_finish(fig, axs, f"{stem}_{stag}", frame=False)

    # ── figure C: paired changes over the intervals between consecutive critical points ──
    if P5_INTERVALS:
        C_ROWS = [(zone, kcol, "$\\Delta$" + klabel) for _, kcol, _, klabel in P5_COMPONENTS for zone in P5_ZONES] \
               + [(None, "t_gyr", "elapsed time\n$\\Delta t$ [Gyr]")]
        fig, axs = pres_grid(len(C_ROWS), [""] * len(columns), height=2.7, width=4.6)
        for j, (ckey, _, cname_) in enumerate(columns):
            gk_col = _members(gk_all, ckey)
            for i, (zone, col, ylab) in enumerate(C_ROWS):
                ax = axs[i, j]
                R = K5[K5["zone"] == (zone or P5_ZONES[0])]
                res = seq_panel(ax, xsC, lines, lambda key, k, R=R, gk_col=gk_col, col=col: _interval_vals(R, _members(gk_col, key), *P5_INTERVALS[k], col),
                                ylabel=ylab if j == 0 else None, zero=0.0)
                if zone:
                    _zone_frame(ax, zone, tag=(j == 0))
                for (key, k), (med, lo, hi, n) in res.items():
                    v = _interval_vals(R, _members(gk_col, key), *P5_INTERVALS[k], col)
                    INT_ROWS.append(dict(sample=stag, zone=zone or "", quantity=col, interval=f"{P5_INTERVALS[k][0]}->{P5_INTERVALS[k][1]}",
                                         column=cname_.replace("$", ""), line=str(key[1]), n_line=len(_members(gk_col, key)), n_paired=n,
                                         median=med, lo16=lo, hi84=hi, frac_negative=float(np.mean(v < 0)) if len(v) else np.nan))
                if i == 0:
                    _seq_header(ax, cname_, lines, gk_col)
                if j > 0:
                    ax.tick_params(labelleft=False)
                if i < len(C_ROWS) - 1:
                    ax.tick_params(labelbottom=False)
                else:
                    ax.tick_params(axis="x", labelsize=P5_FONT["tick"] - 1)
        fig.text(0.0, 1.0, f"{sname} — the change over each interval between consecutive critical points, paired per galaxy", fontsize=P5_FONT["header"], color="0.2", ha="left", va="bottom")
        _seq_class_legend(fig, lines)
        fig.legend([Line2D([], [], color="0.4", lw=2.2, marker="o", mec="k", mew=0.6, ms=6.5), Line2D([], [], color="0.3", lw=1.0, ls="--")],
                   [f"median of the per-galaxy differences, galaxy-bootstrap 16–84 % (>= {NMIN_BIN} paired galaxies)",
                    "no change; a negative $\\Delta t$ = the later critical point of the canonical order came first (e.g. jet mode only after SFT)"],
                   loc="upper center", ncol=1, frameon=False, fontsize=P5_FONT["note"], bbox_to_anchor=(0.5, 0.0),
                   title=_pack([_grammar, "both epochs of a galaxy measured, then the group median", AV_DEF, "zones spherical about the galaxy centre",
                                f"a kappa needs >= {NKIN_MIN} weighted particles", END_DEF]), title_fontsize=P5_FONT["note"])
        pres_finish(fig, axs, f"kappa_intervals_{stag}", frame=False)

P5 = pd.DataFrame(P5_ROWS_CSV)
P5.to_csv(os.path.join(PAPERDIR, "paper_kappa_vs_age.csv"), index=False)
pd.DataFrame(SEQ_ROWS).to_csv(os.path.join(PAPERDIR, "paper_kappa_sequence.csv"), index=False)
pd.DataFrame(INT_ROWS).to_csv(os.path.join(PAPERDIR, "paper_kappa_intervals.csv"), index=False)
print(f"\ntables -> {os.path.join(PAPERDIR, 'paper_kappa_{vs_age,sequence,intervals}.csv')} ({len(P5)} / {len(SEQ_ROWS)} / {len(INT_ROWS)} rows)")
if P5_OBS_BAND:
    _sq = pd.DataFrame(SEQ_ROWS)
    print(f"\ncore at each critical point vs the ALMA-C11 {P5_OBS_DET} band (class medians pooled over the A_V bins are in the CSV per column; here: all bins of the sample):")
    for sname, mrange, stag in P5_MASS_SAMPLES:
        m_sample, edges, _bin_of = p5_sample(mrange)
        gk_all = list(GK[m_sample])
        R = K5[K5["zone"] == CORE_LAB]
        for col, nm in (("ssfr", "sSFR /yr"), ("age_mw_gyr", "age [Gyr]"), ("fdust", "Mdust/M*"), ("fH2", "MH2/M*")):
            if col not in P5_OBS_BAND:
                continue
            lo, hi = P5_OBS_BAND[col]
            cells = []
            for st in P5_HAVE:
                cells.append(f"{P5_STAGE_SHORT[st]}: " + " | ".join(
                    (lambda v: f"{c[:3]} {np.median(v):.3g} ({np.mean((v >= lo) & (v <= hi)):.0%} in band, N={len(v)})" if len(v) else f"{c[:3]} —")(
                        (lambda a: a[np.isfinite(a)])(_stage_vals(R, _members(gk_all, ("cls", c)), st, col))) for c in P3_CLASSES))
            print(f"  {stag} core {nm:>9s} [band {lo:.3g}–{hi:.3g}]  " + "  ||  ".join(cells))
print("galaxies with a measurable kappa per critical point (n_measured / n_with_epoch), summed over the bins of each column:")
_agg = P5.groupby(["sample", "component", "zone", "column", "stage"])[["n_measured", "n_with_epoch"]].sum()
_agg["frac"] = _agg["n_measured"] / _agg["n_with_epoch"].replace(0, np.nan)
print(_agg.unstack("stage").reindex(columns=P5_HAVE, level=1).round(2).to_string())
if INT_ROWS:
    print("\npaired change per interval, all A_V bins pooled per class (median [boot 16, 84] (N paired)):")
    for sname, mrange, stag in P5_MASS_SAMPLES:
        m_sample, edges, _bin_of = p5_sample(mrange)
        gk_all = list(GK[m_sample])
        print(f"  {sname.replace('$', '').replace(chr(92) + 'log', 'log').replace(chr(92) + 'star', '*')}:")
        for zone, col, lab in [(z, kc, f"d{cn} {P5_ZONE_SHORT[z]}") for cn, kc, _, _ in P5_COMPONENTS for z in P5_ZONES] + [(P5_ZONES[0], "t_gyr", "dt [Gyr]")] \
                             + [(z, "age_mw_gyr", f"dage {P5_ZONE_SHORT[z]}") for z in P5_ZONES]:
            R = K5[K5["zone"] == zone]
            cells = []
            for a, b in P5_INTERVALS:
                cells.append(f"{P5_STAGE_SHORT[a]}->{P5_STAGE_SHORT[b]}: " + " | ".join(
                    (lambda m: f"{c[:3]} {m[0]:+.2f} [{m[1]:+.2f},{m[2]:+.2f}] ({m[3]})" if np.isfinite(m[0]) else f"{c[:3]} — ({m[3]})")(_bmed(_interval_vals(R, _members(gk_all, ('cls', c)), a, b, col)))
                    for c in P3_CLASSES))
            print(f"    {lab:>16s}  " + "  ||  ".join(cells))


## Part 6 — figures G and H: cheap observables of old quiescent galaxies against their ISM content

Two figures (`P6_FIGS`: stem, panel grid, model mode, options), the observed points coloured by their $A_V$ (`P6_OBS_COLOUR`, linear 0–1.4 mag), no fitted plane and nothing painted. The literature always carries both cuts; the models carry the mass cut and, per figure, the age cut (`logage_min`, None = none) and their **source** (`model`: `"sim"` = the SIMBA truth, `"cigale"` = the CIGALE fits of the mock core photometry; default `P6_MODEL` = sim):

| figure | panels | model tracks | what it shows |
|---|---|---|---|
| `paper_sigma_age` (G) | $\Sigma_{\rm e} = M_\star/2\pi R_{\rm e}^2$ vs stellar age | the strong-coupling quenched galaxies split at `P6_SPLIT` = $\log(M_{\rm dust}/M_\star) = -3.75$ (per-galaxy median): **dusty** (red, N = 5) and **dust-poor** (orange dashed, N = 25) running medians | where dusty and dust-poor models sit among the observed QGs, LEGA-C and 3D-HST; the dusty models are older (4.1 vs 3.2 Gyr, $p = 0.10$) and no more compact — their real distinction is a rotating H$_2$ disc ($\kappa_{\rm rot}^{\rm H_2}$ 0.79 vs 0.26, $p < 10^{-3}$; printed) |
| `paper_ism_prediction` (H) | rows $M_{\rm dust}/M_\star$, $M_{\rm H_2}/M_\star$ (expensive) × columns stellar age, $\Sigma_{\rm e}$, SED sSFR (cheap) | weak / intermediate / strong running median + band, **no age cut**, **`model="cigale"`**: the CIGALE fits of the mock **core** (0–3.2 kpc, `dust_on`) photometry supply $M_\star$, $M_{\rm dust}$, SFR and age (172 of the 178 mass-cut quenched galaxies have one; N = 60 / 67 / 28); all quenched galaxies under the mass cut as the thin dotted line (`P6_PRED_ALLQ`) | the ISM content SIMBA predicts for a quiescent galaxy of a given cheap observable, measured the way the observations are (same SED code, same quantities), with the measured ALMA-C11 / Spilker+18 / ADF22-QG1 values as the check: sSFR orders the dust content in every class ($\rho$ = +0.5 to +0.6) and age does in the weak / intermediate ones ($\rho \approx -0.5$, young = dusty); $\Sigma_{\rm e}$ only in the weak class ($\rho = -0.5$); every detected observed value sits 0.7–1.6 dex above the tracks, the ALMA-C11 non-detection limits sit on them |
| `paper_ism_prediction_sim` (H$_{\rm sim}$) | the grid of H | the same classes, no age cut, **`model="sim"`**: the SIMBA truth in the standard apertures (0–10 kpc ISM over the 0–32 kpc stars), all quenched dotted | H without CIGALE and without the core: the estimator and the aperture at once. Against the caesar catalogue of the same galaxies (checked on the 266 Q of the selection table) the projected 0–10 kpc disc carries +0.26 dex more dust and +0.34 dex more gas than the FOF galaxy (a disc is a cylinder through the 100 kpc cut-out) and the 0–100 kpc rung +1.3 dex — no rung is the catalogue galaxy, which is why `paper_ism_prediction_boxes.ipynb` takes its m25 reference from the catalogue rows of the sample and keeps this figure as the optional `M25_REF_MODEL = "sim"` |

* **Observed points** — every ALMA-C11 source (fiducial CIGALE $A_V$, $M_\star$, SFR, mass-weighted age; $M_{\rm dust}$ and $M_{\rm H_2}$ with their censors —
  arrows = upper limits; $R_{\rm e}$ and Sérsic $n$ = COSMOS-Web semi-major Sérsic fit, ACS where JWST is missing; the six controls as small circles), the eight
  Spilker+18 LEGA-C passive galaxies (CO(2–1) only, diamonds; $A_V$, $M_\star$, sSFR, age, $R_{\rm e}$, $n$ from their LEGA-C DR3 / de Graaff+21 rows) and
  ADF22-QG1 (Umehata+25: CO(3–2) at $z = 3.09$, dust undetected; sSFR < 1.8e-12, $t_{50}$, $A_V$, $R_{\rm e}$ from Kubo+17/21; `P6_UMEHATA`).
* **Observed tracks** — LEGA-C (de Graaff+21 MAGPHYS) and 3D-HST (Momcheva+16 FAST; van der Wel+14/21 sizes to rest 5000 Å) running medians of y in
  equal-count x bins with the 16–84 % band, drawn where the literature carries both axes, under the user's two cuts only: $\log M_\star > 10.25$ and
  $\log({\rm age/yr}) > 9$ (SED-fit ages, not homogeneous with each other nor with the mass-weighted simulated ages; no UVJ or redshift cut unless
  `P6_LIT_QUIESCENT` / `P6_LIT_Z`).
* **Model tracks** — the quenched galaxies under the same two cuts (anchor $\log M_\star$ of the selection table, the model's age); rows = sightlines,
  equal-count x bins of $\ge$ `P6_NMIN_X` galaxies (2 for a group smaller than twice that), galaxy-bootstrap 16–84 % band. Two sources (`_build_models`,
  one frame per source in `P6_MODELS`):
  * `model="sim"` (figure G): dust and H$_2$ from the projected 0–10 kpc disc (H$_2$ × 1.36 for He, SIMBA H$_2$ is hydrogen-only) over the 0–32 kpc stars;
    mass-weighted age of the 0–32 kpc stars; sSFR = SFR$_{100}$/$M_\star$ of the 0–32 kpc stars (zero drawn at 1e-13); RT $A_V$ of the 0–10 kpc disc;
    $\kappa_{\rm rot}^{\rm H_2}$ of the spherical r < 10 kpc rung (Part 8j0, sightline-independent).
  * `model="cigale"` (figure H): the m25 Part 7f table `cigale_region_results.fits` — one CIGALE run per (galaxy, region, sightline) on the region's own
    mock photometry with its own SFH, metallicity and dust-mass pins — restricted to `P6_CIG_REGION` = **core** (0–3.2 kpc) and `P6_CIG_ARM` = `dust_on`:
    $M_\star$ = `bayes.stellar.m_star`, $M_{\rm dust}$ = `bayes.dust.mass`, SFR = `P6_CIG_SFR` (`bayes.sfh.sfr`, the observed fiducial column), age =
    `bayes.stellar.age_m_star` (pinned to the injected SFH, so it is the core's true mass-weighted age), $A_V$ = `Av_ISM` minus the region's dust_off
    zero point (`P6_CIG_AV_ZP`). $M_{\rm H_2}$ has no CIGALE product: the SIMBA H$_2$ of the same region (`P6_REGION_AP`, × 1.36) over the CIGALE $M_\star$.
    $\Sigma_{\rm e}$ keeps the simulated half-mass radius with the CIGALE core $M_\star$ aperture-corrected to the 0–32 kpc disc by the simulated curve of
    growth (`mstar_tot`); $\kappa_{\rm rot}^{\rm H_2}$ of the r < 3.2 kpc rung. Closure printed: core fit − truth $\log M_\star$ −0.06 dex (16–84 %
    −0.18 to −0.01), $\log M_{\rm dust}$ −0.08 dex (−0.38 to +0.10), 37 of 688 sightline rows fitted at zero dust (dropped, never floored).
    The `sim` split statistics of figure G are printed for the default source only.
* **Tables** — `paper_<stem>.csv` (model / literature tracks + `obs_vs_track`: each detected observed value against the strong track interpolated at its x,
  inside the track's range) and `paper_<stem>_points.csv` (the observed points with every column).

Other axes available for `P6_FIGS`: $A_V$, $R_{\rm e}$, $\kappa_{\rm rot}^{\rm H_2}$ (models only), Sérsic $n$ (observed + literature only); a panel draws
whichever datasets carry both of its axes (`who`). Needs Part 4a (`_frame`, `run_median`, the sampling floors). Inputs: `aperture_truth`, `annulus_ism_truth`,
`annulus_av_allincl`, `annulus_kinematics`, `cigale_region_results` (m25 Part 7f, any figure with `model="cigale"`), the selection table,
`obs_data/almac11/{almac11_gas_dust,age_sersic_sigma}.csv`,
`obs_data/literature/{av_re_literature,spilker18_legac}.csv`.


In [ ]:
# ── Part 6 — figures G and H: cheap observables of old quiescent galaxies against their ISM content; observations coloured by A_V ──
# G ("sigma_age"): Sigma_e vs stellar age, the strong-coupling models split at P6_SPLIT into a dusty and a dust-poor running median (no plane, nothing
# painted). H ("ism_prediction"): the prediction — the ISM content an observer cannot cheaply measure (rows: M_dust / M*, M_H2 / M*) as the models'
# running median against what they can (columns: stellar age, Sigma_e, sSFR), with the measured ALMA-C11 / Spilker+18 / ADF22-QG1 values as the check.
# Observed points: ALMA-C11 (fiducial CIGALE), Spilker+18 LEGA-C passive galaxies (structure + SED from LEGA-C DR3; CO only), ADF22-QG1 (Umehata+25);
# observed tracks: LEGA-C / 3D-HST running medians where the literature carries both axes; model tracks: the quenched galaxies, their M*, M_dust, SFR
# and age either the SIMBA truth (model "sim") or the CIGALE fits of the mock photometry of ONE region (model "cigale": the core) — per figure (P6_FIGS
# option `model`, default P6_MODEL): G keeps the truth, H is drawn from the CIGALE core fits — the same code and quantities as the observed points —
# and H_sim ("ism_prediction_sim") is H with the truth in the standard apertures: the estimator (CIGALE vs truth) and the core (0–3.2 kpc vs 0–10 / 0–32 kpc)
# at once; paper_ism_prediction_boxes can draw it behind its catalogue tracks (M25_REF_MODEL = "sim") — its default reference is the catalogue itself.
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.patheffects import withStroke
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import FuncFormatter
from matplotlib.lines import Line2D
from scipy.stats import spearmanr, mannwhitneyu

# (file stem, rows of (x, y) panels, model mode, options): "split" = one running median per P6_SPLIT group of the classes; "class" = one per class;
# options: classes (default P6_CLASSES), logage_min (default P6_LOGAGE_MIN; None = no age cut on the models), model (default P6_MODEL: "sim" | "cigale")
_H_GRID = [[("age", "fdust"), ("sigma_e", "fdust"), ("ssfr", "fdust")], [("age", "fh2"), ("sigma_e", "fh2"), ("ssfr", "fh2")]]
P6_FIGS = [("sigma_age", [[("age", "sigma_e")]], "split", {}),
           ("ism_prediction", _H_GRID, "class", dict(classes=["weak", "intermediate", "strong"], logage_min=None, model="cigale")),
           ("ism_prediction_sim", _H_GRID, "class", dict(classes=["weak", "intermediate", "strong"], logage_min=None, model="sim"))]   # H with the SIMBA truth
P6_OBS_COLOUR = "av"              # the observed points are coloured by this P6_AXES key (linear, P6_OBS_CLIM); the models carry no colour scale
P6_OBS_CLIM   = (0.0, 1.4)
P6_SPLIT      = ("lfdust", -3.75) # "split" mode: per-galaxy median log(M_dust / M*) at or above the threshold -> "dusty", below -> "dust-poor"
P6_SPLIT_STYLE = {"dusty":     dict(color="#b2182b", ls="-", marker="o"),            # warm colours + line style + marker: nothing to confuse with the A_V scale of the points
                  "dust-poor": dict(color="#e08214", ls=(0, (4.0, 1.6)), marker="s")}
P6_PRED_ALLQ  = True              # "class" mode: add the running median of ALL quenched galaxies under the cuts (thin dotted) behind the class track
P6_MODEL     = "sim"              # default source of the model tracks (a figure overrides it: P6_FIGS option `model`): "sim" = the SIMBA truth in the P6_AP /
                                  # P6_AP_TOT apertures | "cigale" = the m25 Part 7f CIGALE fits of the mock photometry of ONE region (P6_CIG_REGION) supply
                                  # M*, M_dust, SFR, age and A_V; M_H2 (no CIGALE product) is the SIMBA H2 of that region over the CIGALE M*
P6_CIG_REGION = "core"            # "cigale": the region fitted — core (0–3.2 kpc) | outskirt (3.2–32 kpc) | cgm (32–100 kpc); one run per (galaxy, region, sightline)
P6_CIG_ARM   = "dust_on"          # "cigale": the RT arm of the fits (dust_on: stars + dust; agn_on: + the Nenkova torus)
P6_CIG_SFR   = "bayes.sfh.sfr"    # "cigale": the SFR column — bayes.sfh.sfr is the observed fiducial (obs README); bayes.sfh.sfr100Myrs would match the sim sfr100
P6_CIG_AV_ZP = True               # "cigale": subtract the per-region dust_off zero point (Av_zp, the BC03-vs-FSPS colour offset of m25 7f) from Av_ISM
P6_REGION_AP = {"core": ("ap3kpc", None), "outskirt": ("ap32kpc", "ap3kpc"), "cgm": ("ap100kpc", "ap32kpc")}   # m25 REGION_DEFS: (ap_out, ap_in) cumulative rungs
P6_REGION_KIN = {"core": "ap3kpc", "outskirt": "ap32kpc", "cgm": "ap100kpc"}   # the 8j0 spherical rung of kappa_rot(H2) per region (cumulative: 8j0 has no 3.2–32 shell)
CIGALE_REGION_FITS = os.path.join(TABLEDIR, "cigale_region_results.fits")     # m25 Part 7f: one row per (arm, region, sightline, galaxy); dust.mass in kg
MSUN_KG      = 1.98892e30
P6_AP        = "ap10kpc"          # "sim": aperture of A_V, of M_dust / M_H2 and of kappa_rot(H2): the projected 0–10 kpc disc / the spherical r < 10 kpc rung (8j0)
P6_AP_TOT    = "ap32kpc"          # "sim": total stellar mass (Sigma_e, the fractions), sfr100 (sSFR) and the mass-weighted stellar age: the projected 0–32 kpc disc
                                  # ("cigale": the aperture-correction target of the region M* behind Sigma_e — see G6["mstar_tot"])
P6_COG       = ["ap1kpc", "ap3kpc", "ap10kpc", "ap32kpc"]   # curve of growth (m25 7e cumulative apertures) -> projected half-mass radius per sightline
P6_COG_KPC   = [1.0, 3.162, 10.0, 31.623]
P6_CLASSES   = ["strong"]         # the model tracks: AGN coupling classes drawn (a figure may override: P6_FIGS options)
P6_MASS_MIN  = 10.25              # the user's two cuts (models, LEGA-C, 3D-HST): anchor / catalogue log M* above this ...
P6_LOGAGE_MIN = 9.0               # ... and log(age / yr) above this (sims: mass-weighted age of the P6_AP_TOT stars; literature: the SED-fit age of each code);
                                  # the literature always; the models per figure (P6_FIGS options)
P6_HE        = 1.36               # SIMBA H2 is hydrogen-only; the observed alpha_CO masses include helium -> 1.36 M_H2 on the model side
P6_NBIN_X, P6_NMIN_X = 4, 3       # model tracks: at most this many equal-count x bins (rows = sightlines), at least this many galaxies per bin (Part 4a run_median);
                                  # a group of fewer than 2 x P6_NMIN_X galaxies is binned with 2 galaxies per bin instead (a handful still makes a track)
P6_LIT_NBIN, P6_LIT_NMIN = 8, 15  # literature tracks: equal-count x bins and objects per bin
P6_NKIN_MIN  = 10                 # a kappa_rot(H2) needs this many H2-weighted particles in the rung (8j0 n_H2)
P6_SSFR_FLOOR = 1e-13             # sSFR below this is DRAWN at the floor (log axis; quenched models at SFR = 0, FAST sSFR = -99)
P6_CMAP      = ("YlGnBu", 0.2, 1.0)   # sequential, truncated so the lightest step still reads on white
P6_RE        = "maj"              # "maj": semi-major half-light radii (van der Wel+14 convention; ALMA-C11 re_kpc) | "circ": circularised (re sqrt(q); ALMA-C11 Re_circ)
P6_LIT       = [("LEGA-C", "-"), ("3D-HST", (0, (4.5, 2.2)))]   # (sample, line style)
P6_LIT_Z     = None               # (zlo, zhi) or None: no redshift cut on the literature (the user's rule: mass + age only)
P6_LIT_QUIESCENT = False          # True: keep only UVJ-quiescent literature galaxies (Williams+09 box)
P6_OBS_CTRL  = True               # draw the six ALMA-C11 controls (no ALMA detection: limits only) as small circles
P6_UMEHATA   = dict(name="ADF22-QG1", ref="Umehata+25", z=3.0922, logM=11.11, re_kpc=1.01, av=0.5, logage=8.79, logMH2=10.26, logMd_ul=8.02, ssfr_ul=1.8e-12,
                    note="log M* FAST++ with NIRCam (Umehata+25 Tab. 1); r_eff 1.01 +- 0.04 kpc Subaru AO K' (Kubo+17); A_V,FAST++ 0.5 +- 0.1, log t50 8.79 and "
                         "sSFR_FAST++ < 1.8e-12 (Kubo+21 Tab. 1); log M_H2 10.26 +- 0.07 (alpha_CO 4.4, r31 0.5) and log M_d < 8.02 (Umehata+25 Tab. 1)")
OBS_STRUCT_CSV = os.path.join(os.getcwd(), "obs_data", "almac11", "age_sersic_sigma.csv")        # ALMA-C11 optical R_e (COSMOS-Web / ACS), Sersic n
LIT_CSV        = os.path.join(os.getcwd(), "obs_data", "literature", "av_re_literature.csv")      # LEGA-C + 3D-HST: A_V, R_e, n, log M*, sSFR, SED age
SPILKER_CSV    = os.path.join(os.getcwd(), "obs_data", "literature", "spilker18_legac.csv")       # Spilker+18 LEGA-C rows of the KS compilation (M_H2, no dust)
P6_FONT        = dict(label=12.0, tick=10.0, legend=9.0, note=8.3, tag=10.5)
_REGN = {"core": "0–3.2 kpc core", "outskirt": "3.2–32 kpc outskirt", "cgm": "32–100 kpc CGM"}[P6_CIG_REGION]
# what the model side of each axis is, per model source (the `{sims}` slot of the P6_AXES obs_note), and the legend line of the model tracks
P6_SIM_TXT = {"sim":    dict(av="RT 0–10 kpc", age="mass-weighted 0–32 kpc", ssfr=r"SFR$_{100\,\rm Myr}$ of the 0–32 kpc stars", sigma_e="projected half-mass radius",
                             fdust="0–10 kpc dust over the 0–32 kpc stars", fh2=r"1.36 x the 0–10 kpc H$_2$ over the 0–32 kpc stars"),
              "cigale": dict(av=f"CIGALE fit of the {_REGN}", age=f"CIGALE mass-w. age of the {_REGN} fit (pinned to its SFH)", ssfr=f"CIGALE SFR / $M_\\star$ of the {_REGN} fit",
                             sigma_e=f"projected half-mass radius, CIGALE {_REGN} $M_\\star$ aperture-corrected to 0–32 kpc",
                             fdust=f"CIGALE $M_{{\\rm dust}}$ / $M_\\star$ of the {_REGN} fit", fh2=f"1.36 x the {_REGN} H$_2$ over the CIGALE $M_\\star$ of the fit")}
P6_MODEL_NOTE = {"sim": "SIMBA truth: 0–10 kpc ISM, 0–32 kpc stars",
                 "cigale": f"CIGALE fits of the mock {_REGN} photometry ({P6_CIG_ARM}):\n$M_\\star$, $M_{{\\rm dust}}$, SFR and age of the fit"}
# axes: col = plot-space column of G6 / OBS6 / literature rows (log10 for log axes, linear otherwise), raw = the linear column, floor = values below are
# drawn there (log axes; without one a non-positive value is dropped), who = which datasets carry the quantity (s = models, o = observed points, l = literature),
# cens = OBS6 column with det / ul / none for the observed values
P6_AXES = {"av":      dict(col="lav", raw="av", log=True, lim=(2e-3, 4.0), who="sol", label=r"$A_V$ [mag]", short="A_V", nm=r"$A_V$", obs_note="SED fit / {sims} (sims)"),
           "sigma_e": dict(col="lsig", raw="sig_e", log=True, lim=(1.5e8, 5e10), who="sol", label=r"$\Sigma_{\rm e} = M_\star\,/\,2\pi R_{\rm e}^2$  [M$_\odot$ kpc$^{-2}$]",
                           short="Sigma_e", nm=r"$\Sigma_{\rm e}$", obs_note="semi-major $R_{\\rm e}$ / {sims} (sims)"),
           "age":     dict(col="age", raw="age", log=False, lim=(0.3, 8.8), who="sol", label="stellar age  [Gyr]", short="age",
                           obs_note="SED fit (CIGALE mass-w., MAGPHYS light-w., FAST) / {sims} (sims)"),
           "ssfr":    dict(col="lssfr", raw="ssfr", log=True, floor=P6_SSFR_FLOOR, lim=(6e-14, 4e-10), who="sol", cens="ssfr_censor", label=r"sSFR  [yr$^{-1}$]", short="sSFR",
                           obs_note="SED fit / {sims} (sims); < 1e-13 drawn at 1e-13"),
           "re":      dict(col="lre", raw="re_kpc", log=True, lim=(0.4, 40.0), who="sol", label=r"$R_{\rm e}$  [kpc]", short="R_e", nm=r"$R_{\rm e}$",
                           obs_note="semi-major half-light radius (rest 5000 Å) / projected half-mass radius (sims)"),
           "kappa_H2": dict(col="kH2", raw="kH2", log=False, lim=(0.0, 1.0), who="s", label=r"$\kappa_{\rm rot}^{\rm H_2}$  (r < 10 kpc)", short="kappa_H2", nm=r"$\kappa_{\rm rot}^{\rm H_2}$",
                            obs_note=f"models only: rotational energy fraction of the H$_2$, >= {P6_NKIN_MIN} particles"),
           "sersic_n": dict(col="ln", raw="n", log=True, lim=(0.4, 10.0), who="ol", label="Sérsic index $n$", short="n",
                            obs_note="observed only: COSMOS-Web / ACS (ALMA-C11), van der Wel+12 (LEGA-C, 3D-HST)"),
           "fdust":   dict(col="lfdust", raw="fdust", log=True, lim=(2e-6, 6e-3), who="so", cens="fdust_censor", label=r"$M_{\rm dust}\,/\,M_\star$", short="M_dust/M*", nm=r"$M_{\rm dust}/M_\star$",
                           obs_note="CIGALE $M_{\\rm dust}$ / $M_\\star$ (FAST++ for ADF22-QG1) / {sims} (sims)"),
           "fh2":     dict(col="lfh2", raw="fh2", log=True, lim=(2e-4, 6e-1), who="so", cens="fh2_censor", label=r"$M_{\rm H_2}\,/\,M_\star$", short="M_H2/M*", nm=r"$M_{\rm H_2}/M_\star$",
                           obs_note=r"$\alpha_{\rm CO}$ masses incl. He / {sims} (sims)")}

_cm0 = plt.get_cmap(P6_CMAP[0])
P6_CM = LinearSegmentedColormap.from_list(P6_CMAP[0] + "_p6", _cm0(np.linspace(P6_CMAP[1], P6_CMAP[2], 256)))
P6_NORM = Normalize(*P6_OBS_CLIM)
_cof = lambda v: P6_CM(P6_NORM(float(v))) if np.isfinite(v) else "white"


def _sigma_e(mstar, re_kpc):
    """Effective surface density [Msun kpc^-2]: half of the mass inside the half-light / half-mass radius."""
    mstar, re_kpc = np.asarray(mstar, float), np.asarray(re_kpc, float)
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(np.isfinite(mstar) & np.isfinite(re_kpc) & (re_kpc > 0), 0.5 * mstar / (np.pi * re_kpc ** 2), np.nan)


def _r_half(M, R=np.asarray(P6_COG_KPC)):
    """Projected half-mass radius from a cumulative curve of growth M(<R): log-log interpolation between the aperture radii;
    inside the innermost aperture a flat Sigma is assumed (M ∝ R^2)."""
    M = np.asarray(M, float)
    if not np.all(np.isfinite(M)) or M[-1] <= 0:
        return np.nan
    f = M / M[-1]
    if f[0] >= 0.5:
        return float(R[0] * np.sqrt(0.5 / max(f[0], 1e-9)))
    lf = np.log10(np.maximum(f, 1e-9))
    return float(10 ** np.interp(np.log10(0.5), lf, np.log10(R)))


def _fs(spec, v):
    """Raw axis values -> plot space (log10 for log axes, identity otherwise); a log axis floors at spec['floor'] if given, else drops non-positive values."""
    v = np.asarray(v, float)
    if not spec["log"]:
        return v
    fl = spec.get("floor")
    with np.errstate(divide="ignore", invalid="ignore"):
        if fl:
            return np.where(np.isfinite(v), np.log10(np.maximum(v, fl)), np.nan)
        return np.where(np.isfinite(v) & (v > 0), np.log10(np.where(v > 0, v, 1.0)), np.nan)


def _raw(spec, v):
    return 10 ** np.asarray(v, float) if spec["log"] else np.asarray(v, float)


def _add_fit_cols(df, cols):
    """Adds the plot-space column of every axis to df from its raw column (cols = {axis key: raw column name})."""
    for key, raw in cols.items():
        df[P6_AXES[key]["col"]] = _fs(P6_AXES[key], df[raw])
    return df


# ── the simulated galaxies: one row per (galaxy, sightline); columns mstar (the M* of the fractions and of sSFR), mstar_tot (behind Sigma_e), sfr,
#    age_m_star_myr, M_dust, M_H2, A_V — from the SIMBA truth (model "sim") or from the CIGALE fit of the P6_CIG_REGION photometry ("cigale") ──
_apt6 = _frame(APERTURE_TRUTH_FITS, "m25 Part 7e", KEY4 + ["nstar_ap", "mstar", "sfr100", "age_m_star_myr"])
_ism6 = _frame(ANNULUS_ISM_FITS, "m25 Part 8a", KEY4 + ["M_dust", "M_H2"])
_avt6 = _frame(ANNULUS_AV_FITS, "m25 Part 8b", KEY4 + ["A_V"])
_kin6 = _frame(ANNULUS_KIN_FITS, "m25 Part 8j0", ["snap", "gal_id", "aperture", "kappa_H2", "n_H2"])
_K3 = ["snap", "gal_id", "incl"]
_cog = _apt6[_apt6["aperture"].isin(P6_COG)].pivot_table(index=_K3, columns="aperture", values="mstar")[P6_COG]
_lm6 = {(int(a), int(b)): float(m) for a, b, m in zip(SEL["snap"], SEL["gal_id"], SEL["log_mstar"])}
_pop6 = {(int(a), int(b)): p for a, b, p in zip(SEL["snap"], SEL["gal_id"], _s(SEL["pop"]))}
_med_cols = [P6_AXES[k]["col"] for k in P6_AXES if "s" in P6_AXES[k]["who"]] + ["r_half"]


def _rows_sim():
    _tot = _apt6[_apt6["aperture"] == P6_AP_TOT].set_index(_K3)[["nstar_ap", "mstar", "sfr100", "age_m_star_myr"]].rename(columns={"sfr100": "sfr"})
    _ism_ap = _ism6[_ism6["aperture"] == P6_AP].set_index(_K3)[["M_dust", "M_H2"]]
    _av_ap = _avt6[_avt6["aperture"] == P6_AP].set_index(_K3)[["A_V"]]
    G = _tot.join(_cog, how="inner").join(_ism_ap, how="inner").join(_av_ap, how="inner").reset_index()
    G["mstar_tot"] = G["mstar"]
    return G, P6_AP


def _rows_cigale():
    # the Part 7f table: bayes.* = the fit; mstar / nstar_ap = the region truth it was joined to (the sampling floor); Av_zp = the region's dust_off zero point
    _cg = _frame(CIGALE_REGION_FITS, "m25 Part 7f", KEY4 + ["arm", "nstar_ap", "mstar", "bayes.stellar.m_star", "bayes.dust.mass", P6_CIG_SFR,
                                                               "bayes.stellar.age_m_star", "bayes.attenuation.Av_ISM", "Av_zp"])
    _cg = _cg[(_cg["arm"].astype(str).str.strip() == P6_CIG_ARM) & (_cg["aperture"] == P6_CIG_REGION)].set_index(_K3)
    if _cg.index.has_duplicates:
        raise RuntimeError(f"{CIGALE_REGION_FITS}: several {P6_CIG_ARM} {P6_CIG_REGION} fits per (galaxy, sightline) — pick one before using them")
    _ap_out, _ap_in = P6_REGION_AP[P6_CIG_REGION]
    _h2 = _ism6[_ism6["aperture"] == _ap_out].set_index(_K3)["M_H2"]
    if _ap_in is not None:                                   # a shell: the difference of the two cumulative rungs
        _h2 = (_h2 - _ism6[_ism6["aperture"] == _ap_in].set_index(_K3)["M_H2"]).clip(lower=0.0)
    _msim_tot = _apt6[_apt6["aperture"] == P6_AP_TOT].set_index(_K3)["mstar"].rename("msim_tot")
    G = pd.DataFrame({"nstar_ap": _cg["nstar_ap"], "mstar": _cg["bayes.stellar.m_star"], "mstar_sim": _cg["mstar"], "sfr": _cg[P6_CIG_SFR],
                      "age_m_star_myr": _cg["bayes.stellar.age_m_star"], "M_dust": _cg["bayes.dust.mass"] / MSUN_KG,
                      "A_V": _cg["bayes.attenuation.Av_ISM"] - (_cg["Av_zp"].fillna(0.0) if P6_CIG_AV_ZP else 0.0)})
    G = G.join(_h2.rename("M_H2"), how="inner").join(_cog, how="inner").join(_msim_tot, how="inner").reset_index()
    # Sigma_e needs a total: the CIGALE region M* aperture-corrected to the 0–32 kpc disc by the simulated curve of growth (an observer's aperture correction)
    with np.errstate(divide="ignore", invalid="ignore"):
        G["mstar_tot"] = G["mstar"] * G["msim_tot"] / G["mstar_sim"]
    return G, P6_REGION_KIN[P6_CIG_REGION]


def _build_models(model):
    """(per-galaxy medians GG, sightline rows G) of the quenched galaxies under the mass cut for one model source ("sim" | "cigale"); prints the sample."""
    G, kin_lab = {"sim": _rows_sim, "cigale": _rows_cigale}[model]()
    _kin_ap = _kin6[_kin6["aperture"] == kin_lab].set_index(["snap", "gal_id"])
    G = G.join(_kin_ap[["kappa_H2", "n_H2"]], on=["snap", "gal_id"], how="left")          # sightline-independent: one kappa per galaxy, repeated on its rows
    G["r_half"] = [_r_half(m) for m in G[P6_COG].to_numpy(float)]
    ok_star = np.nan_to_num(G["nstar_ap"].to_numpy(float), nan=0.0) >= NSTAR_AP_MIN
    G["mtot"] = np.where(ok_star & (G["mstar"].to_numpy(float) > 0), G["mstar"].to_numpy(float), np.nan)
    G["age"] = np.where(ok_star, G["age_m_star_myr"].to_numpy(float) / 1e3, np.nan)
    G["sig_e"] = _sigma_e(np.where(np.isfinite(G["mtot"]), G["mstar_tot"].to_numpy(float), np.nan), G["r_half"])
    G["av"] = np.where(np.isfinite(G["A_V"]) & (G["A_V"] > 0), G["A_V"].to_numpy(float), np.nan)
    G["ssfr"] = np.maximum(G["sfr"].to_numpy(float), 0.0) / G["mtot"].to_numpy(float)       # zero SFR kept (drawn at the floor: a quenched galaxy IS at zero)
    G["kH2"] = np.where(np.isfinite(G["kappa_H2"]) & (G["n_H2"].fillna(0).to_numpy(float) >= P6_NKIN_MIN), G["kappa_H2"].to_numpy(float), np.nan)
    G["fdust"] = G["M_dust"].to_numpy(float) / G["mtot"].to_numpy(float)          # zero dust / H2 -> dropped by _fs (never floored)
    G["fh2"] = P6_HE * G["M_H2"].to_numpy(float) / G["mtot"].to_numpy(float)
    _add_fit_cols(G, {"av": "av", "sigma_e": "sig_e", "age": "age", "ssfr": "ssfr", "re": "r_half", "kappa_H2": "kH2", "fdust": "fdust", "fh2": "fh2"})
    G["gkey"] = [f"{int(s)}_{int(g)}" for s, g in zip(G["snap"], G["gal_id"])]
    G["cls"] = [CLASS_OF.get((int(s), int(g)), "unclassified") for s, g in zip(G["snap"], G["gal_id"])]
    G["log_mstar"] = [_lm6.get((int(s), int(g)), np.nan) for s, g in zip(G["snap"], G["gal_id"])]
    G["pop"] = [_pop6.get((int(s), int(g)), "?") for s, g in zip(G["snap"], G["gal_id"])]
    GG = G.groupby("gkey").agg(**{c: (c, "median") for c in _med_cols}, cls=("cls", "first"), pop=("pop", "first"), log_mstar=("log_mstar", "first"), n_los=("incl", "size"))
    GG = GG[(GG["pop"] == "Q") & (GG["log_mstar"] > P6_MASS_MIN)]
    GG["grp"] = np.where(np.isfinite(GG[P6_SPLIT[0]]), np.where(GG[P6_SPLIT[0]] >= P6_SPLIT[1], "dusty", "dust-poor"), "none")
    G = G[G["gkey"].isin(GG.index)].reset_index(drop=True)
    G["grp"] = GG["grp"].reindex(G["gkey"]).to_numpy()
    n_aged = int((np.log10(GG["age"] * 1e9) > P6_LOGAGE_MIN).sum())
    print(f"Part 6 models [{model}] = {P6_MODEL_NOTE[model].replace(chr(10), ' ')}: {len(GG)} quenched galaxies with log M* > {P6_MASS_MIN}, {n_aged} of them with "
          f"log(age/yr) > {P6_LOGAGE_MIN} ({', '.join(f'{c} {int((GG.cls == c).sum())}' for c in ['weak', 'intermediate', 'strong', 'no_event'])}); {len(G)} sightline rows; "
          f"kappa_rot(H2) measurable for {int(np.isfinite(GG.kH2).sum())} galaxies; no dust {int(GG.lfdust.isna().sum())}, no H2 {int(GG.lfh2.isna().sum())} (per-galaxy medians); "
          f"R_1/2 inside 1 kpc (extrapolated) for {int((G.r_half < 1).sum())} rows")
    if model == "cigale":
        nq = int(((_s(SEL["pop"]) == "Q") & (np.asarray(SEL["log_mstar"], float) > P6_MASS_MIN)).sum())
        ap_out, ap_in = P6_REGION_AP[P6_CIG_REGION]
        with np.errstate(divide="ignore", invalid="ignore"):
            dm = np.log10(G["mstar"] / G["mstar_sim"])
            dd = np.log10(G["M_dust"] / _ism6[_ism6["aperture"] == ap_out].set_index(_K3)["M_dust"].reindex(pd.MultiIndex.from_frame(G[_K3])).to_numpy())
        dd = dd[np.isfinite(dd)]
        print(f"  {P6_CIG_ARM} {P6_CIG_REGION} fits for {len(GG)} of the {nq} quenched galaxies under the mass cut; over the {len(G)} sightline rows the fit − truth of the "
              f"{_REGN}: log M* {np.nanmedian(dm):+.2f} dex (16–84 % [{np.nanpercentile(dm, 16):+.2f}, {np.nanpercentile(dm, 84):+.2f}]), log M_dust {np.median(dd):+.2f} dex "
              f"(16–84 % [{np.percentile(dd, 16):+.2f}, {np.percentile(dd, 84):+.2f}]; {int((G['M_dust'] <= 0).sum())} rows fitted at zero dust); sSFR from {P6_CIG_SFR}; "
              f"M_H2 = SIMBA {ap_out}" + (f" − {ap_in}" if ap_in else "") + f"; kappa_H2 of the {kin_lab} rung")
    return GG, G


P6_MODELS = {m: _build_models(m) for m in dict.fromkeys([P6_MODEL] + [o.get("model", P6_MODEL) for _, _, _, o in P6_FIGS])}


def _model_sample(logage_min, model=P6_MODEL):
    """The quenched, mass-cut galaxies (per-galaxy medians) and their sightline rows of one model source above the age cut (None = no age cut)."""
    gg_all, g_all = P6_MODELS[model]
    gg = gg_all if logage_min is None else gg_all[np.log10(gg_all["age"] * 1e9) > logage_min]
    return gg, g_all[g_all["gkey"].isin(gg.index)]


GG6, G6 = _model_sample(P6_LOGAGE_MIN)                       # the default source under both cuts: the split statistics below
_GS = GG6[GG6["cls"].isin(P6_CLASSES)]
print(f"  [{P6_MODEL}] {' + '.join(P6_CLASSES)} (N={len(_GS)}) split at log(M_dust/M*) = {P6_SPLIT[1]:g}: dusty {int((_GS.grp == 'dusty').sum())}, dust-poor {int((_GS.grp == 'dust-poor').sum())}"
      + (f", no dust {int((_GS.grp == 'none').sum())}" if (_GS.grp == "none").any() else "") + "; per-galaxy medians, the Mann–Whitney p of the two groups, Spearman with log(M_dust/M*):")
for k in ("age", "re", "sigma_e", "ssfr", "kappa_H2", "fh2"):
    a, b = _GS.loc[_GS.grp == "dusty", P6_AXES[k]["col"]].dropna(), _GS.loc[_GS.grp == "dust-poor", P6_AXES[k]["col"]].dropna()
    pmw = mannwhitneyu(a, b).pvalue if min(len(a), len(b)) >= 2 else np.nan
    rho = spearmanr(_GS[P6_AXES[k]["col"]], _GS["lfdust"], nan_policy="omit")
    print(f"    {P6_AXES[k]['short']:>9s}: dusty {_raw(P6_AXES[k], a.median()):8.3g} (n={len(a):2d})  dust-poor {_raw(P6_AXES[k], b.median()):8.3g} (n={len(b):2d})  "
          f"p = {pmw:.3f};  rho = {rho[0]:+.2f} (p = {rho[1]:.2g})")
print("  the dusty group:", ", ".join(f"{g} (age {r.age:.1f} Gyr, R_e {r.r_half:.1f} kpc, kappa_H2 {r.kH2:.2f}, log f_dust {r.lfdust:.2f})"
                                       for g, r in _GS[_GS.grp == "dusty"].sort_values("lfdust", ascending=False).iterrows()))

# ── the observed points (every panel reads the columns it needs) ──
OBS6 = []
_gd = pd.read_csv(need(OBS_GD_CSV, "git pull")).set_index("id")
_st = pd.read_csv(need(OBS_STRUCT_CSV, "git pull")).set_index("id")
_tf = lambda v: str(v).strip().lower() in ("true", "1")
for sid, r in _gd.iterrows():
    if sid not in _st.index:
        print(f"  ALMA-C11 {sid}: no structure row -> skipped")
        continue
    s = _st.loc[sid]
    re = float(s["re_kpc"]) if P6_RE == "maj" else float(s["re_cosmosweb_kpc"] if np.isfinite(s["re_cosmosweb_kpc"]) else s["re_acs_kpc"])
    ctrl = str(sid).startswith("ctrl")
    if ctrl and not P6_OBS_CTRL:
        continue
    M = 10 ** float(r["logMstar"])
    OBS6.append(dict(sample="ALMA-C11 controls" if ctrl else "ALMA-C11", id=sid, z=float(r["z"]), av=float(r["AV_" + str(r["fid_run"]).strip()]), logM=float(r["logMstar"]),
                     re_kpc=re, re_source=str(s["re_source"]), sig_e=float(_sigma_e(M, re)), age=10 ** float(r["logage"]) / 1e9, n=float(s["n"]), n_source=str(s["n_source"]),
                     ssfr=float(r["SFR"]) / M, ssfr_censor="det", fdust=10 ** float(r["log_fdust"]), fdust_censor="ul" if _tf(r["dust_ul"]) else "det",
                     fh2=10 ** float(r["logMH2"]) / M, fh2_censor="ul" if _tf(r["co_ul"]) else "det",          # logMH2 = the fiducial mass or the 3 sigma limit
                     marker="o", size=42 if ctrl else 120, edge="0.35" if ctrl else C_OBS_EDGE))
_lit = pd.read_csv(need(LIT_CSV, "git pull"))
_lit["re_use"] = _lit["re_5000_kpc"] if P6_RE == "maj" else _lit["re_5000_kpc"] * np.sqrt(_lit["q"].clip(lower=0.05))
_lit["sig_e"] = _sigma_e(10 ** _lit["logM"].to_numpy(float), _lit["re_use"].to_numpy(float))
_lit["ssfr"] = np.where(_lit["logssfr"] > -50, 10 ** _lit["logssfr"].to_numpy(float), 0.0)     # FAST -99 = unconstrained -> the floor
_lit["age"] = 10 ** _lit["logage_fit"].to_numpy(float)
_sp = pd.read_csv(need(SPILKER_CSV, "git pull"))
_legac = _lit[_lit["sample"] == "LEGA-C"].drop_duplicates("id").set_index("id")
for _, r in _sp.iterrows():
    lid = int(str(r["galaxy"]).replace("LEGA-C", "").strip())
    if lid not in _legac.index:
        print(f"  Spilker+18 {r['galaxy']}: not in the LEGA-C literature rows -> skipped")
        continue
    L = _legac.loc[lid]
    OBS6.append(dict(sample="Spilker+18", id=f"LEGA-C {lid}", z=float(L["z"]), av=float(L["av"]), logM=float(L["logM"]), re_kpc=float(L["re_use"]), re_source="LEGA-C DR3 F814W",
                     sig_e=float(L["sig_e"]), age=float(L["age"]), n=float(L["n"]), n_source="van der Wel+12", ssfr=float(L["ssfr"]), ssfr_censor="det",
                     fdust=np.nan, fdust_censor="none", fh2=10 ** float(r["logMH2_h"]) / 10 ** float(L["logM"]), fh2_censor="det",
                     marker="D", size=64, edge="0.2"))
U = P6_UMEHATA
OBS6.append(dict(sample=f"{U['name']} ({U['ref']})", id=U["name"], z=U["z"], av=U["av"], logM=U["logM"], re_kpc=U["re_kpc"], re_source="Subaru AO K' (Kubo+17)",
                 sig_e=float(_sigma_e(10 ** U["logM"], U["re_kpc"])), age=10 ** U["logage"] / 1e9, n=np.nan, n_source="", ssfr=U["ssfr_ul"], ssfr_censor="ul",
                 fdust=10 ** (U["logMd_ul"] - U["logM"]), fdust_censor="ul", fh2=10 ** (U["logMH2"] - U["logM"]), fh2_censor="det",
                 marker="*", size=300, edge="k"))
OBS6 = pd.DataFrame(OBS6)
_add_fit_cols(OBS6, {"av": "av", "sigma_e": "sig_e", "age": "age", "ssfr": "ssfr", "re": "re_kpc", "sersic_n": "n", "fdust": "fdust", "fh2": "fh2"})
_oq, _oc = OBS6[OBS6["sample"] == "ALMA-C11"], OBS6[OBS6["sample"] == "ALMA-C11 controls"]
_oa = OBS6[OBS6["sample"].str.startswith("ALMA")]
print(f"  ALMA-C11: {len(_oq)} QGs (A_V {_oq.av.min():.2f}–{_oq.av.max():.2f}; median age {_oq.age.median():.1f} Gyr, R_e {_oq.re_kpc.median():.1f} kpc, n {_oq.n.median():.1f}) "
      f"and {len(_oc)} controls (A_V {_oc.av.min():.2f}–{_oc.av.max():.2f}; median age {_oc.age.median():.1f} Gyr, R_e {_oc.re_kpc.median():.1f} kpc, n {_oc.n.median():.1f}); "
      f"Spearman of A_V over the {len(_oa)} with n {spearmanr(_oa.av, _oa.n)[0]:+.2f}, R_e {spearmanr(_oa.av, _oa.re_kpc)[0]:+.2f}, age {spearmanr(_oa.av, _oa.age)[0]:+.2f}")

# ── the literature under the user's two cuts ──
_L = _lit[(_lit["logM"] > P6_MASS_MIN) & (_lit["logage_fit"] + 9 > P6_LOGAGE_MIN)].copy()
if P6_LIT_Z is not None:
    _L = _L[_L["z"].between(*P6_LIT_Z)]
if P6_LIT_QUIESCENT:
    _L = _L[_L["quiescent_uvj"].astype(object).eq(True)]
_add_fit_cols(_L, {"av": "av", "sigma_e": "sig_e", "age": "age", "ssfr": "ssfr", "re": "re_use", "sersic_n": "n"})
for name, ls in P6_LIT:
    g = _L[_L["sample"] == name]
    print(f"  {name}: {len(g)} galaxies under the cuts (z {g['z'].quantile(0.05):.2f}–{g['z'].quantile(0.95):.2f} 5–95 %; A_V median {np.median(g['av']):.2f}; "
          f"age median {np.median(g['age']):.2f} Gyr; R_e median {np.nanmedian(g['re_use']):.2f} kpc; n median {np.nanmedian(g['n']):.2f}; "
          f"{int((g['ssfr'] <= P6_SSFR_FLOOR).sum())} at the sSFR floor)")


def _run_lit(g, xs, ys):
    """Running median of y in equal-count x bins (plot-space columns) of >= P6_LIT_NMIN objects -> DataFrame (x, y, lo, hi, n)."""
    x, y = g[xs["col"]].to_numpy(float), g[ys["col"]].to_numpy(float)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]
    if len(x) < 2 * P6_LIT_NMIN:
        return pd.DataFrame(columns=["x", "y", "lo", "hi", "n"])
    nb = max(2, min(P6_LIT_NBIN, len(x) // P6_LIT_NMIN))
    e = np.quantile(x, np.linspace(0, 1, nb + 1))
    ib = np.clip(np.searchsorted(e, x, side="right") - 1, 0, nb - 1)
    pts = [(np.median(x[ib == b]), np.median(y[ib == b]), np.percentile(y[ib == b], 16), np.percentile(y[ib == b], 84), int((ib == b).sum()))
           for b in range(nb) if (ib == b).sum() >= P6_LIT_NMIN]
    return pd.DataFrame(pts, columns=["x", "y", "lo", "hi", "n"])


def _model_track(xs, ys, keys, G, seed=0):
    """Running median of y against x over the sightline rows G of the given galaxies (galaxy bootstrap band) -> (track, n_gal)."""
    rws = G[G["gkey"].isin(keys) & np.isfinite(G[xs["col"]]) & np.isfinite(G[ys["col"]])]
    ng = rws["gkey"].nunique()
    nmin = P6_NMIN_X if ng >= 2 * P6_NMIN_X else 2
    tr = run_median(rws[xs["col"]], rws[ys["col"]], rws["gkey"], nbins=P6_NBIN_X, nmin=nmin, n=500, seed=seed) if ng >= 2 * nmin else pd.DataFrame(columns=["x", "y", "lo", "hi", "n_gal", "n_rows"])
    return tr, ng


def _track_rows(rows, kind, pan, sample, tr, ng, xs, ys):
    for _, r in tr.iterrows():
        rows.append(dict(kind=kind, panel=pan, sample=sample, n_gal_group=ng, x=float(_raw(xs, r["x"])), y=float(_raw(ys, r["y"])),
                         y_lo16=float(_raw(ys, r["lo"])), y_hi84=float(_raw(ys, r["hi"])), n_gal=int(r["n_gal"]), n_rows=int(r["n_rows"])))


def _draw_track(ax, tr, xs, ys, color, ls="-", marker="o", band=True):
    X, Y = _raw(xs, tr["x"]), _raw(ys, tr["y"])
    if band:
        ax.fill_between(X, _raw(ys, tr["lo"]), _raw(ys, tr["hi"]), color=color, alpha=0.14, lw=0, zorder=3)
    ax.plot(X, Y, color=color, lw=2.8, ls=ls, marker=marker, ms=7, mec="k", mew=0.6, zorder=4, path_effects=[withStroke(linewidth=4.6, foreground="white")])


def p6_panel(ax, xkey, ykey, mode, classes, GG, G, rows, tag, xlab=True, ylab=True):
    """One plane: literature tracks, model tracks (split groups or classes of the figure's sample GG / G), the observed points coloured by A_V
    -> what was drawn (for the legends) + the checks."""
    xs, ys = P6_AXES[xkey], P6_AXES[ykey]
    pan = f"{xkey}_{ykey}"
    drawn = dict(lit={}, grp={}, cls={}, allq=0, obs=False, tracks={})
    if xs["log"]:
        ax.set_xscale("log")
    if ys["log"]:
        ax.set_yscale("log")
    ax.set_xlim(*xs["lim"]); ax.set_ylim(*ys["lim"])
    if "l" in xs["who"] and "l" in ys["who"]:
        for name, ls in P6_LIT:
            g = _L[_L["sample"] == name]
            tr = _run_lit(g, xs, ys)
            if not len(tr):
                continue
            drawn["lit"][name] = (ls, len(g))
            X, Y, LO, HI = _raw(xs, tr["x"]), _raw(ys, tr["y"]), _raw(ys, tr["lo"]), _raw(ys, tr["hi"])
            ax.fill_between(X, LO, HI, facecolor="0.3", edgecolor="none", alpha=0.07, lw=0, zorder=1)
            ax.plot(X, LO, color="0.45", lw=0.7, ls=ls, zorder=1.5); ax.plot(X, HI, color="0.45", lw=0.7, ls=ls, zorder=1.5)
            ax.plot(X, Y, color="0.15", lw=2.0, ls=ls, zorder=2, path_effects=[withStroke(linewidth=3.6, foreground="white")])
            for _, r in tr.iterrows():
                rows.append(dict(kind="literature_track", panel=pan, sample=name, x=float(_raw(xs, r["x"])), y=float(_raw(ys, r["y"])),
                                 y_lo16=float(_raw(ys, r["lo"])), y_hi84=float(_raw(ys, r["hi"])), n_rows=int(r["n"])))
    if "s" in xs["who"] and "s" in ys["who"]:
        if mode == "split":
            for grp, st in P6_SPLIT_STYLE.items():
                tr, ng = _model_track(xs, ys, GG.index[GG["cls"].isin(classes) & (GG["grp"] == grp)], G)
                print(f"  {pan} {grp}: {ng} galaxies, {len(tr)} track points" + (f"; {ys['short']} {_raw(ys, tr['y'].min()):.3g}–{_raw(ys, tr['y'].max()):.3g} over "
                      f"{xs['short']} {_raw(xs, tr['x'].min()):.3g}–{_raw(xs, tr['x'].max()):.3g}" if len(tr) else ""))
                if len(tr):
                    drawn["grp"][grp] = ng; drawn["tracks"][grp] = tr
                    _draw_track(ax, tr, xs, ys, st["color"], st["ls"], st["marker"])
                    _track_rows(rows, "model_track", pan, f"SIMBA {'+'.join(classes)} {grp}", tr, ng, xs, ys)
        else:
            if P6_PRED_ALLQ:
                tr, ng = _model_track(xs, ys, GG.index, G, seed=1)
                if len(tr):
                    drawn["allq"] = ng; drawn["tracks"]["allQ"] = tr
                    ax.plot(_raw(xs, tr["x"]), _raw(ys, tr["y"]), color="0.35", lw=1.4, ls=":", zorder=2.5)
                    _track_rows(rows, "model_track_allQ", pan, "SIMBA all quenched", tr, ng, xs, ys)
            for c in classes:
                tr, ng = _model_track(xs, ys, GG.index[GG["cls"] == c], G)
                Gc = GG[GG["cls"] == c]
                rho = spearmanr(Gc[xs["col"]], Gc[ys["col"]], nan_policy="omit") if len(Gc) >= 5 else (np.nan, np.nan)
                print(f"  {pan} {c}: {ng} galaxies, {len(tr)} track points" + (f"; {ys['short']} {_raw(ys, tr['y'].min()):.3g}–{_raw(ys, tr['y'].max()):.3g} over "
                      f"{xs['short']} {_raw(xs, tr['x'].min()):.3g}–{_raw(xs, tr['x'].max()):.3g}" if len(tr) else "") + f"; Spearman({xs['short']}, {ys['short']}) = {rho[0]:+.2f} (p = {rho[1]:.2g})")
                if len(tr):
                    drawn["cls"][c] = ng; drawn["tracks"][c] = tr
                    _draw_track(ax, tr, xs, ys, CLASS_COLOR_PRES[c])
                    _track_rows(rows, "model_track", pan, f"SIMBA {c}", tr, ng, xs, ys)
    if "o" in xs["who"] and "o" in ys["who"]:
        drawn["obs"] = True
        for _, r in OBS6.iterrows():
            if not (np.isfinite(r[xs["col"]]) and np.isfinite(r[ys["col"]])):
                continue
            x, y = float(_raw(xs, r[xs["col"]])), float(_raw(ys, r[ys["col"]]))
            ax.scatter(x, y, s=r["size"], marker=r["marker"], c=[_cof(r[P6_AXES[P6_OBS_COLOUR]["raw"]])], edgecolors=r["edge"], linewidths=1.4, zorder=6 + (r["marker"] == "*"))
            if ys.get("cens") and r[ys["cens"]] == "ul":      # upper limit on y: arrow down
                ax.annotate("", (x, y), xytext=(0, -15), textcoords="offset points", arrowprops=dict(arrowstyle="<|-", color=r["edge"], lw=1.2, mutation_scale=9), zorder=8)
            if xs.get("cens") and r[xs["cens"]] == "ul":      # upper limit on x: arrow left
                ax.annotate("", (x, y), xytext=(-15, 0), textcoords="offset points", arrowprops=dict(arrowstyle="<|-", color=r["edge"], lw=1.2, mutation_scale=9), zorder=8)
        # the check: measured values against each class / the dusty track (interpolated in plot space inside the track's x range)
        for key in ([c for c in classes if c in drawn["tracks"]] if mode == "class" else [k for k in ("dusty",) if k in drawn["tracks"]]):
            if not ys.get("cens"):
                continue
            tr = drawn["tracks"][key]
            O = OBS6[(OBS6[ys["cens"]] == "det") & np.isfinite(OBS6[xs["col"]]) & np.isfinite(OBS6[ys["col"]]) & OBS6[xs["col"]].between(tr["x"].min(), tr["x"].max())]
            if len(O) >= 3:
                d = O[ys["col"]].to_numpy(float) - np.interp(O[xs["col"]].to_numpy(float), tr["x"].to_numpy(float), tr["y"].to_numpy(float))
                print(f"    measured − {key} track over the {len(O)} detections inside the track's {xs['short']} range: median {np.median(d):+.2f} dex, 16–84 % [{np.percentile(d, 16):+.2f}, {np.percentile(d, 84):+.2f}]"
                      + (f" ({', '.join(sorted(set(O['sample'])))})"))
                for _, o in O.iterrows():
                    rows.append(dict(kind="obs_vs_track", panel=pan, sample=o["sample"], id=o["id"], track=key, x=float(_raw(xs, o[xs["col"]])), y=float(_raw(ys, o[ys["col"]])),
                                     y_track=float(_raw(ys, np.interp(o[xs["col"]], tr["x"], tr["y"]))), offset_dex=float(o[ys["col"]] - np.interp(o[xs["col"]], tr["x"], tr["y"]))))
    ax.set_xlabel(xs["label"] if xlab else "", fontsize=P6_FONT["label"]); ax.set_ylabel(ys["label"] if ylab else "", fontsize=P6_FONT["label"])
    ax.tick_params(direction="in", top=True, right=True, which="both", labelsize=P6_FONT["tick"], labelbottom=xlab, labelleft=ylab); ax.grid(False)
    for axis, spec in ((ax.yaxis, ys), (ax.xaxis, xs)):
        if spec["log"] and spec["lim"][0] >= 0.1 and spec["lim"][1] < 1e4:
            axis.set_major_formatter(FuncFormatter(lambda v, p: f"{v:g}"))
    if tag:
        ax.set_title(tag, loc="left", fontsize=P6_FONT["tag"], pad=6)
    return drawn


def p6_figure(stem, grid, mode, opts=None):
    """One figure: a grid of panels (shared A_V colour bar), the observed and the model legends below -> CSVs paper_<stem>{,_points}.csv."""
    opts = opts or {}
    classes, logage_min, model = opts.get("classes", P6_CLASSES), opts.get("logage_min", P6_LOGAGE_MIN), opts.get("model", P6_MODEL)
    GG, G = _model_sample(logage_min, model)
    age_txt = f"age > {10 ** (logage_min - 9):g} Gyr" if logage_min is not None else "no age cut"
    print(f"\n── figure {stem} ({mode}; {' + '.join(classes)}; models = {model}; log M* > {P6_MASS_MIN:g}, {age_txt}: {len(GG)} galaxies, "
          f"{', '.join(f'{c} {int((GG.cls == c).sum())}' for c in classes)}): "
          + " | ".join(", ".join(f"{P6_AXES[y]['short']} vs {P6_AXES[x]['short']}" for x, y in row) for row in grid) + " ──")
    rows, nr, nc = [], len(grid), max(len(r) for r in grid)
    single = (nr == 1 and nc == 1)
    ncol = max(nc, 2 if single else 3)
    fig = plt.figure(figsize=((10.2, 8.8) if single else (4.9 * nc + 1.6, 4.1 * nr + 2.9)))
    gs = GridSpec(nr + 1, ncol, height_ratios=[1.0] * nr + [0.30 if single else 0.46], width_ratios=([1.12, 0.88] if single else [1.0] * ncol),
                  hspace=0.16 if single else 0.10, wspace=0.0 if single else 0.08, figure=fig)
    axes = [[fig.add_subplot(gs[0, :])]] if single else [[fig.add_subplot(gs[i, j]) for j in range(len(grid[i]))] for i in range(nr)]
    lax, lax2, lax3 = fig.add_subplot(gs[nr, 0]), fig.add_subplot(gs[nr, 1]), (None if single else fig.add_subplot(gs[nr, 2:]))
    for a in (lax, lax2, lax3):
        if a is not None:
            a.axis("off")
    drawn, k = [], 0
    for i, row in enumerate(grid):
        for j, (x, y) in enumerate(row):
            tag = "" if single else f"({'abcdefghij'[k]})"; k += 1
            drawn.append(p6_panel(axes[i][j], x, y, mode, classes, GG, G, rows, tag, xlab=(i == nr - 1), ylab=(j == 0)))
    flat = [a for row in axes for a in row]
    sm = plt.cm.ScalarMappable(cmap=P6_CM, norm=P6_NORM); sm.set_array([])
    cb = fig.colorbar(sm, ax=flat, pad=0.012, extend="max", fraction=0.045 if single else 0.02, aspect=30 if single else 40)
    cb.set_label(P6_AXES[P6_OBS_COLOUR]["label"] + " of the observed galaxies", fontsize=P6_FONT["label"]); cb.ax.tick_params(labelsize=P6_FONT["tick"])
    nq, ns = int((OBS6["sample"] == "ALMA-C11").sum()), int((OBS6["sample"] == "Spilker+18").sum())
    lits = {kk: v for d in drawn for kk, v in d["lit"].items()}
    ykeys = [y for row in grid for x, y in row]
    lim_note = "arrow = upper limit (dust, CO, sSFR)" if any(P6_AXES[y].get("cens") for y in ykeys) or any(P6_AXES[x].get("cens") for row in grid for x, y in row) else ""
    h_obs = [(Line2D([], [], marker="o", ls="", ms=10, mfc=_cof(np.mean(P6_OBS_CLIM)), mec=C_OBS_EDGE, mew=1.3), rf"ALMA-C11 QGs, $z\approx0.4$ (N={nq}): colour = $A_V$ (fiducial CIGALE)" + (f"\n{lim_note}" if lim_note else "")),
             (Line2D([], [], marker="D", ls="", ms=7.5, mfc=_cof(0.3), mec="0.2", mew=1.3), rf"Spilker+18 LEGA-C passive, $z\approx0.7$ (N={ns}): CO only" + ("" if single else "") + ("; DR3 structure + SED" if single else "")),
             (Line2D([], [], marker="*", ls="", ms=15, mfc=_cof(U["av"]), mec="k", mew=1.2), rf"{U['name']} ({U['ref']}), $z = {U['z']:.2f}$: $A_V$ = {U['av']:g} (FAST++)")]
    if P6_OBS_CTRL:
        h_obs.insert(1, (Line2D([], [], marker="o", ls="", ms=6.5, mfc=_cof(0.15), mec="0.35", mew=1.2), "ALMA-C11 controls (no ALMA detection: limits only)"))
    h_obs += [(Line2D([], [], color="0.15", lw=2.0, ls=ls), f"{name} running median, thin = 16–84 % (N={n})") for name, (ls, n) in lits.items()]
    if mode == "split":
        ngrp = {grp: max(d["grp"].get(grp, 0) for d in drawn) for grp in P6_SPLIT_STYLE}
        sgn = {"dusty": r"\geq", "dust-poor": "<"}
        h_mod = [(Line2D([], [], color=st["color"], lw=2.8, ls=st["ls"], marker=st["marker"], ms=7, mec="k", mew=0.6),
                  rf"{grp}: $\log\,(M_{{\rm dust}}/M_\star) {sgn[grp]} {P6_SPLIT[1]:g}$ (N={ngrp[grp]})") for grp, st in P6_SPLIT_STYLE.items() if ngrp[grp]]
        mod_title = (f"SIMBA m25 quenched, {' + '.join(CLASS_NAME[c] for c in classes)} AGN coupling (N={int(GG['cls'].isin(classes).sum())}),\n"
                     f"log $M_\\star$ > {P6_MASS_MIN:g}, {age_txt}: running median (band 16–84 %),\nsplit on $M_{{\\rm dust}}/M_\\star$; {P6_MODEL_NOTE[model]}")
    else:
        ncls = {c: max(d["cls"].get(c, 0) for d in drawn) for c in classes}
        h_mod = [(Line2D([], [], color=CLASS_COLOR_PRES[c], lw=2.8, marker="o", ms=7, mec="k", mew=0.6), f"{CLASS_NAME[c]} AGN coupling (N={ncls[c]})")
                 for c in classes if ncls[c]]
        if any(d["allq"] for d in drawn):
            h_mod.append((Line2D([], [], color="0.35", lw=1.4, ls=":"), f"all quenched galaxies under the cut (N={max(d['allq'] for d in drawn)})"))
        mod_title = (f"SIMBA m25 quenched, log $M_\\star$ > {P6_MASS_MIN:g}, {age_txt}:\nthe ISM content read off a cheap observable,\nrunning median per class (band 16–84 %)\n{P6_MODEL_NOTE[model]}")
    ytop = 1.0 if single else 0.80          # the strip starts under the x labels of the bottom row
    leg1 = lax.legend([h for h, _ in h_obs], [l for _, l in h_obs], loc="upper left", bbox_to_anchor=(-0.04, ytop), frameon=False, fontsize=P6_FONT["legend"], handlelength=1.6,
                      title="observed", title_fontsize=P6_FONT["legend"] + 1)
    leg1._legend_box.align = "left"
    leg2 = lax2.legend([h for h, _ in h_mod], [l for _, l in h_mod], loc="upper left", bbox_to_anchor=(0.10 if single else -0.02, ytop), frameon=False, fontsize=P6_FONT["legend"],
                       handlelength=2.2, title=mod_title, title_fontsize=P6_FONT["legend"] + 0.5)
    leg2._legend_box.align = "left"
    keys_used = list(dict.fromkeys([x for row in grid for x, y in row] + ykeys))
    notes = [f"{P6_AXES[kk].get('nm', P6_AXES[kk]['short'])}: {P6_AXES[kk]['obs_note'].replace('{sims}', P6_SIM_TXT[model].get(kk, ''))}" for kk in keys_used]
    notes.append(f"literature: log $M_\\star$ > {P6_MASS_MIN:g}, SED-fit age > {10 ** (P6_LOGAGE_MIN - 9):g} Gyr" + ("" if P6_LIT_Z is None else f", {P6_LIT_Z[0]:g} < z < {P6_LIT_Z[1]:g}")
                 + ("; UVJ-quiescent" if P6_LIT_QUIESCENT else ""))
    if single:
        flat[0].text(0.01, 0.99, "\n".join(notes), transform=flat[0].transAxes, fontsize=P6_FONT["note"], color="0.35", ha="left", va="top", linespacing=1.3, zorder=9)
    else:
        lax3.text(-0.02, ytop, "\n".join(["definitions"] + notes), transform=lax3.transAxes, fontsize=P6_FONT["note"], color="0.35", ha="left", va="top", linespacing=1.45, wrap=True)
    paper_save(fig, stem)
    plt.show()
    pd.DataFrame(rows).to_csv(os.path.join(PAPERDIR, f"paper_{stem}.csv"), index=False)
    OBS6.to_csv(os.path.join(PAPERDIR, f"paper_{stem}_points.csv"), index=False)
    print(f"  tables -> {os.path.join(PAPERDIR, f'paper_{stem}{{,_points}}.csv')}")
    return drawn


P6_DRAWN = {stem: p6_figure(stem, grid, mode, opts) for stem, grid, mode, opts in P6_FIGS}
print(f"\n  {'source':>26s} {'z':>5s} {'A_V':>5s} {'age':>5s} {'R_e':>6s} {'Sigma_e':>8s} {'n':>5s} {'log sSFR':>9s} {'log fdust':>10s} {'log fH2':>10s}")
for _, r in OBS6.sort_values(["sample", "av"]).iterrows():
    fmt = lambda v, c: ("< " if c == "ul" else "  ") + (f"{np.log10(v):.2f}" if np.isfinite(v) and c != "none" and v > 0 else " —  ")
    print(f"  {str(r['sample'])[:26]:>26s} {r['z']:5.2f} {r['av']:5.2f} {r['age']:5.1f} {r['re_kpc']:6.2f} {r['sig_e']:8.2e} {r['n']:5.2f} {fmt(r['ssfr'], r['ssfr_censor']):>9s} "
          f"{fmt(r['fdust'], r['fdust_censor']):>10s} {fmt(r['fh2'], r['fh2_censor']):>10s}   {r['id']}")


## Conventions, caveats, what the pieces are

* **Nothing is measured here.** Every number is read from a table written by `powderday_flux_quenched_m25.ipynb` or
  `ks_tracks_quenched_m25.ipynb`; the helpers are copies of theirs (Part 2d/2e stacks, Part 4b/4c plane, Part 5b/5c clock) with
  the same floors and minimum counts (`BAND_MIN` = 15 galaxies for a 16–84 % band, `NMIN_BIN` = 4 uncensored galaxies for a stage
  median, `NMIN_GRID` = 5 for a clock median, `NMIN_ECDF` = 4 for an ECDF).
* **AGN class.** `AGN_CLASSIFIER` selects the `agn_class_<tag>` column of the selection table and the `ks_tracks_<tag>/` products;
  the KS products are relabelled from that column on load (a warning means the KS notebook was last run under another rule — its
  region products then mix rules, re-run it). Part 1 compares the paper's pair of rules regardless of the switch.
* **Part 1 clocks** are $t - t_{\rm SFT}$ (m25 Part 2d); **Part 2 clocks** are $t - t_{\rm QT}$ (KS Part 5). The Part 3 tracks are
  stage-to-stage (SFT → QT → track end), not on a clock.
* **$A_V$** is the RT attenuation $-2.5\log(F_{\rm on}/F_{\rm off})$ in the rest-frame V band of the m25 notebook's dust_on / dust_off
  runs, in the projected aperture `AV_APERTURE` at the anchor; the median over the four sightlines removes the orientation
  dependence of a single projection. It is an attenuation of the *anchor* galaxy — the bins are galaxy properties fixed at the
  anchor, so a galaxy keeps its bin along the whole track (as in the KS notebook's binned figures).
* **Thin sample.** With $10.5 < \log M_\star < 11.2$ the mass-limited quenched sample is ~90 galaxies; three $A_V$ bins × three
  classes leave 2–15 galaxies per bin and the individual tracks are drawn on purpose — a bin median with fewer than `NMIN_BIN`
  uncensored galaxies is an upper limit (open marker). Per-redshift rows (`P3_ROWS`) are possible but thinner still.
* **Part 4 annuli.** Projected annuli of the m25 ladder (0–1, 1–3.2, 3.2–10, 10–32, 32–100 kpc; the core of figure E is the
  cumulative 0–3.2 kpc disc, its zones `E_ZONES`). $\Sigma_{\rm dust}$ needs `NGAS_ANN_MIN` = 10 gas particles, $M_\star$ and the
  age `NSTAR_AP_MIN` = 20 star particles, $\kappa_{\rm rot}$ 10 weighted gas particles (m25 8j0) — an annulus without dust has
  no value on a log-scaled axis and is absent from the medians, so every dust median is over the *dusty* annuli (the printed N
  say how many). The dust quantities are drawn as linear values on log-scaled axes; their medians and bands are computed in dex
  (a median is invariant under the transform). The stellar age is the mass-weighted
  age of the stars in the annulus at the anchor; over all anchors it mixes the redshift range (`P4_MASS_RANGE` narrows the sample, a
  per-anchor cut is not provided). No rank statistics: the running medians are equal-count bins in age with a galaxy bootstrap.
* **Part 5 zones** are the SPHERICAL core (r < 3.2 kpc) and outskirt shell (3.2–10 kpc) of KS Part 3b about the reduced-file
  centre — not the projected annuli of Part 4 (a rotation measure is 3-D); every particle in the zone (the m25 8j0 convention),
  velocities about the zone's own bulk motion, spin axis = its angular momentum. $\kappa_{\rm rot}$ is measurable only with
  ≥ `NKIN_MIN` = 10 weighted particles *and* velocities in the file (`has_vel`); the stage medians are over the galaxies that
  still have one (survivors, no censoring floor). The stellar age of the zone is the mass-weighted age of its stars (≥ 20 with a
  formation epoch). $A_V$ bins = terciles of each mass sample (`P5_AV_EDGES`); the SF controls are off (`P5_SHOW_SF`): their
  $\kappa_{\rm rot}^{\rm H_2}$ is far above the quenched tracks, so showing them switches the $y$ axis to a log scale.
* **Part 5 critical points.** `agn_ign` / `jet_on` are the m25 Part 2d ignition / jet-onset events (times relative to SFT in
  `agn_classifier_windows.fits`) placed on the nearest history row by KS Part 1 (`kl.attach_bh_stages`); they are new (snap, gx)
  pairs, so the KS Part 2 plan must be rebuilt and re-submitted before Parts 3 / 3b / 4 carry them — until then Part 5 draws
  SFT → QT → end and prints a note. Figure B compares the classes at each point regardless of the age range they cover; figure C
  compares the paired per-galaxy changes over the intervals (survivor bias: an interval needs both endpoints measurable, which for
  the H$_2$ of a quenched core is often not the case after QT).
* **Part 5 sequence design (2026-08-29).** The class palette is teal / amber / dark red (`CLASS_COLOR_PRES`, Part 2a — every paper
  figure; the sequential purples had too little contrast). In the sequence / intervals figures the ZONE of a row is its frame and
  tag colour (`P5_ZONE_COLOR`: core deep blue, outskirt bronze), never a line colour; the columns show their class counts, the class
  legend is figure-level. $M_{\rm dust}/M_\star$, $M_{\rm H_2}/M_\star$ and the sSFR at the critical points come from the KS tracks' PROJECTED
  apertures (`ap3kpc`, `ap10kpc − ap3kpc`), not from the spherical kinematic zones, and need `NSTAR_AP_MIN` stars; a zero dust / H$_2$
  mass is dropped, a zero SFR is floored at `P5_SSFR_FLOOR` and kept (a group median on the floor = most of the group forms no stars
  there — quenching is the point, so no survivor cut). The
  shaded band is the min–max of the ALMA-C11 sources flagged `P5_OBS_DET` (`det_dust`; `P5_OBS_DET_H2` = `det_co` for the H$_2$
  rows, `MH2_fid` / 1.36 to the hydrogen-only convention of SIMBA) in `almac11_gas_dust.csv` — fiducial CIGALE run, WHOLE-galaxy
  sSFR, age, $M_{\rm dust}/M_\star$ and $M_{\rm H_2}/M_\star$ against the zone values of the simulation, so read it as the parameter space the
  observations reach, not as a matched measurement; on the age rows only its lower edge is kept in view when the upper edge would
  dwarf the simulated range (the legend quotes both). The last row of the stellar figure is $\Delta t / t_{\rm cosmic}$
  (`P5_DT_NORM`: the cosmic time at the end of the interval), so a slow quencher reads the same at every redshift.
* **Outputs** go to `output/cis25/plots/paper_m25[_<tag>]/`: `paper_agn_{feedback_sequence,selection_windows,selection_scorecard}`,
  `paper_ks_{agn,dust,kinematics}_R50_H2`, `paper_ks_av_bins_by_class_R50_H2`, `paper_dust_vs_age_annuli`, `paper_core_vs_outskirt`,
  `paper_radial_profiles_{mgt10p25,mlt10p25}`, `paper_kappa_vs_age_{H2,star}_{mgt10p25,mlt10p25}`,
  `paper_{kinematics,stellar,ism}_sequence_{mgt10p25,mlt10p25}`,
  `paper_kappa_intervals_{mgt10p25,mlt10p25}` (PNG + PDF) and `paper_ks_av_bins.csv`, `paper_dust_vs_age_annuli.csv`,
  `paper_core_vs_outskirt.csv`, `paper_radial_profiles.csv`, `paper_kappa_{vs_age,sequence,intervals}.csv`.